# Deteksi, Klasifikasi, dan Penghitungan Kendaraan 3D
### Notebook lengkap: seluruh kode dan tahapnya

Notebook ini memuat **seluruh kode proyek**, disusun berurutan dari latihan model sampai evaluasi:

1. Setelan dan data
2. Latihan dan evaluasi detektor CenterNet
3. Klasifier gambar 4 kelas
4. Pipeline: deteksi, pelacakan, klasifikasi, penghitungan
5. Evaluasi hitungan terhadap ground truth
6. Ringkasan dan keterbatasan

**Bagaimana kode disimpan.** Tiap berkas `.py` ditampilkan per fungsi atau kelas, dengan penjelasan di antaranya.
Sel kodenya diawali `%%writefile`: saat dijalankan, sel itu menulis kode ke folder `kode_notebook/` di samping notebook
(sel pertama sebuah berkas menulis ulang berkasnya, sel berikutnya `-a` menambahkan). Setelah semua sel kode sebuah tahap
dijalankan, tahap itu dijalankan dari folder tersebut. Jadi notebook ini bisa dibaca sebagai dokumentasi kode, dan juga
bisa dijalankan.

Kode di notebook ini disusun dari kode asli di `~/projects` pada 2026-11-14 13:49. Sel pemeriksaan di bagian 1 memastikan isinya sama.

**Tahap yang lama tidak dijalankan secara bawaan.** Melatih detektor dan klasifier butuh berjam-jam tanpa GPU, jadi
saklarnya `False`. Ubah di sel Setelan bila ingin melatih ulang. Hasil latihan dari notebook ditulis ke
`~/projects/hasil/notebook/`, tidak menimpa bobot yang sedang dipakai.

**Menjalankan di Google Colab atau Kaggle (GPU).** Letakkan folder `projects` dan `dataset_publik` di folder rumah
(`/root` pada Colab), karena kode memakai jalur `~/projects/...` dan `~/dataset_publik/...`.

## 1. Setelan dan data

In [ ]:
import os, sys, glob, subprocess, re, time
from IPython.display import display, Image

RUMAH    = os.path.expanduser("~")
PROYEK   = os.path.join(RUMAH, "projects")
PUBLIK   = os.path.join(RUMAH, "dataset_publik")
DATA     = os.path.join(PROYEK, "data_session0_right")
HASIL    = os.path.join(PROYEK, "hasil")
HASIL_NB = os.path.join(HASIL, "notebook")          # keluaran latihan dari notebook (tidak menimpa bobot asli)
KODE     = os.path.abspath("kode_notebook")         # kode yang ditulis sel-sel %%writefile
os.makedirs(HASIL_NB, exist_ok=True)

VIDEO_UJI  = os.path.join(PROYEK, "video_uji", "uji_val_720p.mp4")
CACHE_DET  = os.path.join(PROYEK, "video_uji", "det_session0.pkl")
BOBOT_DET  = os.path.join(PROYEK, "bobot", "detektor")
BOBOT_KLAS = os.path.join(PROYEK, "bobot", "klasifikasi")
MODEL_DET  = os.path.join(BOBOT_DET, "centernet_dari_baru.pth")
FRAME_AWAL, FRAME_AKHIR = 2545, 3181      # rentang uji, tidak dipakai saat melatih detektor

LATIH_DETEKTOR  = False     # True = latih ulang detektor dari bobot lama dan bobot baru (berjam-jam tanpa GPU)
LATIH_KLASIFIER = False     # True = latih ulang klasifier (berjam-jam tanpa GPU)
PAKAI_KLASIFIER_GAMBAR = True

def jalankan(perintah, cwd, ekor=60, tampil=True):
    """Jalankan skrip dengan Python yang sama dengan kernel ini; kembalikan (kode keluar, teks keluaran)."""
    if perintah[0] == "python3":
        perintah = [sys.executable] + perintah[1:]
    p = subprocess.run(perintah, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    baris = [b for b in re.split(r"[\r\n]+", p.stdout) if b.strip() and "it/s]" not in b and "s/it]" not in b]
    if tampil:
        print("\n".join(baris[-ekor:]))
        if p.returncode != 0:
            print("\n[perintah berhenti dengan kode %d]" % p.returncode)
    return p.returncode, "\n".join(baris)

def tampil_gambar(jalur, lebar=900):
    if os.path.exists(jalur):
        display(Image(filename=jalur, width=lebar))
    else:
        print("(gambar tidak ada: %s)" % jalur)

def tampil_frame(bgr, lebar=900):
    import cv2
    ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    display(Image(data=buf.tobytes(), width=lebar))

Data latihan dan uji adalah 3.182 frame dari satu kamera jalan raya (`session0_right`), dengan anotasi kotak 3D per
kendaraan (satu XML per frame) dan berkas kalibrasi kamera. Data ini dipakai sebagai **latihan**, karena kamera milik
proyek belum terpasang. Pembagiannya berurutan menurut waktu: frame 0–2544 untuk latihan, 2545–3181 untuk uji.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter
ann = sorted(glob.glob(os.path.join(DATA, "annotations", "*.xml")))
print("Berkas anotasi:", len(ann))
tipe = {"latih": Counter(), "uji": Counter()}
for a in ann:
    m = re.search(r"_(\d{6})\.", os.path.basename(a))
    bagian = "uji" if m and int(m.group(1)) >= FRAME_AWAL else "latih"
    for o in ET.parse(a).getroot().findall("object"):
        tipe[bagian][o.findtext("type", "?")] += 1
for bagian, c in tipe.items():
    print("%-6s objek: %5d  %s" % (bagian, sum(c.values()), dict(c)))

## 2. Latihan dan evaluasi detektor CenterNet

Detektor adalah CenterNet dengan tulang punggung ResNet-18 (stride 32). Untuk tiap kendaraan, jaringan mengeluarkan
heatmap titik sentuh tanah dan rasio lebar, tinggi, panjang. Detektor ini tidak menghasilkan kotak 2D dan tidak
membedakan jenis kendaraan.

Arahan dosen pembimbing: *"Tes aja dulu. Simpan best train epoch .pth, dan lakukan train dengan pth yang lama dan yang baru."*

| Latihan | Bobot awal (`--init`) | Bobot hasil |
|---|---|---|
| Dari bobot lama | `centernet-3d-bbox_asli.pth` (bobot penulis proyek asal) | `centernet_dari_lama.pth` |
| Dari bobot baru | `centernet_session0_right.pth` (hasil latihan sendiri) | `centernet_dari_baru.pth` |

Tafsiran "pth lama" dan "pth baru" di tabel ini belum dikonfirmasi ke dosen.

Berkas di tahap ini (urutan: yang diimpor lebih dulu). Pustaka luar yang dipakai: `cv2`, `numpy`, `pandas`, `torch`, `torchvision`, `tqdm`.

- `kamera.py` (asli: `~/projects/latih_detektor/kamera.py`)
- `utils.py` (asli: `~/projects/acuan/3d_bbox_highway/utils.py`)
- `target.py` (asli: `~/projects/latih_detektor/target.py`)
- `train_session0_right.py` (asli: `~/projects/latih_detektor/train_session0_right.py`)
- `dekode.py` (asli: `~/projects/latih_detektor/dekode.py`)
- `prediksi.py` (asli: `~/projects/latih_detektor/prediksi.py`)
- `evaluasi.py` (asli: `~/projects/latih_detektor/evaluasi.py`)
- `analisis_galat.py` (asli: `~/projects/latih_detektor/analisis_galat.py`)

### `kamera.py`

Parameter kamera `session0_right` (matriks intrinsik K dan posisi kamera) pada resolusi 1280 × 720. Dipakai untuk mengubah label anotasi menjadi target latihan dan sebaliknya.

*Asli: `~/projects/latih_detektor/kamera.py`, 45 baris.*

In [ ]:
%%writefile kode_notebook/latih_detektor/kamera.py
"""kamera.py - K dan T kamera session0_right (resolusi 1280x720).
Salinan logika cek_bagian1.py baris 13-36 (terbukti: IoU kotak 2D median 0,906).
Dipakai bersama oleh pelatihan dan deteksi/tracking.
"""
import os
import xml.etree.ElementTree as ET
import numpy as np

BASE_DEFAULT = os.path.expanduser("~/projects/data_session0_right")
S = 2.0 / 3.0  # skala 1920x1080 -> 1280x720


def load_camera(base=None):
    """Mengembalikan dict: K (3x3), T (4x4, to_world_from_camera), f, f_full, h_m, R_wc."""
    base = base or BASE_DEFAULT
    cal = ET.parse(os.path.join(base, "calib", "session0_right_calibParams.xml")).getroot()
    f_full = float(cal.findtext("f"))
    h_m = float(cal.findtext("h")) / 1000.0
    v = [float(t) for t in cal.findtext("vanishPoints").split()]
    Kf = np.array([[f_full, 0, 960.], [0, f_full, 540.], [0, 0, 1.]])

    def direction(vp):
        d = np.linalg.inv(Kf) @ np.array([vp[0], vp[1], 1.0])
        return d / np.linalg.norm(d)

    d1, d2 = direction(v[0:2]), direction(v[2:4])
    if d1[2] < 0: d1 = -d1
    if d2[0] < 0: d2 = -d2
    X = d2 - (d2 @ d1) * d1
    X /= np.linalg.norm(X)
    Z = np.cross(X, d1)
    if Z[1] > 0: Z = -Z
    flip = np.array([1, -1, 1])
    R_wc = np.stack([X * flip, Z * flip, d1 * flip], axis=0)
    T = np.identity(4)
    T[:3, :3] = R_wc
    T[:3, 3] = [0, h_m, 0]
    f = f_full * S
    K = np.array([[f, 0, 640.], [0, f, 360.], [0, 0, 1.]])
    return {"K": K, "T": T, "f": f, "f_full": f_full, "h_m": h_m, "R_wc": R_wc}


if __name__ == "__main__":
    c = load_camera()
    print("f(720p)=%.2f  h=%.3f m  det(R)=%.6f" % (c["f"], c["h_m"], np.linalg.det(c["R_wc"])))


### `utils.py`

Fungsi geometri dari proyek asal (antoinekeller/3d_bbox_highway): proyeksi titik dunia ke gambar dan titik gambar ke bidang jalan.

*Asli: `~/projects/acuan/3d_bbox_highway/utils.py`, 329 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/utils.py
"""
Define a bunch of utils class and functions related to geometry and opencv.
"""

import cv2
import numpy as np




#### fungsi `project` — Given the camera intrinsic and extrinsic calibration matrices,

Bersama: fungsi `make_euler` — Define euler matrix with some angle around a specified axis

In [ ]:
%%writefile -a kode_notebook/latih_detektor/utils.py
def project(point_in_world, to_camera_from_world, K):
    """
    Given the camera intrinsic and extrinsic calibration matrices,
    projet a 3D point in world coordinates to the image
    """

    point_in_world = point_in_world.reshape(3, 1)
    point_in_camera = to_camera_from_world[:3, :3].dot(
        point_in_world
    ) + to_camera_from_world[:3, 3].reshape(3, 1)

    point_in_camera /= abs(point_in_camera[2])

    projection = K.dot(point_in_camera)
    projection = projection.flatten()

    return projection[:2]

def make_euler(angle, axis):
    """Define euler matrix with some angle around a specified axis"""

    if axis < 0:
        raise ValueError
    if axis > 3:
        raise ValueError

    if axis == 0:  # Pitch (axis = X)
        pitch_matrix = np.zeros((3, 3))
        pitch_matrix[0, 0] = 1
        pitch_matrix[1, 1] = np.cos(angle)
        pitch_matrix[1, 2] = np.sin(angle)
        pitch_matrix[2, 1] = -np.sin(angle)
        pitch_matrix[2, 2] = np.cos(angle)
        return pitch_matrix

    if axis == 1:  # Yaw (axis = Y)
        yaw_matrix = np.zeros((3, 3))
        yaw_matrix[0, 0] = np.cos(angle)
        yaw_matrix[0, 2] = np.sin(angle)
        yaw_matrix[1, 1] = 1
        yaw_matrix[2, 0] = -np.sin(angle)
        yaw_matrix[2, 2] = np.cos(angle)
        return yaw_matrix

    return False



#### fungsi `get_position_ground` — Find position in world on the driveway (altitude = 0)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/utils.py
def get_position_ground(x, y, K, to_world_from_camera, img_height):
    """
    Find position in world on the driveway (altitude = 0)
    by ray tracing a pixel point [x, y]
    """

    point_in_camera = np.linalg.inv(K).dot(
        np.array([x, img_height - y, 1]).reshape(3, 1)
    )
    point_in_camera = point_in_camera.reshape(3, 1)
    # Find intersection with Y = 0

    point_in_world = to_world_from_camera[:3, :3].dot(
        point_in_camera
    ) + to_world_from_camera[:3, 3].reshape(3, 1)

    camera_pos_in_world = to_world_from_camera[:3, 3].reshape(3, 1)

    direction = point_in_world - camera_pos_in_world
    direction /= np.linalg.norm(direction)

    lambda_ = -camera_pos_in_world[1] / direction[1]

    position_on_ground = (camera_pos_in_world + lambda_ * direction).flatten()

    return position_on_ground




#### fungsi `draw_fps` — Draw fps to demonstrate performance

Bersama: fungsi `draw_frame_id` — Draw frame id to demonstrate performance

In [ ]:
%%writefile -a kode_notebook/latih_detektor/utils.py
def draw_fps(frame, fps):
    """Draw fps to demonstrate performance"""
    cv2.putText(
        frame,
        f"{int(fps)} fps",
        (20, 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        fontScale=0.8,
        color=(0, 165, 255),
        thickness=2,
    )


def draw_frame_id(frame, frame_id):
    """Draw frame id to demonstrate performance"""
    cv2.putText(
        frame,
        f"Frame {frame_id}",
        (frame.shape[1] - 150, frame.shape[0] - 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        fontScale=0.8,
        color=(0, 165, 255),
        thickness=2,
    )

# Define a bunch of RGB colors (used in tracking)
COLORS = [
    (0, 0, 128),
    (0, 0, 255),
    (0, 128, 0),
    (0, 128, 128),
    (0, 128, 255),
    (0, 255, 0),
    (0, 255, 128),
    (0, 255, 255),
    (128, 0, 0),
    (128, 0, 128),
    (128, 0, 255),
    (128, 128, 0),
    (128, 128, 128),
    (128, 128, 255),
    (128, 255, 0),
    (128, 255, 128),
    (128, 255, 255),
    (255, 0, 0),
    (255, 0, 128),
    (255, 0, 255),
    (255, 128, 0),
    (255, 128, 128),
    (255, 128, 255),
    (255, 255, 0),
    (255, 255, 128),
    (255, 255, 255),
]




#### kelas `Box` — Define 3D bounding box

In [ ]:
%%writefile -a kode_notebook/latih_detektor/utils.py
class Box:
    """
    Define 3D bounding box
    """

    def __init__(self, x, z, w, h, l):
        self.x = x
        self.z = z
        self.w = w
        self.h = h
        self.l = l

    def project(self, img, to_world_from_camera, K, color=(255, 0, 255)):
        """
        Project 3D bounding box to the image img
        """
        to_camera_from_world = np.linalg.inv(to_world_from_camera)

        left_bottom_back = project(
            np.array([self.x - self.w / 2, 0, self.z - self.l / 2]),
            to_camera_from_world,
            K,
        )
        left_bottom_front = project(
            np.array([self.x - self.w / 2, 0, self.z + self.l / 2]),
            to_camera_from_world,
            K,
        )
        left_top_back = project(
            np.array([self.x - self.w / 2, self.h, self.z - self.l / 2]),
            to_camera_from_world,
            K,
        )
        left_top_front = project(
            np.array([self.x - self.w / 2, self.h, self.z + self.l / 2]),
            to_camera_from_world,
            K,
        )
        right_bottom_back = project(
            np.array([self.x + self.w / 2, 0, self.z - self.l / 2]),
            to_camera_from_world,
            K,
        )
        right_bottom_front = project(
            np.array([self.x + self.w / 2, 0, self.z + self.l / 2]),
            to_camera_from_world,
            K,
        )
        right_top_back = project(
            np.array([self.x + self.w / 2, self.h, self.z - self.l / 2]),
            to_camera_from_world,
            K,
        )
        right_top_front = project(
            np.array([self.x + self.w / 2, self.h, self.z + self.l / 2]),
            to_camera_from_world,
            K,
        )

        left_bottom_back[1] = img.shape[0] - left_bottom_back[1]
        left_bottom_front[1] = img.shape[0] - left_bottom_front[1]
        left_top_back[1] = img.shape[0] - left_top_back[1]
        left_top_front[1] = img.shape[0] - left_top_front[1]
        right_bottom_back[1] = img.shape[0] - right_bottom_back[1]
        right_bottom_front[1] = img.shape[0] - right_bottom_front[1]
        right_top_back[1] = img.shape[0] - right_top_back[1]
        right_top_front[1] = img.shape[0] - right_top_front[1]

        left_bottom_back = tuple(left_bottom_back.astype(int))
        left_bottom_front = tuple(left_bottom_front.astype(int))
        left_top_back = tuple(left_top_back.astype(int))
        left_top_front = tuple(left_top_front.astype(int))
        right_bottom_back = tuple(right_bottom_back.astype(int))
        right_bottom_front = tuple(right_bottom_front.astype(int))
        right_top_back = tuple(right_top_back.astype(int))
        right_top_front = tuple(right_top_front.astype(int))

        # Draw back rectangle
        cv2.line(
            img,
            left_bottom_back,
            left_top_back,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            left_bottom_back,
            right_bottom_back,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            right_top_back,
            right_bottom_back,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            left_top_back,
            right_top_back,
            color=color,
            thickness=3,
        )

        # Draw front rectangle
        cv2.line(
            img,
            right_bottom_front,
            right_top_front,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            right_top_front,
            left_top_front,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            left_top_front,
            left_bottom_front,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            left_bottom_front,
            right_bottom_front,
            color=color,
            thickness=3,
        )

        # Draw Z lines
        cv2.line(
            img,
            right_bottom_back,
            right_bottom_front,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            right_top_front,
            right_top_back,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            left_top_back,
            left_top_front,
            color=color,
            thickness=3,
        )
        cv2.line(
            img,
            left_bottom_back,
            left_bottom_front,
            color=color,
            thickness=3,
        )

        image_overlay = img.copy()

        # Overlay red color over the whole bbox
        hull = cv2.convexHull(
            np.array(
                [
                    left_bottom_back,
                    left_bottom_front,
                    left_top_back,
                    left_top_front,
                    right_bottom_back,
                    right_bottom_front,
                    right_top_back,
                    right_top_front,
                ],
                dtype=int,
            )
        ).reshape(1, -1, 2)
        cv2.fillPoly(image_overlay, pts=hull, color=(0, 0, 255))

        alpha = 0.3
        img = cv2.addWeighted(img, 1 - alpha, image_overlay, alpha, 0)

        return img


### `target.py`

Mengubah anotasi tiap frame menjadi **target latihan** CenterNet: heatmap titik tanah kendaraan dan nilai regresi rasio lebar, tinggi, panjang pada sel grid stride 32.

*Asli: `~/projects/latih_detektor/target.py`, 146 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/target.py
"""target.py - pembuatan masukan dan target latihan (tanpa PyTorch).
Salinan setia dari centernet-3d-bbox.ipynb (repo antoinekeller/3d_bbox_highway, commit 3fa494cd):
  draw_msra_gaussian (baris 84-108), draw_offset (111-114), make_hm_offset_regr (117-152),
  Normalize (238-248), pembacaan item CarlaDataset.__getitem__ (261-279).
Perubahan atas acuan (semua dicatat):
  1. K dan to_world_from_camera menjadi argumen (acuan: variabel global kamera Carla).
  2. y_min menjadi argumen (acuan: angka 150 tertulis langsung di baris 130).
  3. 'except:' telanjang diganti 'except IndexError:' (perilaku sama untuk kasus yang dimaksud).
  4. aug_fix (default False = persis acuan): jitter warna yang ditahan di 0-255, karena acuan
     membungkus modulo 256 (uji: piksel 3 dengan offset -20 menjadi 239).
  5. Normalize: impor torchvision yang tidak terpakai dibuang.
"""
import os
import sys
import numpy as np
import pandas as pd
import cv2

REPO = os.path.expanduser("~/projects/acuan/3d_bbox_highway")
sys.path.insert(0, REPO)
from utils import get_position_ground  # noqa: E402  (repo acuan, hanya dibaca)

INPUT_W, INPUT_H, MODEL_SCALE = 1280, 720, 32




#### fungsi `draw_msra_gaussian`

Bersama: fungsi `draw_offset`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/target.py
def draw_msra_gaussian(heatmap, center, sigma=2):
    tmp_size = sigma * 6
    mu_x = int(center[0])
    mu_y = int(center[1])
    w, h = heatmap.shape[0], heatmap.shape[1]
    ul = [int(mu_x - tmp_size), int(mu_y - tmp_size)]
    br = [int(mu_x + tmp_size + 1), int(mu_y + tmp_size + 1)]
    if ul[0] >= h or ul[1] >= w or br[0] < 0 or br[1] < 0:
        return heatmap
    size = br[0] - ul[0]
    x = np.arange(0, size, 1, np.float32)
    y = x[:, np.newaxis]
    x0 = y0 = size // 2
    g = np.exp(- ((x - x0) ** 2 + (y - y0) ** 2) / (2 * sigma ** 2))
    g_x = max(0, -ul[0]), min(br[0], h) - ul[0]
    g_y = max(0, -ul[1]), min(br[1], w) - ul[1]
    img_x = max(0, ul[0]), min(br[0], h)
    img_y = max(0, ul[1]), min(br[1], w)
    heatmap[img_y[0]:img_y[1], img_x[0]:img_x[1]] = np.maximum(
        heatmap[img_y[0]:img_y[1], img_x[0]:img_x[1]],
        g[g_y[0]:g_y[1], g_x[0]:g_x[1]])
    return heatmap


def draw_offset(offset, x, y):
    offset[0, int(y), int(x)] = x - int(x)
    offset[1, int(y), int(x)] = y - int(y)
    return offset




#### fungsi `make_hm_offset_regr`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/target.py
def make_hm_offset_regr(target, K, to_world_from_camera, y_min=150):
    hm = np.zeros([INPUT_H // MODEL_SCALE + 1, INPUT_W // MODEL_SCALE])
    offset = np.zeros([2, INPUT_H // MODEL_SCALE + 1, INPUT_W // MODEL_SCALE])
    regr = np.zeros([3, INPUT_H // MODEL_SCALE + 1, INPUT_W // MODEL_SCALE])

    if len(target) == 0:
        return hm, offset, regr

    for i, c in target.iterrows():
        hm = draw_msra_gaussian(hm, [int(c["x"] / MODEL_SCALE), int(c["y"] / MODEL_SCALE)],
                                sigma=1)
        if not (0 <= c["x"] < INPUT_W and y_min <= c["y"] < INPUT_H):
            continue

        assert 0 <= c["x"] < INPUT_W
        assert 0 <= c["y"] < INPUT_H
        offset = draw_offset(offset, c["x"] / MODEL_SCALE, c["y"] / MODEL_SCALE)

        position_on_ground = get_position_ground(
            c["x"], c["y"], K, to_world_from_camera, 720
        )
        depth = np.linalg.norm(position_on_ground - to_world_from_camera[:3, 3])

        for i in range(-1, 2):
            for j in range(-1, 2):
                try:
                    regr[0, int(c["y"] / MODEL_SCALE) + i, int(c["x"] / MODEL_SCALE) + j] = c["w"] / depth
                    regr[1, int(c["y"] / MODEL_SCALE) + i, int(c["x"] / MODEL_SCALE) + j] = c["h"] / depth
                    regr[2, int(c["y"] / MODEL_SCALE) + i, int(c["x"] / MODEL_SCALE) + j] = c["l"] / depth
                except IndexError:
                    pass

    return hm, offset, regr




#### kelas `Normalize`

Bersama: fungsi `load_item` — Mengembalikan img (3,720,1280) float32, hm (23,40), offset (2,23,40), regr (3,23,40) (float64 seperti acuan).

In [ ]:
%%writefile -a kode_notebook/latih_detektor/target.py
class Normalize(object):
    def __init__(self):
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def __call__(self, image):
        image = image.astype(np.float32) / 255
        image -= self.mean
        image /= self.std
        return image


def load_item(img_path, label_path, K, to_world_from_camera, y_min=150, train=False, aug_fix=False):
    """Mengembalikan img (3,720,1280) float32, hm (23,40), offset (2,23,40), regr (3,23,40) (float64 seperti acuan)."""
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(img_path)
    img = cv2.resize(img, (INPUT_W, INPUT_H))

    if train:
        if np.random.rand() > 0.5:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if aug_fix:
            jitter = np.random.randint(-20, 20, size=3)
            img = np.clip(img.astype(np.int16) + jitter, 0, 255).astype(np.uint8)
        else:
            rgb_offset = np.random.randint(-20, 20, size=3).astype('uint8')
            img += rgb_offset

    img = Normalize()(img)
    img = img.transpose([2, 0, 1])

    labels = pd.read_csv(label_path)
    hm, offset, regr = make_hm_offset_regr(labels, K, to_world_from_camera, y_min)
    return img, hm, offset, regr




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/target.py
if __name__ == "__main__":
    # Pemeriksaan pada data asli: python3 target.py
    import kamera
    cam = kamera.load_camera()
    K, T = cam["K"], cam["T"]
    base = os.path.expanduser("~/projects/data_session0_right/converted")
    names = sorted(os.listdir(os.path.join(base, "images")))
    for nm in [names[0], names[len(names) // 2], names[-1]]:
        lp = os.path.join(base, "labels", nm[:-4] + ".txt")
        img, hm, off, regr = load_item(os.path.join(base, "images", nm), lp, K, T, 150, train=False)
        lab = pd.read_csv(lp)
        uniq = len({(int(x / 32), int(y / 32)) for x, y in zip(lab["x"], lab["y"])})
        n_near = int((lab["y"] >= 150).sum())
        filled = int((regr[0] != 0).sum())
        nz = regr[0][regr[0] != 0]
        print("%s | img %s %s | hm %s | objek %d (y>=150: %d) | sel pusat hm>=1: %d (sel unik: %d) | sel ukuran terisi: %d (maks %d)"
              % (nm, img.shape, img.dtype, hm.shape, len(lab), n_near, int((hm >= 1).sum()), uniq, filled, 9 * n_near))
        if filled:
            print("    w/depth: min %.4f maks %.4f | offset maks %.3f" % (nz.min(), nz.max(), off.max()))


### `train_session0_right.py`

Loop latihan detektor. Opsi `--init` memilih bobot awal (bobot lama atau baru), dan bobot epoch terbaik disimpan ke `--out`. Berisi juga tambalan gradien NaN.

*Asli: `~/projects/latih_detektor/train_session0_right.py`, 281 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/train_session0_right.py
#!/usr/bin/env python3
"""train_session0_right.py - latih CenterNet-3D-bbox pada session0_right (CPU).
Salinan kode latih centernet-3d-bbox.ipynb (repo antoinekeller/3d_bbox_highway, commit 3fa494cd):
  model (baris 330-381), focal_loss/_regr_loss/global_loss (390-475), train/validasi (476-578),
  optimizer, StepLR, jumlah epoch, dan aturan simpan model terbaik (579-597).
Perubahan atas acuan (semua dicatat):
  1. Kamera K,T dari kamera.py (acuan: kamera Carla); target dari target.py.
  2. Folder data dari converted/images dan converted/labels; pembagian urutan 80/20 tanpa sklearn
     (aturan sama dengan train_test_split(test_size=0.2, shuffle=False): n_uji = ceil(0.2*n)).
  3. Target (hm, offset, regr) dijadikan float32 (acuan: float64 yang dicampur dengan float32 di loss).
  4. Log juga memuat loss ukuran (whl) latih dan validasi, waktu epoch, dan waktu per iterasi.
  5. plt/summary/assert cuda dibuang. Bobot disimpan ke --out (bukan ke centernet-3d-bbox.pth).
  6. Opsi: --y-min (acuan 150), --shuffle (acuan: tanpa acak), --aug-fix (acuan: jitter membungkus),
     --no-pretrained (hanya untuk uji tanpa internet), --max-train/--max-val/--epochs.
  7. Opsi --init: muat bobot awal dari .pth (bobot penulis atau hasil latihan sendiri).
  8. sigmoid pada global_loss dibatasi ke [1e-6, 1-1e-6] (acuan: tanpa batas; gradien NaN bila sigmoid jenuh 0/1).
  9. Langkah optimizer dilewati bila gradien tak berhingga; jumlahnya dicetak (acuan: hanya memeriksa loss).
"""
import argparse
import csv
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import kamera  # noqa: E402
import target as tg  # noqa: E402




#### fungsi `split_ids` — Urutan 80/20 seperti train_test_split(test_size=0.2, shuffle=False).

Bersama: kelas `Data`; kelas `centernet`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/train_session0_right.py
def split_ids(names, test_frac=0.2):
    """Urutan 80/20 seperti train_test_split(test_size=0.2, shuffle=False)."""
    n_test = int(np.ceil(test_frac * len(names)))
    return names[:len(names) - n_test], names[len(names) - n_test:]


class Data(torch.utils.data.Dataset):
    def __init__(self, names, images_dir, labels_dir, K, T, y_min, train, aug_fix):
        self.names, self.images_dir, self.labels_dir = names, images_dir, labels_dir
        self.K, self.T, self.y_min, self.train, self.aug_fix = K, T, y_min, train, aug_fix

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        nm = self.names[idx]
        img, hm, offset, regr = tg.load_item(
            os.path.join(self.images_dir, nm),
            os.path.join(self.labels_dir, nm[:-4] + ".txt"),
            self.K, self.T, self.y_min, self.train, self.aug_fix)
        return img, hm.astype(np.float32), offset.astype(np.float32), regr.astype(np.float32)


class centernet(nn.Module):
    def __init__(self, pretrained=True):
        super(centernet, self).__init__()
        basemodel = torchvision.models.resnet18(
            weights='ResNet18_Weights.IMAGENET1K_V1' if pretrained else None)
        self.base_model = nn.Sequential(*list(basemodel.children())[:-2])
        num_ch = 512
        head_conv = 64
        self.outc = nn.Sequential(
            nn.Conv2d(num_ch, head_conv, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(head_conv, 1, kernel_size=1, stride=1))
        self.outo = nn.Sequential(
            nn.Conv2d(num_ch, head_conv, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(head_conv, 2, kernel_size=1, stride=1))
        self.outr = nn.Sequential(
            nn.Conv2d(num_ch, head_conv, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(head_conv, 3, kernel_size=1, stride=1))

    def forward(self, x):
        x = self.base_model(x)
        assert not torch.isnan(x).any()
        outc = self.outc(x)
        assert not torch.isnan(outc).any()
        outo = self.outo(x)
        assert not torch.isnan(outo).any()
        outr = self.outr(x)
        return outc, outo, outr




#### fungsi `focal_loss`

Bersama: fungsi `_regr_loss`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/train_session0_right.py
def focal_loss(pred_mask, gt, gamma=0.2):
    gt = gt.unsqueeze(1).float()
    pos_inds = gt.ge(1.0).float()
    neg_inds = gt.lt(1.0).float()
    neg_weights = torch.pow(1 - gt, 4)
    pos_loss = - torch.pow(1 - pred_mask, gamma) * torch.log(pred_mask + 1e-12) * pos_inds
    neg_loss = - torch.pow(pred_mask, gamma) * torch.log(1 - pred_mask + 1e-12) * neg_inds * neg_weights
    loss = 0
    num_pos = pos_inds.float().sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum()
    if num_pos == 0:
        loss = neg_loss
    else:
        pos_loss /= num_pos
        neg_loss /= num_pos
        loss = pos_loss + neg_loss
    assert not torch.isnan(pos_loss)
    assert not torch.isnan(neg_loss)
    return loss, pos_loss, neg_loss


def _regr_loss(off_pred, off_gt, whl_pred, whl_gt, hm_gt):
    mask = hm_gt.unsqueeze(1).float().ge(1.0).float()
    num = mask.float().sum()
    mask_2 = mask.expand_as(off_gt).float()
    mask_3 = mask.expand_as(whl_gt).float()
    assert off_pred.size() == mask_2.size()
    off_pred = off_pred * mask_2
    off_gt = off_gt * mask_2
    whl_pred = whl_pred * mask_3
    whl_gt = whl_gt * mask_3
    assert whl_pred.size() == mask_3.size()
    assert mask_3.float().sum() == 3 * num
    off_loss = nn.functional.l1_loss(off_pred, off_gt, reduction='sum') / (num + 1e-4)
    whl_loss = 20.0 * nn.functional.l1_loss(whl_pred, whl_gt, reduction='sum') / (num + 1e-4)
    return off_loss, whl_loss




#### fungsi `global_loss`

Bersama: fungsi `run_pass` — Satu lintasan data. optimizer ada -> latih; tidak ada -> validasi. Mengembalikan (rata2 loss dict, detik/iterasi).

In [ ]:
%%writefile -a kode_notebook/latih_detektor/train_session0_right.py
def global_loss(hm_pred, hm_gt, off_pred, off_gt, whl_pred, whl_gt):
    pred_mask = torch.sigmoid(hm_pred[:, 0]).clamp(1e-6, 1 - 1e-6)
    pred_mask = pred_mask.unsqueeze(1).float()
    foc_loss, pos_loss, neg_loss = focal_loss(pred_mask, hm_gt)
    off_loss, whl_loss = _regr_loss(off_pred, off_gt, whl_pred, whl_gt, hm_gt)
    assert not torch.isnan(off_loss)
    return foc_loss, pos_loss, neg_loss, off_loss, whl_loss


def run_pass(model, loader, device, optimizer=None, desc=""):
    """Satu lintasan data. optimizer ada -> latih; tidak ada -> validasi. Mengembalikan (rata2 loss dict, detik/iterasi)."""
    training = optimizer is not None
    names = ["loss", "pos", "neg", "off", "whl"]
    tot = dict.fromkeys(names, 0.0)
    model.train() if training else model.eval()
    t0, n_iter, n_skip = time.time(), 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        t = tqdm(loader, desc=desc)
        for idx, (img, hm, offset, regr) in enumerate(t):
            img = img.to(device)
            hm_gt, offset_gt, whl_gt = hm.to(device), offset.to(device), regr.to(device)
            if training:
                optimizer.zero_grad()
            preds_hm, preds_offset, preds_whl = model(img)
            foc_loss, pos_loss, neg_loss, off_loss, whl_loss = global_loss(
                preds_hm, hm_gt, preds_offset, offset_gt, preds_whl, whl_gt)
            loss = foc_loss + off_loss + whl_loss
            if torch.isnan(loss):
                print("NAN loss")
                continue
            tot["loss"] += float(loss)
            tot["pos"] += float(pos_loss)
            tot["neg"] += float(neg_loss)
            tot["off"] += float(off_loss)
            tot["whl"] += float(whl_loss)
            if training:
                loss.backward()
                if not all(bool(torch.isfinite(q.grad).all()) for q in model.parameters() if q.grad is not None):
                    n_skip += 1
                    optimizer.zero_grad()
                    continue
                optimizer.step()
            n_iter += 1
            t.set_description(f'{desc} (l={tot["loss"]/(idx+1):.2f}) (pos={tot["pos"]/(idx+1):.3f}) '
                              f'(neg={tot["neg"]/(idx+1):.3f}) (off={tot["off"]/(idx+1):.3f}) (whl={tot["whl"]/(idx+1):.3f})')
    if n_skip:
        print('PERINGATAN: %d langkah dilewati karena gradien tak berhingga (%s)' % (n_skip, desc))
    n = len(loader)  # acuan membagi dengan len(loader), juga bila ada iterasi NaN yang dilewati
    return {k: v / n for k, v in tot.items()}, (time.time() - t0) / max(n_iter, 1)




#### fungsi `parse_args`

Bersama: fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/train_session0_right.py
def parse_args(argv=None):
    p = argparse.ArgumentParser(description="Latih CenterNet-3D-bbox pada session0_right (CPU).")
    conv = os.path.expanduser("~/projects/data_session0_right/converted")
    p.add_argument("--images", default=os.path.join(conv, "images"))
    p.add_argument("--labels", default=os.path.join(conv, "labels"))
    p.add_argument("--out", default=os.path.expanduser("~/projects/bobot/detektor/centernet_session0_right.pth"))
    p.add_argument("--force", action="store_true", help="izinkan menimpa --out yang sudah ada")
    p.add_argument("--logdir", default=os.path.expanduser("~/projects/hasil/log_latih_detektor"))
    p.add_argument("--tag", default="latih", help="nama pendek untuk file log")
    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--lr", type=float, default=5e-5)
    p.add_argument("--y-min", type=float, default=150)
    p.add_argument("--max-train", type=int, default=None)
    p.add_argument("--max-val", type=int, default=None)
    p.add_argument("--shuffle", action="store_true", help="acak urutan data latih (acuan: tidak)")
    p.add_argument("--aug-fix", action="store_true", help="jitter warna tanpa pembungkusan (acuan: membungkus)")
    p.add_argument("--no-pretrained", action="store_true", help="tanpa bobot ImageNet; hanya untuk uji tanpa internet")
    p.add_argument("--init", default=None, help="bobot awal (.pth): bobot penulis atau hasil latihan sendiri")
    p.add_argument("--seed", type=int, default=None)
    return p.parse_args(argv)


def main(argv=None):
    a = parse_args(argv)
    if os.path.exists(a.out) and not a.force:
        sys.exit("GALAT: %s sudah ada. Pakai --out nama lain atau --force." % a.out)
    os.makedirs(a.logdir, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    log_txt = os.path.join(a.logdir, "%s_%s.txt" % (a.tag, stamp))
    log_csv = os.path.join(a.logdir, "%s_%s.csv" % (a.tag, stamp))
    logf = open(log_txt, "a")

    def log(msg):
        print(msg)
        logf.write(msg + "\n")
        logf.flush()

    if a.seed is not None:
        random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)

    cam = kamera.load_camera()
    K, T = cam["K"], cam["T"]
    names = sorted(f for f in os.listdir(a.images) if f.lower().endswith(".jpg"))
    train_ids, val_ids = split_ids(names)
    n_train_full, n_val_full = len(train_ids), len(val_ids)
    if a.max_train: train_ids = train_ids[:a.max_train]
    if a.max_val: val_ids = val_ids[:a.max_val]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log("torch %s | torchvision %s | numpy %s | device %s" % (torch.__version__, torchvision.__version__, np.__version__, device))
    log("argumen: %s" % vars(a))
    log("gambar: %d | pembagian penuh latih/uji: %d/%d | dipakai: %d/%d" % (len(names), n_train_full, n_val_full, len(train_ids), len(val_ids)))
    log("latih %s .. %s | uji %s .. %s" % (train_ids[0], train_ids[-1], val_ids[0], val_ids[-1]))
    log("kamera: f(720p)=%.2f h=%.3f m | y_min=%s | shuffle=%s | aug_fix=%s" % (cam["f"], cam["h_m"], a.y_min, a.shuffle, a.aug_fix))
    log("log: %s | %s" % (log_txt, log_csv))

    train_ds = Data(train_ids, a.images, a.labels, K, T, a.y_min, True, a.aug_fix)
    val_ds = Data(val_ids, a.images, a.labels, K, T, a.y_min, False, False)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=a.batch_size, shuffle=a.shuffle)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=a.batch_size, shuffle=False)

    model = centernet(pretrained=not (a.no_pretrained or a.init)).to(device)
    if a.init:
        model.load_state_dict(torch.load(a.init, map_location=device))
        log("bobot awal dimuat dari %s" % a.init)
    optimizer = optim.Adam(model.parameters(), lr=a.lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    cols = ["epoch", "lr", "train_loss", "train_pos", "train_neg", "train_off", "train_whl",
            "val_loss", "val_pos", "val_neg", "val_off", "val_whl", "detik_epoch", "detik_per_iter_latih"]
    with open(log_csv, "w", newline="") as fcsv:
        csv.writer(fcsv).writerow(cols)

    best = float("inf")
    for epoch in range(a.epochs):
        te = time.time()
        lr = optimizer.param_groups[0]["lr"]
        log("epoch %d/%d | lr %g" % (epoch + 1, a.epochs, lr))
        tr, sec_iter = run_pass(model, train_loader, device, optimizer, "latih")
        scheduler.step()
        va, _ = run_pass(model, val_loader, device, None, "uji")
        dt = time.time() - te
        log("  latih: " + " ".join("%s=%.4f" % kv for kv in tr.items()))
        log("  uji  : " + " ".join("%s=%.4f" % kv for kv in va.items()))
        log("  waktu epoch %.1f s | %.2f s/iterasi latih" % (dt, sec_iter))
        with open(log_csv, "a", newline="") as fcsv:
            csv.writer(fcsv).writerow([epoch + 1, lr, tr["loss"], tr["pos"], tr["neg"], tr["off"], tr["whl"],
                                       va["loss"], va["pos"], va["neg"], va["off"], va["whl"], round(dt, 1), round(sec_iter, 3)])
        if va["loss"] < best:
            best = va["loss"]
            torch.save(model.state_dict(), a.out)
            log("  val loss membaik ke %.4f -> disimpan %s" % (best, a.out))
    log("selesai. val loss terbaik %.4f" % best)




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/train_session0_right.py
if __name__ == "__main__":
    main()


### `dekode.py`

Kebalikan dari target: mengubah keluaran jaringan menjadi kotak 3D (dan kotak 2D untuk evaluasi). Bila dijalankan langsung, melakukan uji bolak-balik label → target → dekode → label.

*Asli: `~/projects/latih_detektor/dekode.py`, 157 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/dekode.py
"""dekode.py - dekode keluaran jaringan menjadi kotak 3D dan kotak 2D (numpy saja, tanpa PyTorch).
Salinan dari centernet-3d-bbox.ipynb (commit 3fa494cd): select (baris 154-174), pred2box (175-204),
bagian hitung showbox (206-233), urutan dekode pengujian (664-679: sigmoid -> select -> pred2box),
dan 8 sudut Box.project di utils.py (baris 155-205) tanpa menggambar.
Perubahan atas acuan: sigmoid ditulis dengan numpy (0.5*(1+tanh(x/2)), identik secara matematis,
tanpa galat limpahan); select bekerja pada salinan agar heatmap masukan tidak berubah;
iou_2d dan kotak 2D ditambahkan (acuan hanya menggambar).
Pemakaian uji: python3 dekode.py [jumlah_frame]   (uji bolak-balik pada label asli)
"""
import os
import sys
import numpy as np

REPO = os.path.expanduser("~/projects/acuan/3d_bbox_highway")
sys.path.insert(0, REPO)
from utils import get_position_ground, project  # noqa: E402

MODEL_SCALE = 32




#### fungsi `sigmoid`

Bersama: fungsi `select`; fungsi `pred2box`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/dekode.py
def sigmoid(x):
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(x, dtype=np.float64)))


def select(hm, threshold):
    pred = hm > threshold
    pred_centers = np.argwhere(pred)
    for i, ci in enumerate(pred_centers):
        for j in range(i + 1, len(pred_centers)):
            cj = pred_centers[j]
            if np.linalg.norm(ci - cj) <= 2:
                score_i = hm[ci[0], ci[1]]
                score_j = hm[cj[0], cj[1]]
                if score_i > score_j:
                    hm[cj[0], cj[1]] = 0
                else:
                    hm[ci[0], ci[1]] = 0
    return hm


def pred2box(hm, offset, regr, thresh=0.99):
    pred = hm > thresh
    pred_center = np.where(hm > thresh)
    pred_r = regr[:, pred].T
    boxes = []
    scores = hm[pred]
    pred_center = np.asarray(pred_center).T
    for (center, b) in zip(pred_center, pred_r):
        offset_xy = offset[:, center[0], center[1]]
        arr = np.array([(center[1] + offset_xy[0]) * MODEL_SCALE, (center[0] + offset_xy[1]) * MODEL_SCALE,
                        b[0], b[1], b[2]])
        boxes.append(arr)
    return np.asarray(boxes), scores




#### fungsi `box_corners_2d` — 8 sudut Box(x, z, w, h, l) di piksel gambar (y dari atas), tanpa dibulatkan.

Bersama: fungsi `bbox_2d`; fungsi `iou_2d`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/dekode.py
def box_corners_2d(x, z, w, h, l, to_world_from_camera, K, img_height):
    """8 sudut Box(x, z, w, h, l) di piksel gambar (y dari atas), tanpa dibulatkan."""
    to_camera_from_world = np.linalg.inv(to_world_from_camera)
    pts = []
    for dx in (-w / 2, w / 2):
        for dy in (0.0, h):
            for dz in (-l / 2, l / 2):
                p = project(np.array([x + dx, dy, z + dz]), to_camera_from_world, K)
                p[1] = img_height - p[1]
                pts.append(p)
    return np.array(pts)


def bbox_2d(corners):
    return (float(corners[:, 0].min()), float(corners[:, 1].min()),
            float(corners[:, 0].max()), float(corners[:, 1].max()))


def iou_2d(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0




#### fungsi `decode` — hm: logit (23,40) bila apply_sigmoid. offset (2,23,40), whl (3,23,40). Mengembalikan daftar dict.

Bersama: fungsi `selftest` — Uji bolak-balik: label -> target (target.py) -> dekode -> harus kembali ke label.

In [ ]:
%%writefile -a kode_notebook/latih_detektor/dekode.py
def decode(hm, offset, whl, K, to_world_from_camera, img_height=720, thresh=0.1, apply_sigmoid=True):
    """hm: logit (23,40) bila apply_sigmoid. offset (2,23,40), whl (3,23,40). Mengembalikan daftar dict."""
    hm = sigmoid(hm) if apply_sigmoid else np.array(hm, dtype=np.float64)
    hm = select(hm, thresh)
    boxes, scores = pred2box(hm, offset, whl, thresh=thresh)
    out = []
    for box, s in zip(boxes, scores):
        pos = get_position_ground(box[0], box[1], K, to_world_from_camera, img_height)
        depth = float(np.linalg.norm(pos - to_world_from_camera[:3, 3]))
        x, z = float(pos[0]), float(pos[2])
        w, h, l = float(box[2] * depth), float(box[3] * depth), float(box[4] * depth)
        corners = box_corners_2d(x, z, w, h, l, to_world_from_camera, K, img_height)
        out.append(dict(px=float(box[0]), py=float(box[1]), x=x, z=z, w=w, h=h, l=l,
                        depth=depth, score=float(s), bbox2d=bbox_2d(corners)))
    return out


def selftest(n_frames=300):
    """Uji bolak-balik: label -> target (target.py) -> dekode -> harus kembali ke label.
    Objek yang sel pusatnya tertimpa isian 3x3 objek lain yang digambar belakangan (atau sel pusat
    yang sama) dipisahkan, karena targetnya memang menjadi milik objek lain (perilaku acuan)."""
    import glob
    import pandas as pd
    import kamera
    import target as tg
    cam = kamera.load_camera()
    K, T = cam["K"], cam["T"]
    folder = os.path.expanduser("~/projects/data_session0_right/converted/labels")
    files = sorted(glob.glob(os.path.join(folder, "*.txt")))[:n_frames]
    n_near = n_ok = n_over = n_sel = n_hilang = 0
    n_far = zero_far = 0
    max_pos = max_rel = max_iou_gap = 0.0
    for fp in files:
        df = pd.read_csv(fp)
        hm, off, regr = tg.make_hm_offset_regr(df, K, T, 150)
        dets = decode(hm, off, regr, K, T, 720, thresh=0.99, apply_sigmoid=False)
        rows = [(int(c["y"] / 32), int(c["x"] / 32), c) for _, c in df.iterrows()]
        for i, (ry, rx, c) in enumerate(rows):
            if c["y"] < 150:
                n_far += 1
                d = [q for q in dets if int(q["py"] / 32) == ry and int(q["px"] / 32) == rx]
                if d and d[0]["w"] == 0 and d[0]["h"] == 0 and d[0]["l"] == 0:
                    zero_far += 1
                continue
            n_near += 1
            ditimpa = any(j > i and cj["y"] >= 150 and abs(ry2 - ry) <= 1 and abs(rx2 - rx) <= 1
                          for j, (ry2, rx2, cj) in enumerate(rows))
            if ditimpa:
                n_over += 1
                continue
            d = [q for q in dets if abs(q["px"] - c["x"]) < 1e-3 and abs(q["py"] - c["y"]) < 1e-3]
            if not d:
                if any(j != i and ((ry2 - ry) ** 2 + (rx2 - rx) ** 2) ** 0.5 <= 2
                       for j, (ry2, rx2, cj) in enumerate(rows)):
                    n_sel += 1
                else:
                    n_hilang += 1
                continue
            q = d[0]
            n_ok += 1
            max_pos = max(max_pos, abs(q["px"] - c["x"]), abs(q["py"] - c["y"]))
            for k in ("w", "h", "l"):
                max_rel = max(max_rel, abs(q[k] - c[k]) / c[k])
            pos = get_position_ground(c["x"], c["y"], K, T, 720)
            ref = bbox_2d(box_corners_2d(pos[0], pos[2], c["w"], c["h"], c["l"], T, K, 720))
            max_iou_gap = max(max_iou_gap, 1.0 - iou_2d(q["bbox2d"], ref))
    print("frame: %d | objek y>=150: %d" % (len(files), n_near))
    print("  tidak tertimpa: %d | kembali ke label: %d | ditekan select (pusat lain berjarak sel <= 2): %d | hilang tanpa sebab: %d"
          % (n_near - n_over, n_ok, n_sel, n_hilang))
    print("  tertimpa isian 3x3 objek lain yang digambar belakangan: %d (%.1f%%)" % (n_over, 100.0 * n_over / max(n_near, 1)))
    print("  galat posisi piksel maks: %.2e | galat relatif w,h,l maks: %.2e | 1-IoU maks: %.2e" % (max_pos, max_rel, max_iou_gap))
    print("objek y<150: %d | didekode dengan ukuran nol (w=h=l=0): %d" % (n_far, zero_far))




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/dekode.py
if __name__ == "__main__":
    selftest(int(sys.argv[1]) if len(sys.argv) > 1 else 300)


### `prediksi.py`

Menjalankan detektor pada frame uji dan menyimpan prediksinya ke berkas `.pkl` untuk dievaluasi.

*Asli: `~/projects/latih_detektor/prediksi.py`, 77 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/prediksi.py
#!/usr/bin/env python3
"""prediksi.py - jalankan model pada gambar uji (atau latih) dan simpan hasil dekodenya ke file pickle.
Urutan dekode mengikuti notebook (baris 646-679): gambar BGR tanpa penukaran kanal -> model -> sigmoid
-> select -> pred2box -> ukuran x kedalaman. Ambang bawaan 0,1 (notebook baris 641).
Bobot dimuat dengan map_location='cpu' (bobot penulis disimpan dari GPU).
Pemakaian: python3 prediksi.py --weights ~/projects/uji_kecil.pth --max 6 --tag uji
"""
import argparse
import os
import pickle
import sys
import time

import cv2
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import kamera  # noqa: E402
import target as tg  # noqa: E402
import dekode  # noqa: E402
from train_session0_right import centernet, split_ids  # noqa: E402




#### fungsi `baca_gambar` — Sama dengan jalur data uji (target.load_item, train=False): BGR, resize 1280x720, Normalize, (3,H,W).

Bersama: fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/prediksi.py
def baca_gambar(path):
    """Sama dengan jalur data uji (target.load_item, train=False): BGR, resize 1280x720, Normalize, (3,H,W)."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.resize(img, (tg.INPUT_W, tg.INPUT_H))
    return tg.Normalize()(img).transpose([2, 0, 1])


def main(argv=None):
    p = argparse.ArgumentParser()
    conv = os.path.expanduser("~/projects/data_session0_right/converted")
    p.add_argument("--images", default=os.path.join(conv, "images"))
    p.add_argument("--weights", required=True)
    p.add_argument("--split", choices=["val", "train"], default="val")
    p.add_argument("--max", type=int, default=None)
    p.add_argument("--thresh", type=float, default=0.1)
    p.add_argument("--tag", default="pred")
    p.add_argument("--outdir", default=os.path.expanduser("~/projects/hasil/prediksi_detektor"))
    a = p.parse_args(argv)

    cam = kamera.load_camera()
    K, T = cam["K"], cam["T"]
    names = sorted(f for f in os.listdir(a.images) if f.lower().endswith(".jpg"))
    tr, va = split_ids(names)
    ids = va if a.split == "val" else tr
    if a.max:
        ids = ids[:a.max]

    model = centernet(pretrained=False)
    model.load_state_dict(torch.load(a.weights, map_location="cpu"))
    model.eval()

    os.makedirs(a.outdir, exist_ok=True)
    out_path = os.path.join(a.outdir, "%s_%s.pkl" % (a.tag, a.split))
    dets, t0 = {}, time.time()
    for k, nm in enumerate(ids):
        x = torch.from_numpy(np.ascontiguousarray(baca_gambar(os.path.join(a.images, nm)))).float().unsqueeze(0)
        with torch.no_grad():
            hm, offset, whl = model(x)
        hm = hm.cpu().numpy().squeeze(0).squeeze(0)
        offset = offset.cpu().numpy().squeeze(0)
        whl = whl.cpu().numpy().squeeze(0)
        dets[nm] = dekode.decode(hm, offset, whl, K, T, 720, thresh=a.thresh, apply_sigmoid=True)
        if k < 3 or (k + 1) % 50 == 0 or k + 1 == len(ids):
            print("%s | deteksi: %d | %d/%d | %.2f s/gambar" % (nm, len(dets[nm]), k + 1, len(ids), (time.time() - t0) / (k + 1)))
    with open(out_path, "wb") as f:
        pickle.dump(dict(args=vars(a), names=ids, dets=dets), f)
    print("tersimpan:", out_path, "| gambar:", len(ids), "| total deteksi:", sum(len(v) for v in dets.values()))




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/prediksi.py
if __name__ == "__main__":
    main()


### `evaluasi.py`

Menghitung metrik deteksi (AP, presisi, recall) dari berkas prediksi `.pkl` terhadap anotasi.

*Asli: `~/projects/latih_detektor/evaluasi.py`, 197 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/evaluasi.py
#!/usr/bin/env python3
"""evaluasi.py - hitung metrik dari file pickle hasil prediksi.py (numpy + pandas saja, tanpa PyTorch).

Masukan : ~/projects/hasil/prediksi_detektor/<tag>_val.pkl  (dict: args, names, dets)
          label asli di converted/labels/image_NNNN.txt (kolom x,y,w,h,l; piksel 720p, meter)
Kotak 2D ground-truth dibuat dengan cara yang sama dengan uji bolak-balik di dekode.py
(get_position_ground -> 8 sudut Box -> min/maks), jadi pembanding dan prediksi memakai rumus yang sama.

Aturan (semuanya pilihan kita, bukan bawaan notebook; notebook tidak punya evaluasi angka):
 1. Objek ground-truth "dekat" = y >= y_min (default 150), yaitu objek yang punya target ukuran saat latihan.
 2. Deteksi dibagi menurut posisi pusatnya, mengikuti aturan latihan: py < y_min = "zona jauh"
    (tidak punya target ukuran, tidak dihitung sebagai FP, dilaporkan terpisah); py >= y_min = zona
    dekat. Di zona dekat, deteksi dengan w, h, atau l <= 0 adalah kotak tidak valid dan dihitung FP
    (kepala ukuran tidak punya aktivasi akhir, jadi keluarannya bisa negatif; ini terbukti dari pkl uji).
 3. Pencocokan: deteksi diurutkan dari skor tertinggi; tiap deteksi dipasangkan dengan ground-truth
    yang belum terpakai dan IoU 2D tertingginya, dianggap TP bila IoU >= ambang (default 0,5).
 4. Objek jauh (y < y_min) tidak punya target ukuran, jadi hanya dilaporkan "pusat terdeteksi":
    ada deteksi (ukuran apa pun) di sel 32x32 yang sama dengan pusat label.
 5. AP = luas di bawah kurva presisi-recall dengan interpolasi semua titik (gaya PASCAL VOC,
    dari ingatan, belum dicek ke makalahnya) pada deteksi berukuran.
Pemakaian: python3 evaluasi.py --pred ~/projects/hasil/prediksi_detektor/uji_val.pkl
"""
import argparse
import csv
import os
import pickle
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import dekode  # noqa: E402
from dekode import iou_2d, bbox_2d, box_corners_2d  # noqa: E402

SCALE = 32  # dekode.MODEL_SCALE




#### fungsi `muat_gt` — Mengembalikan (dekat, jauh): daftar dict objek label. 'dekat' punya depth dan bbox2d.

Bersama: fungsi `berukuran`; fungsi `cocokkan_frame` — dets: deteksi berukuran. Mengembalikan daftar (skor, tp, indeks_deteksi, indeks_gt, iou).

In [ ]:
%%writefile -a kode_notebook/latih_detektor/evaluasi.py
def muat_gt(path, K, T, y_min):
    """Mengembalikan (dekat, jauh): daftar dict objek label. 'dekat' punya depth dan bbox2d."""
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return [], []
    dekat, jauh = [], []
    for _, c in df.iterrows():
        g = dict(x=float(c["x"]), y=float(c["y"]), w=float(c["w"]), h=float(c["h"]), l=float(c["l"]))
        if g["y"] >= y_min:
            pos = dekode.get_position_ground(g["x"], g["y"], K, T, 720)
            g["depth"] = float(np.linalg.norm(pos - T[:3, 3]))
            g["bbox2d"] = bbox_2d(box_corners_2d(pos[0], pos[2], g["w"], g["h"], g["l"], T, K, 720))
            dekat.append(g)
        else:
            jauh.append(g)
    return dekat, jauh


def berukuran(d):
    return d["w"] > 0 and d["h"] > 0 and d["l"] > 0


def cocokkan_frame(dets, gts, thr):
    """dets: deteksi berukuran. Mengembalikan daftar (skor, tp, indeks_deteksi, indeks_gt, iou)."""
    urut = sorted(range(len(dets)), key=lambda i: -dets[i]["score"])
    terpakai, hasil = set(), []
    for i in urut:
        terbaik, bj = 0.0, None
        for j, g in enumerate(gts):
            if j in terpakai:
                continue
            v = iou_2d(dets[i]["bbox2d"], g["bbox2d"])
            if v > terbaik:
                terbaik, bj = v, j
        if bj is not None and terbaik >= thr:
            terpakai.add(bj)
            hasil.append((dets[i]["score"], True, i, bj, terbaik))
        else:
            hasil.append((dets[i]["score"], False, i, None, terbaik))
    return hasil




#### fungsi `ap_semua_titik`

Bersama: fungsi `med`; fungsi `evaluasi`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/evaluasi.py
def ap_semua_titik(skor, tp, n_gt):
    if n_gt == 0 or len(skor) == 0:
        return 0.0
    o = np.argsort(-np.asarray(skor), kind="stable")
    tp = np.asarray(tp, dtype=float)[o]
    fp = 1.0 - tp
    ctp, cfp = np.cumsum(tp), np.cumsum(fp)
    rec = ctp / n_gt
    prec = ctp / np.maximum(ctp + cfp, 1e-12)
    rec = np.concatenate(([0.0], rec, [1.0]))
    prec = np.concatenate(([0.0], prec, [0.0]))
    for k in range(len(prec) - 2, -1, -1):
        prec[k] = max(prec[k], prec[k + 1])
    idx = np.where(rec[1:] != rec[:-1])[0]
    return float(np.sum((rec[idx + 1] - rec[idx]) * prec[idx + 1]))


def med(x):
    return float(np.median(x)) if len(x) else float("nan")


def evaluasi(pkl, labels, K, T, y_min=150, iou_thr=0.5, ambang_skor=(0.1, 0.3, 0.5, 0.7, 0.9)):
    dets_all, names = pkl["dets"], pkl["names"]
    n_gt = 0
    rek = []          # per deteksi berukuran: skor, tp, galat
    nol_skor = []     # skor deteksi di zona jauh (py < y_min)
    tv_skor = []      # skor deteksi zona dekat dengan kotak tidak valid (w, h, atau l <= 0)
    jauh_skor = []    # skor terbaik per objek jauh (0 bila tidak ada deteksi di selnya)
    for nm in names:
        dekat, jauh = muat_gt(os.path.join(labels, nm[:-4] + ".txt"), K, T, y_min)
        n_gt += len(dekat)
        dets = dets_all[nm]
        zona_jauh = [d for d in dets if d["py"] < y_min]
        dekat_det = [d for d in dets if d["py"] >= y_min]
        ukur = [d for d in dekat_det if berukuran(d)]
        tak_valid = [d for d in dekat_det if not berukuran(d)]
        nol_skor += [d["score"] for d in zona_jauh]
        tv_skor += [d["score"] for d in tak_valid]
        rek += [dict(skor=d["score"], tp=False, iou=0.0) for d in tak_valid]
        for (s, tp, i, j, iou) in cocokkan_frame(ukur, dekat, iou_thr):
            r = dict(skor=s, tp=tp, iou=iou)
            if tp:
                g, d = dekat[j], ukur[i]
                r.update(ew=abs(d["w"] - g["w"]), eh=abs(d["h"] - g["h"]), el=abs(d["l"] - g["l"]),
                         rw=abs(d["w"] - g["w"]) / g["w"], rh=abs(d["h"] - g["h"]) / g["h"],
                         rl=abs(d["l"] - g["l"]) / g["l"], ed=abs(d["depth"] - g["depth"]),
                         rd=abs(d["depth"] - g["depth"]) / g["depth"])
            rek.append(r)
        for g in jauh:
            sel = (int(g["y"] / SCALE), int(g["x"] / SCALE))
            ss = [d["score"] for d in dets if (int(d["py"] / SCALE), int(d["px"] / SCALE)) == sel]
            jauh_skor.append(max(ss) if ss else 0.0)
    ap = ap_semua_titik([r["skor"] for r in rek], [r["tp"] for r in rek], n_gt)
    tabel = []
    for t in ambang_skor:
        pilih = [r for r in rek if r["skor"] >= t]
        tp = [r for r in pilih if r["tp"]]
        fp = len(pilih) - len(tp)
        fn = n_gt - len(tp)
        baris = dict(skor_min=t, TP=len(tp), FP=fp, FN=fn,
                     presisi=len(tp) / max(len(tp) + fp, 1), recall=len(tp) / max(n_gt, 1),
                     deteksi_zona_jauh=sum(1 for s in nol_skor if s >= t),
                     kotak_tak_valid=sum(1 for s in tv_skor if s >= t),
                     jauh_pusat_terdeteksi=sum(1 for s in jauh_skor if s >= t),
                     med_iou_tp=med([r["iou"] for r in tp]),
                     med_galat_w_m=med([r["ew"] for r in tp]), med_galat_h_m=med([r["eh"] for r in tp]),
                     med_galat_l_m=med([r["el"] for r in tp]), med_galat_depth_m=med([r["ed"] for r in tp]),
                     med_rel_w=med([r["rw"] for r in tp]), med_rel_h=med([r["rh"] for r in tp]),
                     med_rel_l=med([r["rl"] for r in tp]), med_rel_depth=med([r["rd"] for r in tp]))
        tabel.append(baris)
    return dict(n_frame=len(names), n_gt_dekat=n_gt, n_gt_jauh=len(jauh_skor),
                n_det_dekat=len(rek), n_det_zona_jauh=len(nol_skor), n_det_tak_valid=len(tv_skor),
                ap=ap, tabel=tabel)




#### fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/evaluasi.py
def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--pred", required=True)
    p.add_argument("--labels", default=os.path.expanduser("~/projects/data_session0_right/converted/labels"))
    p.add_argument("--y-min", type=float, default=150)
    p.add_argument("--iou", type=float, default=0.5)
    p.add_argument("--csv", default=None)
    a = p.parse_args(argv)

    import kamera
    cam = kamera.load_camera()
    with open(a.pred, "rb") as f:
        pkl = pickle.load(f)
    h = evaluasi(pkl, a.labels, cam["K"], cam["T"], a.y_min, a.iou)

    print("berkas: %s | frame: %d" % (a.pred, h["n_frame"]))
    print("objek dekat (y>=%g): %d | objek jauh: %d" % (a.y_min, h["n_gt_dekat"], h["n_gt_jauh"]))
    print("deteksi zona dekat: %d (di antaranya kotak tidak valid: %d) | deteksi zona jauh (py<%g): %d"
          % (h["n_det_dekat"], h["n_det_tak_valid"], a.y_min, h["n_det_zona_jauh"]))
    print("AP (IoU 2D >= %.2f, deteksi zona dekat): %.4f" % (a.iou, h["ap"]))
    print("%8s %6s %6s %6s %8s %8s | %6s %6s %6s | %7s" %
          ("skor>=", "TP", "FP", "FN", "presisi", "recall", "tkvld", "zjauh", "jauhOK", "medIoU"))
    for b in h["tabel"]:
        print("%8.2f %6d %6d %6d %8.3f %8.3f | %6d %6d %6d | %7.3f" %
              (b["skor_min"], b["TP"], b["FP"], b["FN"], b["presisi"], b["recall"],
               b["kotak_tak_valid"], b["deteksi_zona_jauh"], b["jauh_pusat_terdeteksi"], b["med_iou_tp"]))
    print("galat ukuran pada TP, median (m): lihat CSV; ringkas untuk skor>=%.2f:" % h["tabel"][0]["skor_min"])
    b = h["tabel"][0]
    print("  w %.2f m (%.0f%%) | h %.2f m (%.0f%%) | l %.2f m (%.0f%%) | kedalaman %.2f m (%.1f%%)" %
          (b["med_galat_w_m"], 100 * b["med_rel_w"], b["med_galat_h_m"], 100 * b["med_rel_h"],
           b["med_galat_l_m"], 100 * b["med_rel_l"], b["med_galat_depth_m"], 100 * b["med_rel_depth"]))
    out = a.csv or os.path.splitext(a.pred)[0] + "_evaluasi.csv"
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(h["tabel"][0].keys()))
        w.writeheader()
        w.writerows(h["tabel"])
    print("tersimpan:", out)




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/evaluasi.py
if __name__ == "__main__":
    main()


### `analisis_galat.py`

Menguraikan FN dan FP dari berkas prediksi menurut jarak dan penyebabnya.

*Asli: `~/projects/latih_detektor/analisis_galat.py`, 205 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_detektor/analisis_galat.py
#!/usr/bin/env python3
"""analisis_galat.py - uraikan FN dan FP dari pkl hasil prediksi.py menurut jarak dan penyebab (numpy + pandas saja).

Memakai fungsi pencocokan yang sama dengan evaluasi.py (muat_gt, cocokkan_frame), jadi TP/FP/FN di sini
harus sama dengan tabel evaluasi.py pada ambang skor yang sama (bawaan 0,1).

Kelompok jarak = jarak kamera ke titik tanah objek (meter), dibagi N kelompok dengan jumlah objek sama
(bawaan 5), batasnya dicetak. FP dikelompokkan dengan jarak deteksinya sendiri; FP di luar rentang jarak
objek label masuk kelompok paling dekat atau paling jauh.

Penyebab (semuanya aturan kita sendiri, bukan bawaan notebook):
 FN (objek zona dekat yang tidak tercocokkan):
   tanpa_deteksi : tidak ada deteksi dalam radius 2 sel (jarak yang sama dengan select) dari pusat label.
                   Dari pkl tidak bisa dibedakan antara skor heatmap di bawah ambang dan ditekan select,
                   karena pkl hanya menyimpan hasil sesudah select.
   lokalisasi    : ada deteksi dalam radius 2 sel, tetapi IoU 2D terbaiknya < ambang IoU.
   berimpit      : ada deteksi ber-IoU >= ambang, tetapi deteksi itu sudah dipakai objek label lain
                   (dua kendaraan yang kotaknya saling menutupi).
 FP (deteksi zona dekat yang tidak tercocokkan):
   duplikat      : IoU >= ambang dengan objek yang sudah dicocokkan deteksi lain.
   lokalisasi    : IoU terbaik dengan objek label antara 0,1 dan ambang.
   latar         : IoU terbaik dengan semua objek label < 0,1.
   tak_valid     : w, h, atau l <= 0.
 Objek jauh (y < y_min): dilaporkan apakah pusatnya terdeteksi (deteksi apa pun di sel 32x32 yang sama,
 sama dengan evaluasi.py) dan seberapa besar ukuran deteksinya; target latihannya adalah ukuran nol.
Pemakaian: python3 analisis_galat.py --pred ~/projects/hasil/prediksi_detektor/latih3ep_val.pkl
"""
import argparse
import csv
import os
import pickle
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from dekode import iou_2d  # noqa: E402
from evaluasi import muat_gt, berukuran, cocokkan_frame, med, SCALE  # noqa: E402




#### fungsi `sel_of`

Bersama: fungsi `radius_sel`; fungsi `kumpulkan`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/analisis_galat.py
def sel_of(y, x):
    return (int(y / SCALE), int(x / SCALE))


def radius_sel(d, g, radius=2):
    cy, cx = sel_of(g["y"], g["x"])
    dy, dx = sel_of(d["py"], d["px"])
    return ((cy - dy) ** 2 + (cx - dx) ** 2) ** 0.5 <= radius


def kumpulkan(pkl, labels, K, T, y_min, iou_thr, skor_min):
    gt_rec, fp_rec, far_rec = [], [], []
    far_det_tanpa_label = 0
    for nm in pkl["names"]:
        dekat, jauh = muat_gt(os.path.join(labels, nm[:-4] + ".txt"), K, T, y_min)
        dets = [d for d in pkl["dets"][nm] if d["score"] >= skor_min]
        zona_jauh = [d for d in dets if d["py"] < y_min]
        dekat_det = [d for d in dets if d["py"] >= y_min]
        ukur = [d for d in dekat_det if berukuran(d)]
        tak_valid = [d for d in dekat_det if not berukuran(d)]
        hasil = cocokkan_frame(ukur, dekat, iou_thr)
        gt_cocok = {j: (i, iou) for (_, tp, i, j, iou) in hasil if tp}
        det_cocok = {i for (_, tp, i, j, iou) in hasil if tp}
        for j, g in enumerate(dekat):
            r = dict(frame=nm, x=g["x"], y=g["y"], depth=g["depth"], skor="")
            if j in gt_cocok:
                i, iou = gt_cocok[j]
                d = ukur[i]
                r.update(status="TP", sebab="", iou=iou, skor=d["score"],
                         rw=abs(d["w"] - g["w"]) / g["w"], rh=abs(d["h"] - g["h"]) / g["h"],
                         rl=abs(d["l"] - g["l"]) / g["l"], ed=abs(d["depth"] - g["depth"]))
            else:
                best = max((iou_2d(d["bbox2d"], g["bbox2d"]) for d in ukur), default=0.0)
                if best >= iou_thr:
                    sebab = "berimpit"
                elif any(radius_sel(d, g) for d in dekat_det):
                    sebab = "lokalisasi"
                else:
                    sebab = "tanpa_deteksi"
                r.update(status="FN", sebab=sebab)
            gt_rec.append(r)
        for i, d in enumerate(ukur):
            if i in det_cocok:
                continue
            best = max((iou_2d(d["bbox2d"], g["bbox2d"]) for g in dekat), default=0.0)
            sebab = "duplikat" if best >= iou_thr else ("lokalisasi" if best >= 0.1 else "latar")
            fp_rec.append(dict(frame=nm, x=d["px"], y=d["py"], depth=d["depth"], skor=d["score"], sebab=sebab))
        for d in tak_valid:
            fp_rec.append(dict(frame=nm, x=d["px"], y=d["py"], depth=d["depth"], skor=d["score"], sebab="tak_valid"))
        sel_jauh = {sel_of(g["y"], g["x"]) for g in jauh}
        for g in jauh:
            c = [d for d in dets if sel_of(d["py"], d["px"]) == sel_of(g["y"], g["x"])]
            if c:
                d = max(c, key=lambda q: q["score"])
                far_rec.append(dict(terdeteksi=True, w=d["w"], h=d["h"], l=d["l"], skor=d["score"]))
            else:
                far_rec.append(dict(terdeteksi=False))
        far_det_tanpa_label += sum(1 for d in zona_jauh if sel_of(d["py"], d["px"]) not in sel_jauh)
    return gt_rec, fp_rec, far_rec, far_det_tanpa_label




#### fungsi `analisis`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/analisis_galat.py
def analisis(pkl, labels, K, T, y_min=150, iou_thr=0.5, skor_min=0.1, n_bin=5, nol_m=1.0):
    gt_rec, fp_rec, far_rec, far_tanpa = kumpulkan(pkl, labels, K, T, y_min, iou_thr, skor_min)
    dep = np.array([r["depth"] for r in gt_rec]) if gt_rec else np.array([0.0])
    edges = np.quantile(dep, np.linspace(0, 1, n_bin + 1))
    inner = edges[1:-1]

    def bin_of(d):
        return int(np.searchsorted(inner, d, side="right"))

    tabel = []
    for b in range(n_bin):
        G = [r for r in gt_rec if bin_of(r["depth"]) == b]
        tp = [r for r in G if r["status"] == "TP"]
        fn = [r for r in G if r["status"] == "FN"]
        F = [r for r in fp_rec if bin_of(r["depth"]) == b]
        tabel.append(dict(
            kelompok=b + 1, dari_m=float(edges[b]), sampai_m=float(edges[b + 1]),
            n_gt=len(G), TP=len(tp), FN=len(fn), recall=len(tp) / max(len(G), 1),
            fn_tanpa_deteksi=sum(1 for r in fn if r["sebab"] == "tanpa_deteksi"),
            fn_lokalisasi=sum(1 for r in fn if r["sebab"] == "lokalisasi"),
            fn_berimpit=sum(1 for r in fn if r["sebab"] == "berimpit"),
            FP=len(F),
            fp_duplikat=sum(1 for r in F if r["sebab"] == "duplikat"),
            fp_lokalisasi=sum(1 for r in F if r["sebab"] == "lokalisasi"),
            fp_latar=sum(1 for r in F if r["sebab"] == "latar"),
            fp_tak_valid=sum(1 for r in F if r["sebab"] == "tak_valid"),
            med_iou_tp=med([r["iou"] for r in tp]),
            med_rel_w=med([r["rw"] for r in tp]), med_rel_h=med([r["rh"] for r in tp]),
            med_rel_l=med([r["rl"] for r in tp]), med_galat_depth_m=med([r["ed"] for r in tp])))
    det = [r for r in far_rec if r["terdeteksi"]]
    mutlak = [max(abs(r["w"]), abs(r["h"]), abs(r["l"])) for r in det]
    jauh = dict(n=len(far_rec), terdeteksi=len(det), tanpa_label=far_tanpa,
                med_abs_w=med([abs(r["w"]) for r in det]), med_abs_h=med([abs(r["h"]) for r in det]),
                med_abs_l=med([abs(r["l"]) for r in det]),
                dekat_nol=sum(1 for v in mutlak if v < nol_m), nol_m=nol_m)
    kasus = [dict(jenis="FN", **{k: r[k] for k in ("sebab", "frame", "x", "y", "depth", "skor")})
             for r in gt_rec if r["status"] == "FN"]
    kasus += [dict(jenis="FP", **{k: r[k] for k in ("sebab", "frame", "x", "y", "depth", "skor")}) for r in fp_rec]
    return dict(n_frame=len(pkl["names"]), n_gt=len(gt_rec), edges=edges, tabel=tabel, jauh=jauh, kasus=kasus,
                n_tp=sum(1 for r in gt_rec if r["status"] == "TP"), n_fp=len(fp_rec))




#### fungsi `cetak`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/analisis_galat.py
def cetak(h, skor_min, iou_thr):
    n_fn = h["n_gt"] - h["n_tp"]
    print("frame: %d | objek dekat: %d | skor >= %.2f | IoU >= %.2f" % (h["n_frame"], h["n_gt"], skor_min, iou_thr))
    print("TP %d | FN %d | FP %d (harus sama dengan baris skor %.2f di evaluasi.py)" % (h["n_tp"], n_fn, h["n_fp"], skor_min))
    print("jarak objek dekat (m): min %.1f | maks %.1f" % (h["edges"][0], h["edges"][-1]))
    kk = ("tanpa_deteksi", "lokalisasi", "berimpit")
    fk = ("duplikat", "lokalisasi", "latar", "tak_valid")
    print("\nPenyebab FN: " + " | ".join("%s %d (%.0f%%)" % (s, sum(b["fn_" + s] for b in h["tabel"]),
          100.0 * sum(b["fn_" + s] for b in h["tabel"]) / max(n_fn, 1)) for s in kk))
    print("Penyebab FP: " + " | ".join("%s %d (%.0f%%)" % (s, sum(b["fp_" + s] for b in h["tabel"]),
          100.0 * sum(b["fp_" + s] for b in h["tabel"]) / max(h["n_fp"], 1)) for s in fk))
    print("\n%2s %13s %5s %5s %5s %7s | %8s %8s %8s | %5s %5s %5s %5s %5s | %6s" %
          ("kl", "jarak (m)", "GT", "TP", "FN", "recall", "tnpdtk", "lokal", "berimp", "FP", "dupl", "lokal", "latar", "tkvld", "medIoU"))
    for b in h["tabel"]:
        print("%2d %5.1f-%-7.1f %5d %5d %5d %7.3f | %8d %8d %8d | %5d %5d %5d %5d %5d | %6.3f" %
              (b["kelompok"], b["dari_m"], b["sampai_m"], b["n_gt"], b["TP"], b["FN"], b["recall"],
               b["fn_tanpa_deteksi"], b["fn_lokalisasi"], b["fn_berimpit"],
               b["FP"], b["fp_duplikat"], b["fp_lokalisasi"], b["fp_latar"], b["fp_tak_valid"], b["med_iou_tp"]))
    print("\nGalat ukuran median pada TP per kelompok jarak (relatif; kedalaman dalam m):")
    for b in h["tabel"]:
        print("  kl %d: w %.0f%% | h %.0f%% | l %.0f%% | kedalaman %.2f m" %
              (b["kelompok"], 100 * b["med_rel_w"], 100 * b["med_rel_h"], 100 * b["med_rel_l"], b["med_galat_depth_m"]))
    j = h["jauh"]
    print("\nObjek jauh (y < y_min): %d | pusat terdeteksi: %d (%.0f%%) | deteksi zona jauh tanpa label di selnya: %d" %
          (j["n"], j["terdeteksi"], 100.0 * j["terdeteksi"] / max(j["n"], 1), j["tanpa_label"]))
    print("  ukuran deteksi pada objek jauh yang terdeteksi, median |w| %.2f m | |h| %.2f m | |l| %.2f m" %
          (j["med_abs_w"], j["med_abs_h"], j["med_abs_l"]))
    print("  ukuran mendekati nol (semua |w|,|h|,|l| < %.1f m): %d dari %d" % (j["nol_m"], j["dekat_nol"], j["terdeteksi"]))




#### fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_detektor/analisis_galat.py
def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--pred", required=True)
    p.add_argument("--labels", default=os.path.expanduser("~/projects/data_session0_right/converted/labels"))
    p.add_argument("--y-min", type=float, default=150)
    p.add_argument("--iou", type=float, default=0.5)
    p.add_argument("--skor-min", type=float, default=0.1)
    p.add_argument("--bin", type=int, default=5)
    p.add_argument("--nol-m", type=float, default=1.0)
    p.add_argument("--csv", default=None, help="awalan berkas keluaran; bawaan = nama pkl")
    a = p.parse_args(argv)

    import kamera
    cam = kamera.load_camera()
    with open(a.pred, "rb") as f:
        pkl = pickle.load(f)
    h = analisis(pkl, a.labels, cam["K"], cam["T"], a.y_min, a.iou, a.skor_min, a.bin, a.nol_m)
    cetak(h, a.skor_min, a.iou)
    awal = a.csv or os.path.splitext(a.pred)[0]
    with open(awal + "_jarak.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(h["tabel"][0].keys()))
        w.writeheader()
        w.writerows(h["tabel"])
    with open(awal + "_kasus.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["jenis", "sebab", "frame", "x", "y", "depth", "skor"])
        w.writeheader()
        w.writerows(h["kasus"])
    print("\ntersimpan:", awal + "_jarak.csv", "|", awal + "_kasus.csv", "(%d baris FN dan FP)" % len(h["kasus"]))




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_detektor/analisis_galat.py
if __name__ == "__main__":
    main()


### Menjalankan latihan dan evaluasi detektor

In [ ]:
D = os.path.join(KODE, "latih_detektor")
perintah = [
    ["python3", "train_session0_right.py", "--init", os.path.join(BOBOT_DET, "centernet-3d-bbox_asli.pth"),
     "--out", os.path.join(HASIL_NB, "centernet_dari_lama.pth")],
    ["python3", "train_session0_right.py", "--init", os.path.join(BOBOT_DET, "centernet_session0_right.pth"),
     "--out", os.path.join(HASIL_NB, "centernet_dari_baru.pth")],
]
for p in perintah:
    print(" ".join(x.replace(RUMAH, "~") for x in p))
if LATIH_DETEKTOR:
    for p in perintah:
        jalankan(p, cwd=D, ekor=30)
else:
    print("\n(LATIH_DETEKTOR = False: tidak dilatih; bobot yang sudah ada dipakai)")
    for f in sorted(glob.glob(os.path.join(BOBOT_DET, "*.pth"))):
        print("  %-34s %6.1f MB" % (os.path.basename(f), os.path.getsize(f) / 1048576))

In [ ]:
pred = sorted(glob.glob(os.path.join(HASIL, "prediksi_detektor", "*.pkl")))
print("Berkas prediksi:", [os.path.basename(p) for p in pred] or "(tidak ada; buat dengan prediksi.py)")
for p in pred:
    print("\n" + "=" * 20, os.path.basename(p), "=" * 20)
    jalankan(["python3", "evaluasi.py", "--pred", p], cwd=D, ekor=30)

## 3. Klasifier gambar 4 kelas

Karena detektor tidak membedakan jenis kendaraan, dilatih klasifier gambar terpisah dengan 4 kelas: **mobil pribadi,
truk, bus, van/pickup**. Modelnya ResNet-18 (bobot awal ImageNet), masukan 128 × 128 dengan letterbox, data dari
dataset publik MIO-TCD.

Model yang hanya dilatih pada MIO-TCD banyak membaca mobil dari kamera ini sebagai van/pickup. Penyebabnya sudut pandang:
MIO-TCD kebanyakan tampak samping, kamera ini melihat dari depan dan belakang. Karena itu model disesuaikan dengan
potongan gambar dari kamera ini. Penyesuaian dengan mobil saja membuat 35 dari 71 van/truk ikut terbaca mobil; setelah
potongan van (yang di anotasi berlabel "Truck") ikut dilatih, kesalahan itu hilang. Hasilnya `klas_ft2.pth`.

Berkas di tahap ini (urutan: yang diimpor lebih dulu). Pustaka luar yang dipakai: `cv2`, `numpy`, `torch`, `torchvision`.

- `siapkan_mio_tcd.py` (asli: `~/projects/latih_klasifikasi/siapkan_mio_tcd.py`)
- `potong_session0.py` (asli: `~/projects/latih_klasifikasi/potong_session0.py`)
- `terapkan_label_truk.py` (asli: `~/projects/latih_klasifikasi/terapkan_label_truk.py`)
- `latih_klasifikasi.py` (asli: `~/projects/latih_klasifikasi/latih_klasifikasi.py`)

### `siapkan_mio_tcd.py`

Menyusun dataset klasifikasi 4 kelas dari MIO-TCD (memetakan kelas MIO-TCD ke mobil pribadi, truk, bus, van/pickup) dan membaginya menjadi train, val, test.

*Asli: `~/projects/latih_klasifikasi/siapkan_mio_tcd.py`, 139 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_klasifikasi/siapkan_mio_tcd.py
#!/usr/bin/env python3
"""siapkan_mio_tcd.py - siapkan dataset klasifikasi 4 kelas dari MIO-TCD (hanya pustaka standar Python).

Dua tahap:
  1) python3 siapkan_mio_tcd.py daftar
       membaca gt_train.csv, mengambil subsampel acak (seed tetap) per jenis MIO-TCD, membagi
       80/10/10 (latih/validasi/uji, acak per gambar, seimbang per jenis), lalu menulis
         <akar>/terpilih.csv       (id, jenis_mio, kelas, split)
         <akar>/daftar_ekstrak.txt (jalur di dalam arsip, untuk tar -T)
  2) python3 siapkan_mio_tcd.py ekstrak
       mengalirkan arsip MIO-TCD dari internet (3,1 GB, TIDAK disimpan) dan hanya mengambil gambar
       yang ada di daftar. Boleh dijalankan ulang: yang sudah ada dilewati dan arsip dibaca lagi.
  3) python3 siapkan_mio_tcd.py susun
       memindahkan gambar hasil ekstraksi ke <keluar>/<split>/<kelas>/ dan menulis objects.csv

Pemetaan 4 kelas (pilihan kita; sesuaikan bila dosen meminta lain):
  car -> mobil_pribadi | single_unit_truck, articulated_truck -> truk | bus -> bus
  pickup_truck, work_van -> van_pickup
Catatan: MIO-TCD tidak punya ukuran kendaraan dan satu kendaraan bisa muncul di beberapa gambar;
pembagian di sini per gambar, jadi angka ujinya bisa sedikit optimistis.
"""
import argparse
import csv
import os
import random
import shutil
import sys

KELAS = {"car": "mobil_pribadi", "single_unit_truck": "truk", "articulated_truck": "truk",
         "bus": "bus", "pickup_truck": "van_pickup", "work_van": "van_pickup"}
JUMLAH = {"car": 10000, "bus": 10000, "single_unit_truck": 5000, "articulated_truck": 5000,
          "pickup_truck": 5000, "work_van": 5000}
SPLIT = (("train", 0.8), ("val", 0.1), ("test", 0.1))
SEED = 42




#### fungsi `daftar`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/siapkan_mio_tcd.py
def daftar(akar):
    per = {j: [] for j in JUMLAH}
    with open(os.path.join(akar, "gt_train.csv"), newline="") as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[-1].strip() in per:
                per[row[-1].strip()].append(row[0].strip())
    rng = random.Random(SEED)
    baris = []
    for j in JUMLAH:
        ids = sorted(per[j])
        if len(ids) < JUMLAH[j]:
            print("PERINGATAN: %s hanya %d gambar (diminta %d)" % (j, len(ids), JUMLAH[j]))
        pilih = rng.sample(ids, min(JUMLAH[j], len(ids)))
        n = len(pilih)
        n_tr, n_va = int(n * SPLIT[0][1]), int(n * SPLIT[1][1])
        for k, i in enumerate(pilih):
            sp = "train" if k < n_tr else ("val" if k < n_tr + n_va else "test")
            baris.append((i, j, KELAS[j], sp))
    with open(os.path.join(akar, "terpilih.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "jenis_mio", "kelas", "split"])
        w.writerows(baris)
    with open(os.path.join(akar, "daftar_ekstrak.txt"), "w") as f:
        f.write("".join("train/%s/%s.jpg\n" % (j, i) for i, j, _, _ in baris))
    ring = {}
    for _, _, k, sp in baris:
        ring[(k, sp)] = ring.get((k, sp), 0) + 1
    print("total terpilih: %d" % len(baris))
    for k in ("mobil_pribadi", "truk", "bus", "van_pickup"):
        print("  %-14s " % k + " ".join("%s=%d" % (sp, ring.get((k, sp), 0)) for sp, _ in SPLIT))
    print("tersimpan: terpilih.csv dan daftar_ekstrak.txt di", akar)




#### fungsi `ekstrak`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/siapkan_mio_tcd.py
def ekstrak(akar, url):
    import tarfile
    import time
    import urllib.request
    with open(os.path.join(akar, "daftar_ekstrak.txt")) as f:
        semua = [l.strip() for l in f if l.strip()]
    perlu = {p for p in semua if not os.path.exists(os.path.join(akar, p))}
    print("diminta: %d | sudah ada: %d | akan diambil: %d" % (len(semua), len(semua) - len(perlu), len(perlu)))
    if not perlu:
        return
    t0, lihat, ambil = time.time(), 0, 0
    with urllib.request.urlopen(url) as resp, tarfile.open(fileobj=resp, mode="r|") as tf:
        for m in tf:
            lihat += 1
            if m.name in perlu:
                try:
                    tf.extract(m, akar, filter="data")
                except TypeError:  # Python lama tanpa parameter filter
                    tf.extract(m, akar)
                perlu.discard(m.name)
                ambil += 1
                if not perlu:
                    break
            if lihat % 50000 == 0:
                print("  dibaca %d entri | terambil %d | %.0f detik" % (lihat, ambil, time.time() - t0))
    print("selesai: terambil %d | belum ketemu %d | %.0f detik" % (ambil, len(perlu), time.time() - t0))
    if perlu:
        print("PERINGATAN: sebagian tidak ketemu di arsip; jalankan ulang tahap ekstrak bila unduhan terputus.")




#### fungsi `susun`

Bersama: fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/siapkan_mio_tcd.py
def susun(akar, keluar):
    hilang, dipindah, rows = 0, 0, []
    with open(os.path.join(akar, "terpilih.csv"), newline="") as f:
        for r in csv.DictReader(f):
            src = os.path.join(akar, "train", r["jenis_mio"], r["id"] + ".jpg")
            nama = "%s__%s.jpg" % (r["jenis_mio"], r["id"])
            dst_dir = os.path.join(keluar, r["split"], r["kelas"])
            if not os.path.exists(src):
                hilang += 1
                continue
            os.makedirs(dst_dir, exist_ok=True)
            shutil.move(src, os.path.join(dst_dir, nama))
            rows.append([os.path.join(r["split"], r["kelas"], nama), r["kelas"], r["jenis_mio"], r["split"]])
            dipindah += 1
    with open(os.path.join(keluar, "objects.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["berkas", "kelas", "jenis_mio", "split"])
        w.writerows(rows)
    print("dipindah: %d | tidak ditemukan: %d | tersimpan: %s" % (dipindah, hilang, keluar))
    if hilang:
        print("PERINGATAN: ada berkas yang belum terekstrak; jalankan ulang ekstraksi (tar -T).")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("tahap", choices=["daftar", "ekstrak", "susun"])
    p.add_argument("--url", default="https://tcd.miovision.com/static/dataset/MIO-TCD-Classification.tar")
    p.add_argument("--akar", default=os.path.expanduser("~/dataset_publik/mio_tcd"))
    p.add_argument("--keluar", default=os.path.expanduser("~/dataset_publik/dataset_klasifikasi"))
    a = p.parse_args()
    if a.tahap == "daftar":
        daftar(a.akar)
    elif a.tahap == "ekstrak":
        ekstrak(a.akar, a.url)
    else:
        susun(a.akar, a.keluar)




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/siapkan_mio_tcd.py
if __name__ == "__main__":
    sys.exit(main())


### `potong_session0.py`

Memotong gambar kendaraan dari frame `session0_right` memakai kotak 2D di anotasi XML, untuk menguji dan menyesuaikan klasifier dengan sudut pandang kamera ini.

*Asli: `~/projects/latih_klasifikasi/potong_session0.py`, 84 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_klasifikasi/potong_session0.py
#!/usr/bin/env python3
"""potong_session0.py - potong gambar kendaraan dari frame session0_right memakai <bbox2d> di XML.

Hasil: folder uji dengan format sama seperti dataset publik (folder per kelas) + objects.csv:
  <keluar>/mobil_pribadi/session0_right_000342_o0.jpg   (jenis XML: Car)
  <keluar>/truk/...                                      (jenis XML: Truck)
  Bus (hanya 1 objek) dan jenis lain dilewati: terlalu sedikit untuk diuji.
Catatan: label 'Car' di XML mencakup mobil pribadi, SUV, minivan, bahkan van/pickup (tidak dipisah),
jadi kelas mobil_pribadi di sini adalah 'Car' apa adanya. Satu kendaraan muncul di banyak frame
berurutan, jadi tiap-N-frame dipakai untuk mengurangi duplikat (jumlah potongan != jumlah kendaraan).
Aturan (pilihan kita): margin 5% di sekeliling kotak; potongan dengan sisi < --min-sisi piksel dilewati.
Pemakaian: python3 potong_session0.py [--tiap 10] [--tiap-truk 3]
"""
import argparse
import csv
import os
import xml.etree.ElementTree as ET

import cv2

PETA = {"Car": "mobil_pribadi", "Truck": "truk"}




#### fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/potong_session0.py
def main():
    p = argparse.ArgumentParser()
    base = os.path.expanduser("~/projects/data_session0_right")
    p.add_argument("--base", default=base)
    p.add_argument("--keluar", default=os.path.expanduser("~/dataset_publik/uji_session0_right"))
    p.add_argument("--tiap", type=int, default=10, help="ambil Car tiap N frame")
    p.add_argument("--tiap-truk", type=int, default=3, help="ambil Truck tiap N frame")
    p.add_argument("--min-sisi", type=int, default=30)
    p.add_argument("--margin", type=float, default=0.05)
    a = p.parse_args()

    xmls = sorted(f for f in os.listdir(os.path.join(a.base, "annotations")) if f.endswith(".xml"))
    for k in PETA.values():
        os.makedirs(os.path.join(a.keluar, k), exist_ok=True)
    rows, skip_kecil, skip_jenis = [], 0, {}
    for idx, fx in enumerate(xmls):
        stem = fx[:-4]
        root = ET.parse(os.path.join(a.base, "annotations", fx)).getroot()
        objs = root.findall("object")
        perlu = [o for o in objs if (o.findtext("type") or "").strip() in PETA]
        for o in objs:
            j = (o.findtext("type") or "").strip()
            if j not in PETA:
                skip_jenis[j] = skip_jenis.get(j, 0) + 1
        ambil = [o for o in perlu if idx % (a.tiap_truk if o.findtext("type").strip() == "Truck" else a.tiap) == 0]
        if not ambil:
            continue
        img = cv2.imread(os.path.join(a.base, "images", stem + ".jpg"))
        if img is None:
            raise SystemExit("gambar tidak terbaca: " + stem)
        H, W = img.shape[:2]
        for oi, o in enumerate(objs):
            if o not in ambil:
                continue
            x, y, w, h = [float(v) for v in o.findtext("bbox2d").split()]
            mx, my = w * a.margin, h * a.margin
            x0, y0 = max(0, int(x - mx)), max(0, int(y - my))
            x1, y1 = min(W, int(x + w + mx)), min(H, int(y + h + my))
            if min(x1 - x0, y1 - y0) < a.min_sisi:
                skip_kecil += 1
                continue
            jenis = o.findtext("type").strip()
            kelas = PETA[jenis]
            nama = "%s_o%d.jpg" % (stem, oi)
            cv2.imwrite(os.path.join(a.keluar, kelas, nama), img[y0:y1, x0:x1])
            l, wd, hh = (float(v) for v in o.findtext("veh_size").split())  # urutan XML: panjang lebar tinggi
            rows.append([os.path.join(kelas, nama), kelas, jenis, stem, oi, x0, y0, x1 - x0, y1 - y0,
                         (o.findtext("perspective") or "").strip(), l, wd, hh])
    with open(os.path.join(a.keluar, "objects.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["berkas", "kelas", "jenis_xml", "frame", "obj", "x", "y", "lebar_px", "tinggi_px",
                    "perspektif", "panjang_m", "lebar_m", "tinggi_m"])
        w.writerows(rows)
    n = {k: sum(1 for r in rows if r[1] == k) for k in PETA.values()}
    print("frame XML: %d | potongan: %s | dilewati terlalu kecil: %d | jenis lain dilewati: %s"
          % (len(xmls), n, skip_kecil, skip_jenis or "-"))
    print("tersimpan:", a.keluar)




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/potong_session0.py
if __name__ == "__main__":
    main()


### `terapkan_label_truk.py`

Menerapkan label hasil pemeriksaan manual: potongan berlabel "Truck" yang ternyata van dipindah ke kelas van, sisanya dibuang.

*Asli: `~/projects/latih_klasifikasi/terapkan_label_truk.py`, 23 baris.*

In [ ]:
%%writefile kode_notebook/latih_klasifikasi/terapkan_label_truk.py
#!/usr/bin/env python3
"""terapkan_label_truk.py - terapkan label manual pada potongan 'Truck' session0_right.
Hasil pemeriksaan lembar kontak: #0-#66 = van putih (1-2 kendaraan), #67-#70 = potongan buruk (papan jalan).
Aksi (aman, tidak menghapus apa pun):
  - salin #0-#66 ke uji_session0_right/van_pickup/
  - pindahkan folder 'truk' asli menjadi 'truk_xml_asli' (agar tidak dibaca sebagai kelas truk)
  - catat #67-#70 di uji_session0_right/potongan_dibuang.txt
Jalankan sekali saja."""
import csv, os, shutil
H = os.path.expanduser("~/dataset_publik")
U = H + "/uji_session0_right"
daftar = list(csv.DictReader(open(H + "/hasil_klasifikasi/daftar_truk.csv")))
assert len(daftar) == 71, "daftar_truk.csv harus 71 baris, ada %d" % len(daftar)
assert os.path.isdir(U + "/truk"), "folder truk sudah dipindah? jangan jalankan dua kali"
van, buang = daftar[0:67], daftar[67:71]
os.makedirs(U + "/van_pickup", exist_ok=True)
for r in van:
    shutil.copy2(U + "/" + r["berkas"], U + "/van_pickup/" + os.path.basename(r["berkas"]))
with open(U + "/potongan_dibuang.txt", "w") as f:
    for r in buang:
        f.write("%s (frame %s): potongan bukan kendaraan yang layak\n" % (r["berkas"], r["frame"]))
os.rename(U + "/truk", U + "/truk_xml_asli")
print("van_pickup: %d potongan; dibuang: %d; folder truk -> truk_xml_asli" % (len(van), len(buang)))


### `latih_klasifikasi.py`

Latihan klasifier ResNet-18. Opsi `--init` dan `--tambah-session` dipakai untuk menyesuaikan model dengan potongan gambar dari kamera ini.

*Asli: `~/projects/latih_klasifikasi/latih_klasifikasi.py`, 296 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/latih_klasifikasi/latih_klasifikasi.py
#!/usr/bin/env python3
"""latih_klasifikasi.py - latih klasifikasi 4 kelas (mobil_pribadi, truk, bus, van_pickup) dari gambar.

Model : ResNet-18 bobot ImageNet (torchvision), lapisan fc diganti 4 keluaran. Tanpa YOLO.
Data  : ~/dataset_publik/dataset_klasifikasi/{train,val,test}/<kelas>/*.jpg   (MIO-TCD, siapkan_mio_tcd.py)
Uji   : ~/dataset_publik/uji_session0_right/{mobil_pribadi,truk}/*.jpg         (potong_session0.py)
Masukan: potongan kendaraan, diskalakan ke 128x128 tanpa mengubah rasio (sisa diberi warna rata-rata
         ImageNet), urutan kanal RGB, normalisasi ImageNet. (Berbeda dari detektor CenterNet yang BGR.)
Augmentasi (latih saja, semuanya pilihan kita): flip horizontal, geser tepi 0-8%, penurunan resolusi
         acak (skala 0,4-1,0 lalu diperbesar lagi, agar model tidak bersandar pada ketajaman gambar,
         karena potongan bus di MIO-TCD lebih besar dan tajam), kecerahan/kontras acak.
Keluaran: bobot terbaik (akurasi validasi tertinggi) + log txt/csv + prediksi_session0.csv.
Catatan : 'Car' di XML session0_right mencakup SUV, minivan, bahkan van/pickup; prediksi van_pickup
          pada potongan itu belum tentu salah, jadi dilaporkan terpisah dan perlu diperiksa manual.
Pemakaian: python3 latih_klasifikasi.py --epochs 1 --max-per-class 20 --tag uji     (uji kecil)
           python3 latih_klasifikasi.py --epochs 5 --tag klas5
           python3 latih_klasifikasi.py --eval-saja ~/projects/bobot/klasifikasi/klas5.pth
"""
import argparse
import csv
import os
import random
import re
import sys
import time

import cv2
import numpy as np

KELAS = ("mobil_pribadi", "truk", "bus", "van_pickup")
UKURAN = 128
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
PAD_BGR = (104, 116, 124)  # rata-rata ImageNet x 255 (urutan BGR) -> 0 setelah normalisasi




#### fungsi `listar` — Daftar (jalur, indeks_kelas) dari root/split/<kelas>/*.jpg.

Bersama: fungsi `nomor_frame` — Nomor frame dari nama berkas, mis. session0_right_000057_o2.jpg -> 57 (tidak ada -> -1).; fungsi `letterbox`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/latih_klasifikasi.py
def listar(root, split, kelas=KELAS, maks=None, seed=0):
    """Daftar (jalur, indeks_kelas) dari root/split/<kelas>/*.jpg."""
    out = []
    for i, k in enumerate(KELAS):
        if k not in kelas:
            continue
        d = os.path.join(root, split, k) if split else os.path.join(root, k)
        if not os.path.isdir(d):
            continue
        nama = sorted(f for f in os.listdir(d) if f.lower().endswith(".jpg"))
        if maks and len(nama) > maks:
            nama = random.Random(seed).sample(nama, maks)
        out += [(os.path.join(d, n), i) for n in nama]
    return out


def nomor_frame(path):
    """Nomor frame dari nama berkas, mis. session0_right_000057_o2.jpg -> 57 (tidak ada -> -1)."""
    m = re.search(r"_(\d{6})_", os.path.basename(path))
    return int(m.group(1)) if m else -1


def letterbox(img, ukuran=UKURAN):
    h, w = img.shape[:2]
    s = ukuran / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    kecil = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    out = np.empty((ukuran, ukuran, 3), np.uint8)
    out[:] = PAD_BGR
    y0, x0 = (ukuran - nh) // 2, (ukuran - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = kecil
    return out




#### fungsi `augment`

Bersama: fungsi `ke_array`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/latih_klasifikasi.py
def augment(img):
    h, w = img.shape[:2]
    if random.random() < 0.5:
        img = img[:, ::-1]
    # geser tepi 0-8%
    ty, by = int(h * random.uniform(0, 0.08)), int(h * random.uniform(0, 0.08))
    lx, rx = int(w * random.uniform(0, 0.08)), int(w * random.uniform(0, 0.08))
    if h - ty - by > 8 and w - lx - rx > 8:
        img = img[ty:h - by, lx:w - rx]
    # penurunan resolusi acak
    if random.random() < 0.5:
        s = random.uniform(0.4, 1.0)
        hh, ww = img.shape[:2]
        kecil = cv2.resize(np.ascontiguousarray(img), (max(2, int(ww * s)), max(2, int(hh * s))),
                           interpolation=cv2.INTER_AREA)
        img = cv2.resize(kecil, (ww, hh), interpolation=cv2.INTER_LINEAR)
    # kecerahan dan kontras
    a, b = random.uniform(0.7, 1.3), random.uniform(-25, 25)
    img = np.clip(np.ascontiguousarray(img).astype(np.float32) * a + b, 0, 255).astype(np.uint8)
    return img


def ke_array(img_bgr):
    rgb = img_bgr[:, :, ::-1].astype(np.float32) / 255.0
    return np.ascontiguousarray(((rgb - MEAN) / STD).transpose(2, 0, 1), dtype=np.float32)




#### fungsi `seragamkan` — Ubah sisi terpanjang menjadi tepat `sisi` piksel (naik atau turun) agar semua potongan

Bersama: fungsi `muat`; fungsi `matriks`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/latih_klasifikasi.py
def seragamkan(img, sisi):
    """Ubah sisi terpanjang menjadi tepat `sisi` piksel (naik atau turun) agar semua potongan
    punya resolusi efektif sama; ketajaman tidak lagi bisa dipakai sebagai petunjuk kelas."""
    h, w = img.shape[:2]
    s = sisi / max(h, w)
    return cv2.resize(np.ascontiguousarray(img), (max(1, int(round(w * s))), max(1, int(round(h * s)))),
                      interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)


def muat(path, latih, sisi=0):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    if latih:
        img = augment(img)
    if sisi:
        img = seragamkan(img, sisi)
    return ke_array(letterbox(img))


def matriks(y_true, y_pred, n=len(KELAS)):
    m = np.zeros((n, n), int)
    for t, p in zip(y_true, y_pred):
        m[t, p] += 1
    return m




#### fungsi `cetak_matriks`

Bersama: fungsi `main`

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/latih_klasifikasi.py
def cetak_matriks(m, judul, baris_idx=None):
    baris_idx = list(range(len(KELAS))) if baris_idx is None else baris_idx
    print("\n%s (baris = kelas benar, kolom = prediksi)" % judul)
    print("%-14s" % "" + "".join("%14s" % k for k in KELAS) + "%10s" % "recall")
    for i in baris_idx:
        tot = m[i].sum()
        print("%-14s" % KELAS[i] + "".join("%14d" % v for v in m[i]) + "%10.3f" % (m[i, i] / tot if tot else float("nan")))
    if len(baris_idx) == len(KELAS):
        print("%-14s" % "presisi" + "".join("%14.3f" % (m[j, j] / m[:, j].sum() if m[:, j].sum() else float("nan")) for j in range(len(KELAS))))
        print("akurasi total: %.4f" % (np.trace(m) / max(m.sum(), 1)))


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--root", default=os.path.expanduser("~/dataset_publik/dataset_klasifikasi"))
    p.add_argument("--uji-session", default=os.path.expanduser("~/dataset_publik/uji_session0_right"))
    p.add_argument("--outdir", default=os.path.expanduser("~/dataset_publik/hasil_klasifikasi"))
    p.add_argument("--tag", default="klas")
    p.add_argument("--epochs", type=int, default=5)
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--max-per-class", type=int, default=None, help="uji kecil: batasi gambar per kelas")
    p.add_argument("--no-pretrained", action="store_true")
    p.add_argument("--eval-saja", default=None, help="jalur .pth: hanya evaluasi, tanpa latihan")
    p.add_argument("--res-tetap", type=int, default=0,
                   help="seragamkan sisi terpanjang potongan ke N piksel (mis. 48) sebelum letterbox; 0 = mati")
    p.add_argument("--init", default=None, help="mulai dari bobot klasifikasi ini (fine-tune)")
    p.add_argument("--tambah-session", action="store_true",
                   help="tambahkan potongan mobil_pribadi session0_right (frame < --frame-bagi) ke data latih; "
                        "uji session0_right otomatis hanya frame >= --frame-bagi")
    p.add_argument("--ulang-session", type=int, default=3, help="pengulangan potongan mobil session0 di data latih")
    p.add_argument("--ulang-van", type=int, default=1, help="pengulangan potongan van_pickup session0 di data latih")
    p.add_argument("--frame-bagi", type=int, default=2545, help="frame < ini = latih, >= ini = uji (sama dengan detektor)")
    p.add_argument("--uji-val-saja", action="store_true", help="uji session0_right hanya frame >= --frame-bagi")
    p.add_argument("--seed", type=int, default=42)
    a = p.parse_args(argv)

    import torch
    import torch.nn as nn
    import torchvision

    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    os.makedirs(a.outdir, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    log_txt = os.path.join(a.outdir, "%s_%s.txt" % (a.tag, stamp))
    log_csv = os.path.join(a.outdir, "%s_%s.csv" % (a.tag, stamp))
    logf = open(log_txt, "a")

    def log(msg):
        print(msg)
        logf.write(msg + "\n")
        logf.flush()

    class Data(torch.utils.data.Dataset):
        def __init__(self, items, latih):
            self.items, self.latih = items, latih

        def __len__(self):
            return len(self.items)

        def __getitem__(self, i):
            path, y = self.items[i]
            return torch.from_numpy(muat(path, self.latih, a.res_tetap)), y

    def loader(items, latih):
        return torch.utils.data.DataLoader(Data(items, latih), batch_size=a.batch, shuffle=latih,
                                           num_workers=a.workers, drop_last=False)

    def buat_model(pretrained):  # noqa
        m = torchvision.models.resnet18(weights="ResNet18_Weights.IMAGENET1K_V1" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, len(KELAS))
        return m

    def lintasan(model, dl, optimizer=None):
        training = optimizer is not None
        model.train() if training else model.eval()
        crit = nn.CrossEntropyLoss()
        tot, n, benar = 0.0, 0, 0
        yt, yp, probs = [], [], []
        ctx = torch.enable_grad() if training else torch.no_grad()
        with ctx:
            for x, y in dl:
                out = model(x)
                loss = crit(out, y)
                if training:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                tot += float(loss.detach()) * len(y)
                n += len(y)
                pred = out.argmax(1)
                benar += int((pred == y).sum())
                yt += y.tolist(); yp += pred.tolist()
                probs += torch.softmax(out.detach(), 1).tolist()
        return tot / max(n, 1), benar / max(n, 1), yt, yp, probs

    def evaluasi_akhir(model, tag):
        te = listar(a.root, "test", maks=a.max_per_class)
        _, acc, yt, yp, _ = lintasan(model, loader(te, False))
        cetak_matriks(matriks(yt, yp), "UJI MIO-TCD (%d gambar, akurasi %.4f)" % (len(te), acc))
        se = listar(a.uji_session, None, kelas=("mobil_pribadi", "truk", "van_pickup"), maks=None)
        if a.tambah_session or a.uji_val_saja:
            se = [(pt, y) for pt, y in se if nomor_frame(pt) >= a.frame_bagi]
            log("uji session0_right dibatasi frame >= %d (tidak dipakai melatih)" % a.frame_bagi)
        if se:
            _, acc2, yt2, yp2, pr2 = lintasan(model, loader(se, False))
            m2 = matriks(yt2, yp2)
            cetak_matriks(m2, "UJI session0_right (%d potongan; label: Car->mobil_pribadi, van manual->van_pickup)" % len(se),
                          sorted(set(y for _, y in se)))
            log("catatan: prediksi van_pickup/bus pada 'Car' belum tentu salah (label XML tidak memisahkan van/pickup).")
            out_csv = os.path.join(a.outdir, "prediksi_session0_%s.csv" % tag)
            with open(out_csv, "w", newline="") as f:
                w = csv.writer(f)
                w.writerow(["berkas", "kelas_xml", "prediksi"] + ["p_" + k for k in KELAS])
                for (path, y), pb, yy in zip(se, pr2, yp2):
                    w.writerow([os.path.relpath(path, a.uji_session), KELAS[y], KELAS[yy]] + ["%.4f" % v for v in pb])
            log("tersimpan: " + out_csv)
        else:
            log("folder uji session0_right tidak ditemukan: " + a.uji_session)

    if a.eval_saja:
        model = buat_model(False)
        model.load_state_dict(torch.load(a.eval_saja, map_location="cpu"))
        evaluasi_akhir(model, a.tag)
        return

    tr = listar(a.root, "train", maks=a.max_per_class)
    if a.tambah_session:
        sc = [(pt, y) for pt, y in listar(a.uji_session, None, kelas=("mobil_pribadi",))
              if 0 <= nomor_frame(pt) < a.frame_bagi]
        sv = [(pt, y) for pt, y in listar(a.uji_session, None, kelas=("van_pickup",))
              if 0 <= nomor_frame(pt) < a.frame_bagi]
        log("tambah session0_right (frame < %d): %d mobil_pribadi x%d, %d van_pickup x%d"
            % (a.frame_bagi, len(sc), a.ulang_session, len(sv), a.ulang_van))
        tr = tr + sc * a.ulang_session + sv * a.ulang_van
    va = listar(a.root, "val", maks=a.max_per_class)
    log("torch %s | torchvision %s | argumen: %s" % (torch.__version__, torchvision.__version__, vars(a)))
    log("latih %d | validasi %d gambar" % (len(tr), len(va)))
    model = buat_model(not a.no_pretrained and not a.init)
    if a.init:
        model.load_state_dict(torch.load(a.init, map_location="cpu"))
        log("bobot awal dimuat dari %s" % a.init)
    opt = torch.optim.Adam(model.parameters(), lr=a.lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(a.epochs, 1))
    with open(log_csv, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "lr", "train_loss", "train_acc", "val_loss", "val_acc", "detik"])
    out = os.path.join(a.outdir, a.tag + ".pth")
    terbaik = -1.0
    dl_tr, dl_va = loader(tr, True), loader(va, False)
    for ep in range(a.epochs):
        t0 = time.time()
        lr = opt.param_groups[0]["lr"]
        l1, a1, _, _, _ = lintasan(model, dl_tr, opt)
        sch.step()
        l2, a2, _, _, _ = lintasan(model, dl_va)
        dt = time.time() - t0
        log("epoch %d/%d | lr %.2e | latih loss %.4f acc %.4f | val loss %.4f acc %.4f | %.0f s"
            % (ep + 1, a.epochs, lr, l1, a1, l2, a2, dt))
        with open(log_csv, "a", newline="") as f:
            csv.writer(f).writerow([ep + 1, lr, l1, a1, l2, a2, round(dt, 1)])
        if a2 > terbaik:
            terbaik = a2
            torch.save(model.state_dict(), out)
            log("  akurasi validasi membaik ke %.4f -> disimpan %s" % (a2, out))
    log("selesai. akurasi validasi terbaik %.4f" % terbaik)
    model.load_state_dict(torch.load(out, map_location="cpu"))
    evaluasi_akhir(model, a.tag)




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/latih_klasifikasi/latih_klasifikasi.py
if __name__ == "__main__":
    main()


### Menjalankan latihan klasifier

In [ ]:
D = os.path.join(KODE, "latih_klasifikasi")
akar = os.path.join(PUBLIK, "dataset_klasifikasi")
if os.path.isdir(os.path.join(akar, "train")):
    print("%-16s %8s %8s %8s" % ("kelas", "train", "val", "test"))
    for kl in sorted(os.listdir(os.path.join(akar, "train"))):
        print("%-16s %8d %8d %8d" % (kl, *[len(os.listdir(os.path.join(akar, b, kl))) if os.path.isdir(os.path.join(akar, b, kl)) else 0
                                          for b in ("train", "val", "test")]))
perintah = [
    ["python3", "latih_klasifikasi.py", "--epochs", "5", "--tag", "nb_klas5"],
    ["python3", "latih_klasifikasi.py", "--init", os.path.join(BOBOT_KLAS, "klas5.pth"), "--tambah-session",
     "--epochs", "3", "--max-per-class", "3000", "--lr", "1e-4", "--tag", "nb_klas_ft2"],
]
for p in perintah:
    print(" ".join(x.replace(RUMAH, "~") for x in p))
if LATIH_KLASIFIER:
    for p in perintah:
        jalankan(p, cwd=D, ekor=40)
else:
    print("\n(LATIH_KLASIFIER = False: tidak dilatih; klas_ft2.pth yang sudah ada dipakai)")
hk = os.path.join(PUBLIK, "hasil_klasifikasi")
tampil_gambar(os.path.join(hk, "diagnosa.png"))
log = sorted(glob.glob(os.path.join(hk, "klas_ft2_*.txt")), key=os.path.getmtime)
if log:
    print("Ekor log", os.path.basename(log[-1]), "(akurasi dan matriks kebingungan):\n")
    print("\n".join(open(log[-1], errors="replace").read().splitlines()[-45:]))

## 4. Pipeline: deteksi, pelacakan, klasifikasi, penghitungan

| Tahap | Berkas |
|---|---|
| Deteksi | `detector.py` |
| Proyeksi ke bidang jalan dan filter | `geometry.py`, `detection_filter.py` |
| Pelacakan | `tracker.py`, `kalman_filter.py` |
| Klasifikasi gambar × panjang | `klasifikasi_gambar.py`, `classifier.py` |
| Penghitungan dua garis | `counter.py` |
| Tampilan berwarna per kelas | `visualize.py`, `box3d.py` |

Klasifikasi: `skor[c] = (rata-rata probabilitas gambar[c] + ε) × (rata-rata kecocokan panjang[c] + ε)`; kelas = skor terbesar.

Dua garis hitung: 50 m untuk jalur yang mendekat, 85 m untuk jalur yang menjauh, karena kendaraan di jalur kanan baru
terlihat pada jarak lebih dari 54 m dan tidak pernah melintasi garis 50 m.

Berkas di tahap ini (urutan: yang diimpor lebih dulu). Pustaka luar yang dipakai: `cv2`, `numpy`, `torch`, `torchvision`, `tqdm`.

- `config.py` (asli: `~/projects/uji_session0_pipeline/config.py`)
- `detection_cache.py` (asli: `~/projects/uji_session0_pipeline/detection_cache.py`)
- `counter.py` (asli: `~/projects/uji_session0_pipeline/counter.py`)
- `geometry.py` (asli: `~/projects/uji_session0_pipeline/geometry.py`)
- `detection_filter.py` (asli: `~/projects/uji_session0_pipeline/detection_filter.py`)
- `kalman_filter.py` (asli: `~/projects/uji_session0_pipeline/kalman_filter.py`)
- `box3d.py` (asli: `~/projects/uji_session0_pipeline/box3d.py`)
- `klasifikasi_gambar.py` (asli: `~/projects/uji_session0_pipeline/klasifikasi_gambar.py`)
- `classifier.py` (asli: `~/projects/uji_session0_pipeline/classifier.py`)
- `tracker.py` (asli: `~/projects/uji_session0_pipeline/tracker.py`)
- `visualize.py` (asli: `~/projects/uji_session0_pipeline/visualize.py`)
- `detector.py` (asli: `~/projects/uji_session0_pipeline/detector.py`)
- `run.py` (asli: `~/projects/uji_session0_pipeline/run.py`)
- `hitung_gt.py` (asli: `~/projects/alat/hitung_gt.py`)

### `config.py`

Semua setelan pipeline: ukuran gambar, kalibrasi kamera, ambang filter, kelas dan templat ukurannya, garis hitung, warna per kelas, dan path bobot klasifier. Blok di akhir berkas membaca profil kamera dari antarmuka web bila ada.

*Asli: `~/projects/uji_session0_pipeline/config.py`, 168 baris.*

In [ ]:
%%writefile kode_notebook/pipeline/config.py
"""
Konfigurasi terpusat. Semua angka yang memengaruhi hasil ada di sini.

Kolom "Dasar" di komentar menyatakan asal nilai:
  [rujukan]    ada dasar literatur (disebut di komentar)
  [kode asal]  diwarisi dari proyek asal antoinekeller/3d_bbox_highway
  [heuristik]  dipilih/diuji manual pada video uji; BELUM ada rujukan
               literatur. Nilainya perlu dievaluasi, jangan dianggap baku.
"""

import os

# ----------------------------------------------------------------------
# Citra dan kalibrasi kamera
# Model kamera lubang jarum (pinhole): Hartley & Zisserman, "Multiple View
# Geometry in Computer Vision", 2nd ed., Cambridge Univ. Press, 2004, bab 6.
# Nilai awal berasal dari simulator CARLA [kode asal]; untuk video nyata
# setel ulang dengan: python -m tools.calibrate_tuner VIDEO.mp4
# ----------------------------------------------------------------------
IMG_W = 1280
IMG_H = 720
PITCH_DEG = 9.80       # kamera menunduk (derajat)
CAM_HEIGHT_M = 8.30    # tinggi kamera dari jalan (meter)
FOCAL_PX = 770.00      # focal length (piksel)
# Kalibrasi kamera nyata dari berkas BrnoCompSpeed (calibParams.xml). Bila diisi, PITCH_DEG,
# CAM_HEIGHT_M, dan FOCAL_PX di atas TIDAK dipakai oleh geometry.py. None = pakai model pitch di atas.
CALIB_XML = os.path.expanduser("~/projects/data_session0_right/calib/session0_right_calibParams.xml")
CALIB_REF_SIZE = (1920, 1080)   # ukuran citra asli yang dipakai berkas kalibrasi (lebar, tinggi)

# ----------------------------------------------------------------------
# Detektor CenterNet
# Zhou, Wang, Krahenbuhl, "Objects as Points", arXiv:1904.07850, 2019.
# Stride 32 karena backbone ResNet-18 dipotong di layer4 [kode asal].
# ----------------------------------------------------------------------
MODEL_SCALE = 32
DEFAULT_CONF = 0.5         # ambang skor deteksi [kode asal]
CACHE_CONF_DEFAULT = 0.2   # ambang terendah saat membuat cache [heuristik]

# ----------------------------------------------------------------------
# Filter deteksi (setelah diproyeksikan ke bidang jalan)
# [kode asal] untuk struktur filter; angka ambang [heuristik]
# ----------------------------------------------------------------------
LANE_HALF_WIDTH_M = 20.0   # session0_right [heuristik] (sebelumnya 11.0)
MIN_WIDTH_M = 0.6          # sebelumnya 1.0; dilonggarkan karena lebar dari model CARLA sering meleset
MAX_WIDTH_M = 2.5          # lebar dipotong sampai batas ini
HEIGHT_RANGE_M = (1.0, 5.0)
MIN_LENGTH_M = 3.0

# ----------------------------------------------------------------------
# Kalman Filter (constant velocity pada sumbu jalan)
# R. E. Kalman, "A New Approach to Linear Filtering and Prediction
# Problems", J. Basic Engineering 82(1), 1960.
# Model constant velocity dan bentuk Joseph: Bar-Shalom, Li, Kirubarajan,
# "Estimation with Applications to Tracking and Navigation", Wiley, 2001.
# Besaran Q dan R di bawah ini [heuristik].
# ----------------------------------------------------------------------
MAX_ACCEL = 2.0            # m/s^2, percepatan wajar mobil di jalan tol
LATERAL_STD = 1.0          # m/s, gerak lateral kecil
DIM_STD_PER_STEP = 0.01    # m per frame; dimensi dibuat hampir konstan
# simpangan baku awal state: x, y, vy, w, h, l
KF_INIT_STD = (1.0, 3.0, 20.0, 2.0, 2.0, 3.0)
# Noise pengukuran membesar dengan jarak (depth, meter): std = a + b * depth^p
KF_STD_X = (0.15, 0.010, 1)
KF_STD_Y = (0.20, 0.0005, 2)
KF_STD_DIM = (0.30, 0.015, 1)

# ----------------------------------------------------------------------
# Tracking
# Asosiasi Hungarian: H. W. Kuhn, Naval Research Logistics Quarterly 2(1-2),
# 1955; implementasi SciPy mengikuti D. F. Crouse, IEEE Trans. Aerospace and
# Electronic Systems 52(4), 2016.
# Kombinasi Kalman + Hungarian + min_hits/max_age: A. Bewley dkk., "Simple
# Online and Realtime Tracking" (SORT), ICIP 2016, arXiv:1602.00763.
# Asosiasi dua tahap (--conf-low): Y. Zhang dkk., "ByteTrack", ECCV 2022,
# arXiv:2110.06864.
# Nilai angka di bawah [heuristik].
# ----------------------------------------------------------------------
MAX_LOST_FRAMES = 30        # track dihapus setelah sekian frame tanpa deteksi (max_age)
MIN_HITS_TO_DISPLAY = 3     # track baru dipakai/ditampilkan setelah sekian deteksi (min_hits)
MAX_LOST_TO_DISPLAY = 5     # track hilang > sekian frame tidak ditampilkan
MAX_TRACK_DEPTH_M = 105.0   # session0_right [heuristik] (sebelumnya 40.0)
CONFLICT_DIST_M = 3.0       # dua track lebih dekat dari ini dianggap duplikat
# Gerbang asosiasi adaptif: batas = base + slope * kedalaman (meter).
# Objek jauh lebih noisy, objek yang baru hilang diberi toleransi lebih besar.
GATE_TRACKED = {"dist": (2.0, 0.05), "lateral": (1.0, 0.01)}
GATE_LOST = {"dist": (4.0, 0.05), "lateral": (2.0, 0.01)}

# ----------------------------------------------------------------------
# Klasifikasi kendaraan (dari ukuran terukur, dengan voting)
# SELURUH bagian ini [heuristik]: ambang panjang dan batas ukuran dipilih
# manual. Detektor tidak membedakan jenis kendaraan (head heatmap 1 kanal).
# ----------------------------------------------------------------------
CLASSES = ("car", "van", "truck", "bus")   # urutan = prioritas saat suara seri
CAR_MAX_LEN_M = 5.2
VAN_MAX_LEN_M = 7.0
TEMPLATES = {
    "car": {"w": 1.85, "h": 1.50, "l": 4.50},
    "van": {"w": 2.00, "h": 2.00, "l": 5.60},
    "truck": {"w": 2.50, "h": 3.20, "l": 9.00},
    "bus": {"w": 2.55, "h": 3.20, "l": 11.00},
}
LIMITS = {
    "car": {"w": (1.6, 2.1), "h": (1.2, 1.9), "l": (3.5, 5.3)},
    "van": {"w": (1.8, 2.3), "h": (1.7, 2.6), "l": (4.8, 7.0)},
    "truck": {"w": (2.2, 2.6), "h": (2.5, 4.2), "l": (6.5, 13.0)},
    "bus": {"w": (2.3, 2.6), "h": (2.8, 4.2), "l": (9.0, 14.0)},
}
DIM_BUF_SIZE = 30          # jumlah pengamatan dimensi yang disimpan (median)

# ----------------------------------------------------------------------
# Kotak 3D untuk tampilan [heuristik]
# ----------------------------------------------------------------------
MEASURED_WEIGHT = 0.5      # 0 = template saja, 1 = ukuran terukur saja
DIM_WARMUP = 10            # bobot ukuran terukur naik sampai sekian pengamatan
YAW_WINDOW = 10            # jumlah posisi terakhir untuk menaksir arah
YAW_MIN_DISPLACEMENT_M = 1.0
YAW_ALPHA = 0.2            # EMA pada sudut ganda
YAW_MAX_DEG = 35.0

# ----------------------------------------------------------------------
# Counting dengan garis virtual [heuristik]
# ----------------------------------------------------------------------
LINE_Y_DEFAULT_M = 50.0   # session0_right [heuristik] (sebelumnya 15.0)
COUNT_LOG = "counting_log.csv"
OUTPUT_DIR = "output_frames"

# ----------------------------------------------------------------------
# Klasifier gambar 4 kelas (ResNet-18, dilatih pada MIO-TCD + potongan session0_right)
# [heuristik] cara menggabungkan dengan panjang; bobot dari dosen/penelitian sendiri.
# ----------------------------------------------------------------------
KLAS_MODEL = os.path.expanduser("~/projects/bobot/klasifikasi/klas_ft2.pth")
KLAS_UKURAN = 128          # sisi letterbox masukan (sama dengan saat latihan)
KLAS_EVERY = 2             # klasifikasi gambar tiap sekian frame per track (hemat waktu)
KLAS_MIN_SIDE_PX = 24      # crop dengan sisi terpanjang lebih kecil dari ini dilewati
KLAS_MIN_VISIBLE = 0.7     # bagian kotak yang harus ada di dalam gambar (kotak terpotong dilewati)
KLAS_MARGIN = 0.05         # tambahan sisi crop (pecahan dari ukuran kotak)
LEN_SIGMA_M = 0.7          # kelunakan prior panjang di luar rentang LIMITS["l"]
LEN_FLOOR = 0.03           # prior panjang minimum agar satu pengamatan tidak mematikan kelas
# Warna kotak/label per kelas (BGR)
CLASS_COLORS = {
    "car": (0, 200, 0),       # hijau
    "van": (0, 215, 255),     # kuning
    "truck": (0, 0, 255),     # merah
    "bus": (255, 128, 0),     # biru
}

# ----------------------------------------------------------------------
# Garis counting kedua untuk jalur yang MENJAUH [heuristik, diukur dari video uji]
# Kendaraan dengan x >= MEDIAN_X_M (kanan median) memakai LINE_Y_AWAY_M, sisanya LINE_Y_DEFAULT_M.
# Garis menjauh harus lebih jauh dari tempat jalur terluar mulai terlihat di tepi gambar.
# ----------------------------------------------------------------------
MEDIAN_X_M = 7.0           # batas dua arah (x dunia, meter)
LINE_Y_AWAY_M = 85.0       # garis untuk kendaraan yang menjauh (meter dari kamera)

# ----------------------------------------------------------------------
# Profil kamera dari antarmuka web. Bila variabel lingkungan PROFIL_KAMERA berisi jalur berkas
# JSON, nilai kalibrasi dan garis hitung di atas ditimpa oleh isi berkas itu (per kamera/node).
# Tanpa variabel itu, config ini berlaku apa adanya.
# ----------------------------------------------------------------------
import os as _os
_profil_kamera = _os.environ.get("PROFIL_KAMERA")
if _profil_kamera and _os.path.exists(_profil_kamera):
    import json as _json
    with open(_profil_kamera, encoding="utf-8") as _f:
        for _k, _v in _json.load(_f).items():
            if _k in ("CALIB_XML", "PITCH_DEG", "CAM_HEIGHT_M", "FOCAL_PX", "LANE_HALF_WIDTH_M",
                      "MAX_TRACK_DEPTH_M", "LINE_Y_DEFAULT_M", "LINE_Y_AWAY_M", "MEDIAN_X_M"):
                globals()[_k] = _v


### `detection_cache.py`

Menyimpan dan memuat deteksi mentah per frame, supaya video yang sama tidak perlu dideteksi ulang.

*Asli: `~/projects/uji_session0_pipeline/detection_cache.py`, 37 baris.*

In [ ]:
%%writefile kode_notebook/pipeline/detection_cache.py
"""
Cache deteksi mentah, supaya detektor cukup dijalankan sekali dan percobaan
tracking berikutnya jauh lebih cepat. Format pickle (kompatibel dengan file
.pkl lama): {"nms": "pool", "min_conf": float, "frames": [daftar box per frame]}.

PERINGATAN: pickle dapat menjalankan kode saat dimuat. Hanya muat file yang
Anda buat sendiri.
"""

import pickle


def load(path, min_conf):
    with open(path, "rb") as f:
        cache = pickle.load(f)
    if cache.get("nms", "pool") != "pool":
        raise SystemExit(
            f"Cache dibuat dengan NMS '{cache['nms']}', sedangkan kode ini hanya "
            "memakai 'pool'. Buat ulang cache."
        )
    if min_conf < cache["min_conf"] - 1e-9:
        raise SystemExit(
            f"Ambang {min_conf} lebih rendah dari ambang cache ({cache['min_conf']}). "
            "Buat ulang cache dengan --cache-conf yang lebih rendah."
        )
    return cache["frames"]


def load_frames(path):
    """Hanya daftar box per frame (untuk alat pelabelan)."""
    with open(path, "rb") as f:
        return pickle.load(f)["frames"]


def save(path, frames, min_conf):
    with open(path, "wb") as f:
        pickle.dump({"nms": "pool", "min_conf": min_conf, "frames": frames}, f)


### `counter.py`

Penghitung garis: kendaraan dihitung sekali saat melintasi garisnya (50 m untuk yang mendekat, 85 m untuk yang menjauh), lalu dicatat ke `counting_log.csv`.

*Asli: `~/projects/uji_session0_pipeline/counter.py`, 144 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/counter.py
"""
Counting dengan garis virtual.

Garis berada pada kedalaman `line_y` (meter dari kamera) dan membentang selebar
jalur valid. Sebuah track dianggap melintas saat posisi y-nya berpindah dari
satu sisi garis ke sisi lain (arah tidak dibedakan). Setiap track dihitung
paling banyak sekali, dengan kelas hasil voting saat itu, dan hanya setelah
track layak tampil (min_hits), supaya deteksi palsu sesaat tidak terhitung.

Dasar: counting dengan garis virtual adalah praktik umum; bentuk aturan ini
[heuristik proyek] dan belum divalidasi terhadap hitungan manual.
"""

import csv

import config as cfg

HEADER = ["frame", "id", "kelas", "panjang_m", "lebar_m", "tinggi_m", "x_m"] + [
    f"suara_{c}" for c in cfg.CLASSES
]




#### kelas `_State`

Bersama: kelas `LineCounter`

In [ ]:
%%writefile -a kode_notebook/pipeline/counter.py
class _State:
    __slots__ = ("prev_y", "crossed", "counted")

    def __init__(self):
        self.prev_y = None
        self.crossed = False
        self.counted = False


class LineCounter:
    def __init__(self, line_y=cfg.LINE_Y_DEFAULT_M, line_y_away=None):
        self.line_y = line_y
        self.line_y_away = line_y if line_y_away is None else line_y_away
        self.counts = {c: 0 for c in cfg.CLASSES}
        self.events = []    # satu baris per kendaraan yang dihitung (format HEADER)
        self.history = {}   # display_id -> ringkasan hidup track yang pernah tampil
        self._state = {}    # track.id -> _State

    def line_for(self, x):
        """Garis yang berlaku untuk posisi lateral x (kanan median = garis menjauh)."""
        return self.line_y_away if x >= cfg.MEDIAN_X_M else self.line_y

    def total(self):
        return sum(self.counts.values())

    def update(self, tracks, frame_idx):
        """Panggil sekali per frame, setelah Tracker.assign_display_ids()."""
        for t in tracks:
            st = self._state.setdefault(t.id, _State())
            y = t.y()
            line = self.line_for(t.x())
            prev, st.prev_y = st.prev_y, y

            # Lintasan dicatat walau track sedang tidak terlihat, supaya tidak hilang.
            if (
                prev is not None
                and not st.crossed
                and prev != y
                and (prev - line) * (y - line) <= 0
            ):
                st.crossed = True

            if st.crossed and not st.counted and t.display_id is not None:
                st.counted = True
                self._count(t, frame_idx)

            if t.display_id is not None:
                self._record(t, st, y, frame_idx)

    def _count(self, t, frame_idx):
        cls = t.cls
        self.counts[cls] += 1
        w, h, l = t.classifier.median_dims()
        votes = t.classifier.votes
        self.events.append(
            [
                frame_idx,
                t.display_id,
                cls,
                round(float(l), 2),
                round(float(w), 2),
                round(float(h), 2),
                round(float(t.x()), 2),
            ]
            + [round(float(votes[c]), 3) for c in cfg.CLASSES]
        )

    def _record(self, t, st, y, frame_idx):
        rec = self.history.setdefault(
            t.display_id,
            {"first": frame_idx, "y_first": y, "y_min": y, "y_max": y, "x_first": t.x()},
        )
        rec["last"] = frame_idx
        rec["y_last"] = y
        rec["y_min"] = min(rec["y_min"], y)
        rec["y_max"] = max(rec["y_max"], y)
        rec["cls"] = t.cls
        rec["counted"] = st.counted

    # ------------------------------------------------------------------
    def save_log(self, path=cfg.COUNT_LOG):
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(HEADER)
            writer.writerows(self.events)

    def print_report(self, log_path=cfg.COUNT_LOG):
        print(f"\nHasil counting (disimpan di {log_path}):")
        for c in cfg.CLASSES:
            print(f"  {c:6}: {self.counts[c]}")
        print(f"  total : {self.total()}")

        uncounted = {k: v for k, v in self.history.items() if not v["counted"]}
        print(
            f"\nTrack yang tampil: {len(self.history)}, "
            f"dihitung: {len(self.history) - len(uncounted)}, "
            f"tidak dihitung: {len(uncounted)}"
        )
        if uncounted:
            print("Track tampil tapi tidak dihitung:")
            print("  id   kelas   frame (awal-akhir)   y awal -> y akhir (m)   x awal   keterangan")
            for k in sorted(uncounted):
                v = uncounted[k]
                line = self.line_for(v["x_first"])
                if v["y_min"] > line:
                    note = "tidak pernah sampai garis (selalu lebih jauh dari garis)"
                elif v["y_max"] < line:
                    note = "lahir setelah garis (selalu lebih dekat dari garis)"
                else:
                    note = "melewati garis tapi tidak terhitung (cek)"
                print(
                    f"  {k:3}  {v['cls']:6}  {v['first']:5}-{v['last']:<5}"
                    f"  {v['y_first']:6.1f} -> {v['y_last']:6.1f}  {v['x_first']:6.1f}   {note}"
                )

        print("\nDetail tiap kendaraan yang dihitung:")
        print("  frame   id  kelas   panjang  lebar  tinggi  x(m)  suara(car/van/truck)")
        for e in self.events:
            print(
                f"  {e[0]:5}  {e[1]:3}  {e[2]:6}  {e[3]:6.2f}  {e[4]:5.2f}  {e[5]:6.2f}"
                f"  {e[6]:5.1f}  {'/'.join(str(v) for v in e[7:])}"
            )


### `geometry.py`

Model kamera: membangun K dan transformasi kamera, proyeksi dunia → gambar, dan gambar → bidang jalan. Bila `CALIB_XML` diisi, kamera dibangun dari berkas kalibrasi.

*Asli: `~/projects/uji_session0_pipeline/geometry.py`, 110 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/geometry.py
"""
Geometri kamera: proyeksi 3D -> 2D dan proyeksi balik piksel ke bidang jalan.

Konvensi dunia: x = kanan, y = atas (0 = permukaan jalan), z = maju.
Pada Track, "y" milik Kalman Filter adalah z dunia.

Dasar: model kamera lubang jarum dan perpotongan sinar dengan bidang
y = 0 (Hartley & Zisserman, Multiple View Geometry, 2004, bab 6).
"""

import xml.etree.ElementTree as ET

import numpy as np

import config as cfg




#### fungsi `pitch_matrix` — Rotasi sudut `angle` (radian) di sumbu X.

Bersama: fungsi `build_camera` — Kembalikan (K, to_world_from_camera). Titik pusat citra = tengah frame.; fungsi `build_camera_from_calib` — Kamera dari berkas kalibrasi BrnoCompSpeed (calibParams.xml): fokus f (px pada ref_size),

In [ ]:
%%writefile -a kode_notebook/pipeline/geometry.py
def pitch_matrix(angle):
    """Rotasi sudut `angle` (radian) di sumbu X."""
    m = np.zeros((3, 3))
    m[0, 0] = 1
    m[1, 1] = np.cos(angle)
    m[1, 2] = np.sin(angle)
    m[2, 1] = -np.sin(angle)
    m[2, 2] = np.cos(angle)
    return m


def build_camera(pitch_deg, height_m, focal_px, img_w, img_h):
    """Kembalikan (K, to_world_from_camera). Titik pusat citra = tengah frame."""
    to_world = np.identity(4)
    to_world[:3, :3] = pitch_matrix(-pitch_deg * np.pi / 180)
    to_world[:3, 3] = np.array([0.0, height_m, 0.0])
    K = np.array(
        [[focal_px, 0.0, img_w / 2], [0.0, focal_px, img_h / 2], [0.0, 0.0, 1.0]]
    )
    return K, to_world


def build_camera_from_calib(xml_path, img_w, img_h, ref_size=(1920, 1080)):
    """Kamera dari berkas kalibrasi BrnoCompSpeed (calibParams.xml): fokus f (px pada ref_size),
    tinggi kamera h (mm), dan dua titik hilang (arah jalan dan arah melintang jalan).
    Rotasi kamera dihitung dari kedua titik hilang (pitch, yaw, dan roll sekaligus), sama dengan
    kamera.py yang dipakai membuat label 3D data latih. Kembalikan (K, to_world_from_camera).
    Konvensi dunia sama dengan build_camera: x = kanan, y = atas, z = maju; kamera di (0, h, 0)."""
    cal = ET.parse(xml_path).getroot()
    f_ref = float(cal.findtext("f"))
    h_m = float(cal.findtext("h")) / 1000.0
    vp = [float(t) for t in cal.findtext("vanishPoints").split()]
    ref_w, ref_h = ref_size
    k_ref = np.array([[f_ref, 0, ref_w / 2], [0, f_ref, ref_h / 2], [0, 0, 1.0]])

    def direction(p):
        d = np.linalg.inv(k_ref) @ np.array([p[0], p[1], 1.0])
        return d / np.linalg.norm(d)

    d1, d2 = direction(vp[0:2]), direction(vp[2:4])
    if d1[2] < 0:
        d1 = -d1
    if d2[0] < 0:
        d2 = -d2
    x_ax = d2 - (d2 @ d1) * d1
    x_ax /= np.linalg.norm(x_ax)
    z_ax = np.cross(x_ax, d1)
    if z_ax[1] > 0:
        z_ax = -z_ax
    flip = np.array([1.0, -1.0, 1.0])
    r_wc = np.stack([x_ax * flip, z_ax * flip, d1 * flip], axis=0)
    to_world = np.identity(4)
    to_world[:3, :3] = r_wc
    to_world[:3, 3] = [0.0, h_m, 0.0]
    f = f_ref * img_w / ref_w
    k = np.array([[f, 0.0, img_w / 2], [0.0, f, img_h / 2], [0.0, 0.0, 1.0]])
    return k, to_world


if getattr(cfg, "CALIB_XML", None):
    K, TO_WORLD_FROM_CAMERA = build_camera_from_calib(
        cfg.CALIB_XML, cfg.IMG_W, cfg.IMG_H, getattr(cfg, "CALIB_REF_SIZE", (1920, 1080))
    )
else:
    K, TO_WORLD_FROM_CAMERA = build_camera(
        cfg.PITCH_DEG, cfg.CAM_HEIGHT_M, cfg.FOCAL_PX, cfg.IMG_W, cfg.IMG_H
    )
TO_CAMERA_FROM_WORLD = np.linalg.inv(TO_WORLD_FROM_CAMERA)
CAMERA_POS = TO_WORLD_FROM_CAMERA[:3, 3].copy()




#### fungsi `project` — Titik dunia (3,) -> piksel (u, v) dengan v dihitung dari BAWAH citra.

Bersama: fungsi `world_to_image` — Titik dunia -> (u, v) piksel OpenCV (v dari ATAS citra), bertipe int.; fungsi `get_position_ground` — Piksel (x, y) OpenCV -> posisi dunia pada bidang jalan (y dunia = 0).

In [ ]:
%%writefile -a kode_notebook/pipeline/geometry.py
def project(point_in_world, to_camera_from_world=TO_CAMERA_FROM_WORLD, k=K):
    """Titik dunia (3,) -> piksel (u, v) dengan v dihitung dari BAWAH citra."""
    p = np.asarray(point_in_world, dtype=float).reshape(3, 1)
    pc = to_camera_from_world[:3, :3].dot(p) + to_camera_from_world[:3, 3].reshape(3, 1)
    pc /= abs(pc[2])
    return k.dot(pc).flatten()[:2]


def world_to_image(point_in_world, img_h=cfg.IMG_H):
    """Titik dunia -> (u, v) piksel OpenCV (v dari ATAS citra), bertipe int."""
    pix = project(point_in_world)
    return int(pix[0]), int(img_h - pix[1])


def get_position_ground(x, y, img_h=cfg.IMG_H):
    """Piksel (x, y) OpenCV -> posisi dunia pada bidang jalan (y dunia = 0)."""
    pc = np.linalg.inv(K).dot(np.array([x, img_h - y, 1.0]).reshape(3, 1))
    pw = TO_WORLD_FROM_CAMERA[:3, :3].dot(pc) + CAMERA_POS.reshape(3, 1)
    direction = pw - CAMERA_POS.reshape(3, 1)
    direction /= np.linalg.norm(direction)
    lam = -CAMERA_POS[1] / direction[1]
    return (CAMERA_POS.reshape(3, 1) + lam * direction).flatten()


### `detection_filter.py`

Memproyeksikan deteksi ke bidang jalan (meter) dan membuang yang di luar jalur atau berukuran tidak wajar.

*Asli: `~/projects/uji_session0_pipeline/detection_filter.py`, 60 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/detection_filter.py
"""
Pasca-proses deteksi: proyeksikan titik kontak ban ke bidang jalan, ubah
dimensi relatif model menjadi meter, lalu buang deteksi yang tidak masuk akal.

Keluaran detektor: x, y (piksel), w, h, l (rasio terhadap jarak). Meter =
rasio * depth, dengan depth = jarak titik kontak dari kamera [kode asal,
proyek antoinekeller/3d_bbox_highway]. Ambang filter: lihat config.py [heuristik].
"""

import numpy as np

import config as cfg
from geometry import CAMERA_POS, get_position_ground




#### fungsi `reject_reason` — Alasan deteksi dibuang, atau None bila lolos.

Bersama: fungsi `to_world_detections` — boxes      : keluaran Detector (atau cache).

In [ ]:
%%writefile -a kode_notebook/pipeline/detection_filter.py
def reject_reason(box, ground, depth):
    """Alasan deteksi dibuang, atau None bila lolos."""
    if abs(ground[0]) > cfg.LANE_HALF_WIDTH_M:
        return "luar jalur"
    if box["w"] * depth < cfg.MIN_WIDTH_M:
        return "w kecil"
    h_lo, h_hi = cfg.HEIGHT_RANGE_M
    if box["h"] * depth < h_lo or box["h"] * depth > h_hi:
        return "tinggi"
    if box["l"] * depth < cfg.MIN_LENGTH_M:
        return "l pendek"
    if box["h"] > box["l"]:
        return "h>l"
    return None


def to_world_detections(boxes, conf_split=None):
    """
    boxes      : keluaran Detector (atau cache).
    conf_split : ambang --conf bila asosiasi dua tahap aktif (hanya untuk
                 menandai deteksi skor rendah di mode --debug).
    Mengembalikan (deteksi dunia, tanda debug, hitungan ditolak per alasan).
    Tanda debug: (x_px, y_px, alasan atau None, skor_rendah).
    """
    world, marks, rejected = [], [], {}
    for box in boxes:
        ground = get_position_ground(box["x"], box["y"])
        depth = float(np.linalg.norm(ground - CAMERA_POS))
        reason = reject_reason(box, ground, depth)
        low = conf_split is not None and box["score"] < conf_split
        marks.append((box["x"], box["y"], reason, low))
        if reason is not None:
            rejected[reason] = rejected.get(reason, 0) + 1
            continue
        world.append(
            {
                "x": float(ground[0]),
                "y": float(ground[2]),
                "w": min(box["w"] * depth, cfg.MAX_WIDTH_M),
                "h": box["h"] * depth,
                "l": box["l"] * depth,
                "score": box["score"],
            }
        )
    return world, marks, rejected


### `kalman_filter.py`

Kalman Filter kecepatan konstan untuk posisi dan ukuran tiap kendaraan.

*Asli: `~/projects/uji_session0_pipeline/kalman_filter.py`, 73 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/kalman_filter.py
"""
Kalman Filter untuk melacak kendaraan pada bidang jalan.

State (6):       x, y, vy, w, h, l
Measurement (5): x, y, w, h, l   (semua dalam METER)

Asumsi: x konstan (jarang pindah lajur), y bergerak dengan kecepatan konstan
(vy), dimensi konstan. vy WAJIB ada di state karena dipakai untuk
memprediksi posisi di frame berikutnya; yang dihapus hanya tampilannya.

Dasar: Kalman (1960); constant velocity dan bentuk Joseph untuk update
covariance: Bar-Shalom dkk. (2001). Lihat config.py untuk besaran Q dan R.
"""

import numpy as np

import config as cfg

TIME_INTERVAL_FALLBACK = 1.0 / 30.0  # dipakai bila FPS video tidak terbaca




#### fungsi `measurement_noise` — Noise pengukuran membesar dengan jarak [heuristik, lihat config.py].

Bersama: kelas `KalmanFilter`

In [ ]:
%%writefile -a kode_notebook/pipeline/kalman_filter.py
def measurement_noise(depth):
    """Noise pengukuran membesar dengan jarak [heuristik, lihat config.py]."""
    depth = max(float(depth), 0.0)

    def std(par):
        a, b, p = par
        return a + b * depth**p

    return np.diag([std(cfg.KF_STD_X) ** 2, std(cfg.KF_STD_Y) ** 2] + [std(cfg.KF_STD_DIM) ** 2] * 3)


class KalmanFilter:
    NX = 6
    NZ = 5

    def __init__(self, first_measurement, dt):
        z = np.asarray(first_measurement, dtype=float).reshape(-1)
        assert z.shape == (self.NZ,)
        self.dt = dt
        self.estimate = np.array([z[0], z[1], 0.0, z[2], z[3], z[4]]).reshape(self.NX, 1)
        self.P = np.diag(np.square(cfg.KF_INIT_STD))

        # matriks pengukuran: z = (x, y, w, h, l) dari state (x, y, vy, w, h, l)
        self.H = np.zeros((self.NZ, self.NX))
        for row, col in enumerate((0, 1, 3, 4, 5)):
            self.H[row, col] = 1.0

        # constant velocity pada y
        self.F = np.identity(self.NX)
        self.F[1, 2] = dt

        self.Q = np.zeros((self.NX, self.NX))
        self.Q[0, 0] = (dt * cfg.LATERAL_STD) ** 2
        self.Q[1, 1] = (dt * dt * cfg.MAX_ACCEL) ** 2 + 1e-4
        self.Q[2, 2] = (dt * cfg.MAX_ACCEL) ** 2
        for i in (3, 4, 5):
            self.Q[i, i] = cfg.DIM_STD_PER_STEP**2

        self.I = np.identity(self.NX)

    def predict(self):
        self.estimate = self.F.dot(self.estimate)
        self.P = self.F.dot(self.P).dot(self.F.T) + self.Q

    def update(self, measurement):
        z = np.asarray(measurement, dtype=float).reshape(self.NZ, 1)
        R = measurement_noise(z[1, 0])  # R bergantung pada jarak pengukuran
        S = self.H.dot(self.P).dot(self.H.T) + R
        K = self.P.dot(self.H.T).dot(np.linalg.inv(S))
        self.estimate = self.estimate + K.dot(z - self.H.dot(self.estimate))
        IKH = self.I - K.dot(self.H)
        self.P = IKH.dot(self.P).dot(IKH.T) + K.dot(R).dot(K.T)  # bentuk Joseph


### `box3d.py`

Kotak 3D untuk tampilan: campuran templat kelas dan ukuran terukur, serta arah dari lintasan.

*Asli: `~/projects/uji_session0_pipeline/box3d.py`, 74 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/box3d.py
"""
Kotak 3D berorientasi untuk tampilan: dimensi dan arah (yaw). Hanya geometri,
tidak memengaruhi tracking, klasifikasi, maupun counting.

Konvensi: x = lateral, y = tinggi (0 = jalan), z = kedalaman.
Seluruh pencampuran ukuran dan penghalusan yaw adalah [heuristik proyek].
"""

import math
from collections import deque

import numpy as np

import config as cfg




#### fungsi `blended_dims` — (w, h, l) = campuran template kelas dan median terukur, di-clip per kelas.

Bersama: kelas `YawEstimator` — Arah kendaraan dari lintasan posisi, dihaluskan dengan EMA.

In [ ]:
%%writefile -a kode_notebook/pipeline/box3d.py
def blended_dims(cls, dim_buf):
    """(w, h, l) = campuran template kelas dan median terukur, di-clip per kelas."""
    tpl, lim = cfg.TEMPLATES[cls], cfg.LIMITS[cls]
    if len(dim_buf) == 0:
        return tpl["w"], tpl["h"], tpl["l"]

    med = np.median(np.array(dim_buf), axis=0)
    a = cfg.MEASURED_WEIGHT * min(1.0, len(dim_buf) / cfg.DIM_WARMUP)
    out = []
    for key, measured in zip(("w", "h", "l"), med):
        lo, hi = lim[key]
        out.append(float(np.clip((1 - a) * tpl[key] + a * measured, lo, hi)))
    return tuple(out)


class YawEstimator:
    """Arah kendaraan dari lintasan posisi, dihaluskan dengan EMA."""

    def __init__(self):
        self.pos_buf = deque(maxlen=cfg.YAW_WINDOW)
        self.yaw = 0.0
        # EMA pada sudut ganda (mobil simetris 180 derajat, tanpa masalah wrap-around)
        self._vec = np.array([1.0, 0.0])

    def update(self, x, z):
        self.pos_buf.append((x, z))
        if len(self.pos_buf) < 5:
            return
        x0, z0 = self.pos_buf[0]
        x1, z1 = self.pos_buf[-1]
        dx, dz = x1 - x0, z1 - z0
        if math.hypot(dx, dz) < cfg.YAW_MIN_DISPLACEMENT_M:
            return  # hampir diam, pertahankan yaw sebelumnya

        theta = math.atan2(dx, dz)  # 0 = lurus searah sumbu z
        if theta > math.pi / 2:  # maju atau mundur sama saja
            theta -= math.pi
        elif theta < -math.pi / 2:
            theta += math.pi

        meas = np.array([math.cos(2 * theta), math.sin(2 * theta)])
        self._vec = (1 - cfg.YAW_ALPHA) * self._vec + cfg.YAW_ALPHA * meas
        yaw = 0.5 * math.atan2(self._vec[1], self._vec[0])
        limit = math.radians(cfg.YAW_MAX_DEG)
        self.yaw = float(np.clip(yaw, -limit, limit))




#### fungsi `box_corners` — 8 sudut kotak di dunia. Indeks 0-3 = alas, 4-7 = atap.

In [ ]:
%%writefile -a kode_notebook/pipeline/box3d.py
def box_corners(cx, cz, dims, yaw):
    """8 sudut kotak di dunia. Indeks 0-3 = alas, 4-7 = atap."""
    w, h, l = dims
    f = np.array([math.sin(yaw), math.cos(yaw)])   # arah panjang
    r = np.array([math.cos(yaw), -math.sin(yaw)])  # arah lebar
    corners = []
    for height in (0.0, h):
        for sl, sw in ((1, 1), (1, -1), (-1, -1), (-1, 1)):
            p = np.array([cx, cz]) + sl * (l / 2) * f + sw * (w / 2) * r
            corners.append(np.array([p[0], height, p[1]]))
    return corners


### `klasifikasi_gambar.py`

Memuat klasifier ResNet-18, memotong gambar kendaraan dari kotak 3D yang diproyeksikan, dan mengirim probabilitas 4 kelas ke track.

*Asli: `~/projects/uji_session0_pipeline/klasifikasi_gambar.py`, 116 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/klasifikasi_gambar.py
"""
Klasifier gambar kendaraan (ResNet-18, 4 kelas) untuk pipeline.

Detektor CenterNet tidak punya kotak 2D, jadi crop diambil dari kotak 3D track yang
diproyeksikan ke gambar (persegi panjang pembatas 8 sudut + margin kecil).
Praproses SAMA dengan latihan (latih_klasifikasi.py): letterbox 128 px, isi abu-abu
rata-rata ImageNet, RGB, normalisasi ImageNet.
torch hanya diimpor bila klasifier dipakai (--tanpa-gambar tidak memerlukannya).
"""

import os

import cv2
import numpy as np

import config as cfg
from box3d import box_corners
from geometry import world_to_image

# urutan keluaran jaringan = KELAS pada latih_klasifikasi.py
NAMA_LATIH = ("mobil_pribadi", "truk", "bus", "van_pickup")
KE_KELAS = {"mobil_pribadi": "car", "truk": "truck", "bus": "bus", "van_pickup": "van"}
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
PAD_BGR = (104, 116, 124)




#### fungsi `letterbox`

Bersama: fungsi `ke_array`; fungsi `crop_dari_track` — Crop BGR dari kotak 3D track, atau None bila terlalu kecil / terlalu terpotong.

In [ ]:
%%writefile -a kode_notebook/pipeline/klasifikasi_gambar.py
def letterbox(img, ukuran):
    h, w = img.shape[:2]
    s = ukuran / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    kecil = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    out = np.empty((ukuran, ukuran, 3), np.uint8)
    out[:] = PAD_BGR
    y0, x0 = (ukuran - nh) // 2, (ukuran - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = kecil
    return out


def ke_array(img_bgr):
    rgb = img_bgr[:, :, ::-1].astype(np.float32) / 255.0
    return np.ascontiguousarray(((rgb - MEAN) / STD).transpose(2, 0, 1), dtype=np.float32)


def crop_dari_track(frame, track):
    """Crop BGR dari kotak 3D track, atau None bila terlalu kecil / terlalu terpotong."""
    H, W = frame.shape[:2]
    pts = np.array([world_to_image(c, H) for c in box_corners(track.x(), track.y(), track.dims(), track.yaw)],
                   dtype=np.float64)
    x0, y0 = pts.min(axis=0)
    x1, y1 = pts.max(axis=0)
    bw, bh = x1 - x0, y1 - y0
    if bw < 2 or bh < 2:
        return None
    x0 -= cfg.KLAS_MARGIN * bw; x1 += cfg.KLAS_MARGIN * bw
    y0 -= cfg.KLAS_MARGIN * bh; y1 += cfg.KLAS_MARGIN * bh
    area = (x1 - x0) * (y1 - y0)
    cx0, cy0, cx1, cy1 = max(0, int(x0)), max(0, int(y0)), min(W, int(round(x1))), min(H, int(round(y1)))
    if cx1 - cx0 < 2 or cy1 - cy0 < 2:
        return None
    if (cx1 - cx0) * (cy1 - cy0) / area < cfg.KLAS_MIN_VISIBLE:
        return None
    if max(cx1 - cx0, cy1 - cy0) < cfg.KLAS_MIN_SIDE_PX:
        return None
    return frame[cy0:cy1, cx0:cx1]




#### kelas `KlasifierGambar`

In [ ]:
%%writefile -a kode_notebook/pipeline/klasifikasi_gambar.py
class KlasifierGambar:
    def __init__(self, path, simpan_crop=None, predict_fn=None):
        self.simpan_crop = simpan_crop
        self._n_simpan = 0
        if simpan_crop:
            os.makedirs(simpan_crop, exist_ok=True)
        if predict_fn is not None:        # untuk uji tanpa torch
            self._predict = predict_fn
            return
        import torch, torchvision          # noqa: E401
        torch.set_num_threads(getattr(cfg, "TORCH_THREADS", 2))
        m = torchvision.models.resnet18(weights=None)
        m.fc = torch.nn.Linear(m.fc.in_features, len(NAMA_LATIH))
        m.load_state_dict(torch.load(path, map_location="cpu"))
        m.eval()
        self._torch, self._model = torch, m

        def _predict(batch):              # batch: (N,3,S,S) float32 -> (N,4) probabilitas
            with torch.no_grad():
                return torch.softmax(m(torch.from_numpy(batch)), dim=1).numpy()
        self._predict = _predict

    def perbarui(self, frame, tracks, frame_idx):
        """Klasifikasi gambar untuk track yang baru diperbarui; hasilnya masuk ke track.classifier."""
        pilih, crops = [], []
        for t in tracks:
            if t.display_id is None or t.unassociated_counter != 0:
                continue
            if (frame_idx + t.id) % cfg.KLAS_EVERY != 0:   # sebar beban antar track
                continue
            cr = crop_dari_track(frame, t)
            if cr is None:
                continue
            pilih.append(t)
            crops.append(ke_array(letterbox(cr, cfg.KLAS_UKURAN)))
            if self.simpan_crop and self._n_simpan < 400:
                cv2.imwrite(os.path.join(self.simpan_crop, "f%04d_id%d.jpg" % (frame_idx, t.display_id)), cr)
                self._n_simpan += 1
        if not pilih:
            return
        probs = self._predict(np.stack(crops))
        for t, p in zip(pilih, probs):
            t.classifier.add_image({KE_KELAS[n]: float(v) for n, v in zip(NAMA_LATIH, p)})




#### fungsi `buat_klasifier_gambar`

In [ ]:
%%writefile -a kode_notebook/pipeline/klasifikasi_gambar.py
def buat_klasifier_gambar(path, simpan_crop=None):
    if not os.path.exists(path):
        raise SystemExit("Bobot klasifier gambar tidak ada: %s (pakai --klas JALUR atau --tanpa-gambar)" % path)
    return KlasifierGambar(path, simpan_crop)


### `classifier.py`

Klasifikasi per track: rata-rata probabilitas gambar dikalikan rata-rata kecocokan panjang. Tanpa gambar, kelas dari panjang saja.

*Asli: `~/projects/uji_session0_pipeline/classifier.py`, 88 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/classifier.py
"""
Klasifikasi kendaraan: car / van / truck / bus.

Dua sumber bukti per track:
  1. PANJANG terukur (proyeksi ke bidang jalan): prior lunak per kelas dari config.LIMITS["l"].
     Dipanggil tiap deteksi lewat update(w, h, l).
  2. GAMBAR: probabilitas klasifier ResNet-18 (klasifikasi_gambar.py) lewat add_image(probs).
Skor tiap sumber dirata-ratakan sepanjang hidup track, lalu dikalikan:
  skor[c] = (rata_gambar[c] + EPS) * (rata_panjang[c] + EPS), dinormalkan.
Tanpa pengamatan gambar, kelas ditentukan dari panjang saja (perilaku lama).
Kelas akhir = skor terbesar; seri dimenangkan kelas lebih awal di config.CLASSES.

Dasar: cara menggabungkan ini [heuristik proyek], belum ada rujukan.
Catatan: panjang tidak bisa membedakan truck dan bus (rentangnya tumpang tindih);
yang membedakan hanya gambar.
"""

from collections import deque
import math

import numpy as np

import config as cfg

EPS = 0.02




#### fungsi `classify_by_length` — Kelas dari panjang saja (ambang tegas); dipakai sebagai cadangan/uji.

Bersama: fungsi `length_prior` — Prior lunak per kelas dari panjang: 1 di dalam rentang LIMITS[c]["l"], meluruh di luarnya.; kelas `VehicleClassifier`

In [ ]:
%%writefile -a kode_notebook/pipeline/classifier.py
def classify_by_length(length):
    """Kelas dari panjang saja (ambang tegas); dipakai sebagai cadangan/uji."""
    if length < cfg.CAR_MAX_LEN_M:
        return "car"
    if length < cfg.VAN_MAX_LEN_M:
        return "van"
    return "truck"


def length_prior(length):
    """Prior lunak per kelas dari panjang: 1 di dalam rentang LIMITS[c]["l"], meluruh di luarnya."""
    raw = {}
    for c in cfg.CLASSES:
        lo, hi = cfg.LIMITS[c]["l"]
        d = max(lo - length, length - hi, 0.0)
        raw[c] = max(cfg.LEN_FLOOR, math.exp(-((d / cfg.LEN_SIGMA_M) ** 2)))
    s = sum(raw.values())
    return {c: v / s for c, v in raw.items()}


class VehicleClassifier:
    def __init__(self):
        self.votes = {c: 0.0 for c in cfg.CLASSES}   # skor gabungan terakhir (ditulis counter ke log)
        self.cls = cfg.CLASSES[0]
        self.dim_buf = deque(maxlen=cfg.DIM_BUF_SIZE)  # (w, h, l) terukur
        self._len_sum = {c: 0.0 for c in cfg.CLASSES}
        self._img_sum = {c: 0.0 for c in cfg.CLASSES}
        self.n_len = 0
        self.n_img = 0

    def update(self, w, h, l):
        """Dipanggil setiap kali track mendapat deteksi."""
        for c, p in length_prior(l).items():
            self._len_sum[c] += p
        self.n_len += 1
        self.dim_buf.append((w, h, l))
        self._pilih()

    def add_image(self, probs):
        """probs: {kelas: probabilitas} dari klasifier gambar untuk satu crop."""
        for c in cfg.CLASSES:
            self._img_sum[c] += float(probs.get(c, 0.0))
        self.n_img += 1
        self._pilih()

    def _pilih(self):
        if self.n_len == 0:
            return
        len_m = {c: self._len_sum[c] / self.n_len for c in cfg.CLASSES}
        if self.n_img == 0:
            fused = len_m
        else:
            fused = {c: (self._img_sum[c] / self.n_img + EPS) * (len_m[c] + EPS) for c in cfg.CLASSES}
            s = sum(fused.values())
            fused = {c: v / s for c, v in fused.items()}
        self.votes = fused
        self.cls = max(cfg.CLASSES, key=lambda c: fused[c])  # seri: yang lebih awal menang

    def median_dims(self):
        """Median (w, h, l) terukur."""
        return np.median(np.array(self.dim_buf), axis=0)


### `tracker.py`

Pelacak: asosiasi deteksi ke track dengan algoritma Hungarian (dua tahap ala ByteTrack bila `--conf-low` diberikan), membuat dan menghapus track.

*Asli: `~/projects/uji_session0_pipeline/tracker.py`, 209 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/tracker.py
"""
Tracking multi-objek pada bidang jalan.

Alur per frame (Tracker.update):
  1. prediksi semua track (Kalman Filter, constant velocity)
  2. asosiasi Hungarian antara track dan deteksi, dengan gerbang adaptif
     - opsional dua tahap (ByteTrack): deteksi skor rendah hanya menyambung
       track yang sudah ada, tidak boleh membuat track baru
  3. deteksi skor tinggi yang tidak cocok -> track baru
  4. hapus track: hilang terlalu lama, keluar jalur, terlalu jauh, duplikat

Rujukan: lihat config.py (Kalman 1960; Kuhn 1955 / Crouse 2016; SORT 2016;
ByteTrack 2022). Bentuk gerbang adaptif dan semua angkanya [heuristik proyek].
"""

import numpy as np
from scipy.optimize import linear_sum_assignment

import config as cfg
from box3d import YawEstimator, blended_dims
from classifier import VehicleClassifier
from kalman_filter import KalmanFilter

IMPOSSIBLE_COST = 1000.0




#### kelas `Track` — State satu kendaraan: Kalman Filter, klasifikasi, dan arah (yaw).

In [ ]:
%%writefile -a kode_notebook/pipeline/tracker.py
class Track:
    """State satu kendaraan: Kalman Filter, klasifikasi, dan arah (yaw)."""

    def __init__(self, track_id, det, dt):
        self.id = track_id          # ID internal
        self.display_id = None      # nomor urut, diberikan saat pertama kali layak tampil
        self.age = 0
        self.hits = 1
        self.unassociated_counter = 0
        self.kf = KalmanFilter([det["x"], det["y"], det["w"], det["h"], det["l"]], dt)
        self.classifier = VehicleClassifier()
        self.yaw_estimator = YawEstimator()
        self._observe()

    # state Kalman: (x, y, vy, w, h, l)
    def x(self):
        return self.kf.estimate[0, 0]

    def y(self):
        return self.kf.estimate[1, 0]

    def w(self):
        return self.kf.estimate[3, 0]

    def h(self):
        return self.kf.estimate[4, 0]

    def l(self):
        return self.kf.estimate[5, 0]

    @property
    def cls(self):
        return self.classifier.cls

    @property
    def yaw(self):
        return self.yaw_estimator.yaw

    def dims(self):
        return blended_dims(self.classifier.cls, self.classifier.dim_buf)

    def dist(self, det):
        return float(np.hypot(self.x() - det["x"], self.y() - det["y"]))

    def update(self, det):
        self.kf.update([det["x"], det["y"], det["w"], det["h"], det["l"]])
        self.unassociated_counter = 0
        self.hits += 1
        self._observe()  # klasifikasi dan yaw hanya diperbarui saat ada deteksi nyata

    def _observe(self):
        self.classifier.update(self.w(), self.h(), self.l())
        self.yaw_estimator.update(self.x(), self.y())

    def is_visible(self):
        return (
            self.hits >= cfg.MIN_HITS_TO_DISPLAY
            and self.unassociated_counter <= cfg.MAX_LOST_TO_DISPLAY
        )

    def gate(self):
        """(jarak maks, jarak lateral maks): makin jauh makin longgar."""
        g = cfg.GATE_LOST if self.unassociated_counter > 0 else cfg.GATE_TRACKED
        depth = max(self.y(), 0.0)
        return (
            g["dist"][0] + g["dist"][1] * depth,
            g["lateral"][0] + g["lateral"][1] * depth,
        )




#### kelas `Tracker`

In [ ]:
%%writefile -a kode_notebook/pipeline/tracker.py
class Tracker:
    def __init__(self, dt):
        self.tracks = []
        self.dt = dt
        self.last_id = -1
        self.next_display_id = 1
        self._conflict_dist_sq = cfg.CONFLICT_DIST_M**2

    # ------------------------------------------------------------------
    def _add(self, det):
        self.last_id += 1
        self.tracks.append(Track(self.last_id, det, self.dt))

    def _kill(self, track_id):
        self.tracks = [t for t in self.tracks if t.id != track_id]

    def assign_display_ids(self):
        """Beri nomor urut ke track yang baru pertama kali layak tampil."""
        new = [t for t in self.tracks if t.display_id is None and t.is_visible()]
        new.sort(key=lambda t: t.y())  # serentak: yang paling dekat kamera lebih dulu
        for t in new:
            t.display_id = self.next_display_id
            self.next_display_id += 1

    def visible_tracks(self):
        return [t for t in self.tracks if t.is_visible() and t.display_id is not None]

    # ------------------------------------------------------------------
    def _match(self, track_idx, dets):
        """Hungarian dengan gerbang adaptif. Mengembalikan [(indeks track, indeks deteksi)]."""
        if not track_idx or not dets:
            return []
        cost = np.full((len(track_idx), len(dets)), IMPOSSIBLE_COST)
        for a, i in enumerate(track_idx):
            track = self.tracks[i]
            max_dist, max_lateral = track.gate()
            for j, det in enumerate(dets):
                d = track.dist(det)
                if d > max_dist or abs(det["x"] - track.x()) > max_lateral:
                    continue
                cost[a, j] = d
        rows, cols = linear_sum_assignment(cost)
        return [(track_idx[a], j) for a, j in zip(rows, cols) if cost[a, j] < IMPOSSIBLE_COST]

    def update(self, detections, conf_high=None):
        """
        detections: daftar dict {x, y, w, h, l, score} dalam meter (dunia).
        conf_high=None : satu tahap, semua deteksi setara.
        conf_high=X    : dua tahap (ByteTrack, Zhang dkk. 2022). Tahap 1: semua track
                         dengan deteksi skor >= X. Tahap 2: track yang tersisa dengan
                         deteksi skor < X. Track baru hanya dari deteksi skor tinggi.
        """
        for t in self.tracks:
            t.kf.predict()

        if conf_high is None:
            high, low = list(detections), []
        else:
            high = [d for d in detections if d.get("score", 1.0) >= conf_high]
            low = [d for d in detections if d.get("score", 1.0) < conf_high]

        if not self.tracks:
            for det in high:
                self._add(det)
            return

        n_old = len(self.tracks)
        track_matched = [False] * n_old
        high_matched = [False] * len(high)

        # tahap 1: semua track <-> deteksi skor tinggi
        for i, j in self._match(list(range(n_old)), high):
            self.tracks[i].update(high[j])
            track_matched[i] = True
            high_matched[j] = True

        # tahap 2: track yang belum cocok <-> deteksi skor rendah
        rest = [i for i, m in enumerate(track_matched) if not m]
        for i, j in self._match(rest, low):
            self.tracks[i].update(low[j])
            track_matched[i] = True

        # deteksi skor tinggi yang tidak cocok -> track baru
        for j, det in enumerate(high):
            if not high_matched[j]:
                self._add(det)

        for t in self.tracks:
            t.age += 1

        # track lama yang tidak mendapat deteksi (track baru ada di akhir daftar)
        snapshot = self.tracks.copy()
        for i, matched in enumerate(track_matched):
            if matched:
                continue
            t = snapshot[i]
            t.unassociated_counter += 1
            if t.x() < -cfg.LANE_HALF_WIDTH_M or t.x() >= cfg.LANE_HALF_WIDTH_M:
                self._kill(t.id)  # keluar jalur
            elif t.unassociated_counter >= cfg.MAX_LOST_FRAMES:
                self._kill(t.id)  # hilang terlalu lama

        for t in self.tracks.copy():
            if t.y() > cfg.MAX_TRACK_DEPTH_M:
                self._kill(t.id)

        # duplikat: pertahankan track yang lebih tua
        snapshot = self.tracks.copy()
        for i, a in enumerate(snapshot):
            for b in snapshot[i + 1:]:
                d2 = (a.x() - b.x()) ** 2 + (a.y() - b.y()) ** 2
                if d2 < self._conflict_dist_sq:
                    self._kill(a.id if a.age < b.age else b.id)


### `visualize.py`

Menggambar kotak 3D, label, dua garis hitung, dan panel hitungan, dengan warna per kelas.

*Asli: `~/projects/uji_session0_pipeline/visualize.py`, 120 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/visualize.py
"""Menggambar hasil: kotak 3D, label, garis counting, panel hitungan, tanda debug."""

import cv2
import numpy as np

import config as cfg
from box3d import box_corners
from geometry import project, world_to_image

# Warna BGR untuk membedakan ID
COLORS = [
    (0, 0, 128), (0, 0, 255), (0, 128, 0), (0, 128, 128), (0, 128, 255),
    (0, 255, 0), (0, 255, 128), (0, 255, 255), (128, 0, 0), (128, 0, 128),
    (128, 0, 255), (128, 128, 0), (128, 128, 128), (128, 128, 255), (128, 255, 0),
    (128, 255, 128), (128, 255, 255), (255, 0, 0), (255, 0, 128), (255, 0, 255),
    (255, 128, 0), (255, 128, 128), (255, 128, 255), (255, 255, 0), (255, 255, 128),
    (255, 255, 255),
]




#### fungsi `draw_label` — Label beberapa baris berlatar warna; bagian bawah di y, dipusatkan di x, tetap dalam frame.

Bersama: fungsi `draw_oriented_box` — Kotak 3D berorientasi milik track.

In [ ]:
%%writefile -a kode_notebook/pipeline/visualize.py
def draw_label(frame, lines, x, y, color, scale=0.6, thickness=1):
    """Label beberapa baris berlatar warna; bagian bawah di y, dipusatkan di x, tetap dalam frame."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    pad = 4
    sizes = [cv2.getTextSize(t, font, scale, thickness)[0] for t in lines]
    w = max(s[0] for s in sizes) + 2 * pad
    line_h = max(s[1] for s in sizes) + 6
    h = line_h * len(lines) + pad
    x0 = int(max(0, min(x - w / 2, frame.shape[1] - w)))
    y0 = int(max(0, min(y - h, frame.shape[0] - h)))
    cv2.rectangle(frame, (x0, y0), (x0 + w, y0 + h), color, thickness=-1)

    b, g, r = color
    text_color = (0, 0, 0) if (0.114 * b + 0.587 * g + 0.299 * r) > 140 else (255, 255, 255)
    for i, t in enumerate(lines):
        cv2.putText(frame, t, (x0 + pad, y0 + (i + 1) * line_h - 2), font, scale,
                    text_color, thickness, cv2.LINE_AA)


def draw_oriented_box(frame, track, color, fill=True):
    """Kotak 3D berorientasi milik track."""
    h_img = frame.shape[0]
    corners = box_corners(track.x(), track.y(), track.dims(), track.yaw)
    pts = [world_to_image(c, h_img) for c in corners]

    if fill:
        overlay = frame.copy()
        cv2.fillPoly(overlay, [np.array(pts[0:4], dtype=np.int32)], color)
        frame = cv2.addWeighted(overlay, 0.35, frame, 0.65, 0)

    edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
             (0, 4), (1, 5), (2, 6), (3, 7)]
    for a, b in edges:
        cv2.line(frame, pts[a], pts[b], color, 2)
    return frame




#### fungsi `draw_tracks` — Kotak 3D dan label (ID, kelas) untuk semua track yang layak tampil.

Bersama: fungsi `draw_counting_line` — Garis virtual di bidang jalan: kiri median pada line_y, kanan median pada line_y_away.

In [ ]:
%%writefile -a kode_notebook/pipeline/visualize.py
def draw_tracks(frame, tracker):
    """Kotak 3D dan label (ID, kelas) untuk semua track yang layak tampil."""
    for t in tracker.visible_tracks():
        color = cfg.CLASS_COLORS.get(t.cls, (200, 200, 200))  # warna per KELAS
        frame = draw_oriented_box(frame, t, color)

        # label di atas titik tengah atap kotak; objek jauh = font lebih kecil
        top = project(np.array([t.x(), t.dims()[1], t.y()]))
        tx, ty = top[0], frame.shape[0] - top[1]
        scale = float(np.clip(0.8 - t.y() / 80.0, 0.45, 0.8))
        draw_label(frame, [f"ID {t.display_id} ({t.cls})"], tx, ty - 6, color, scale=scale)
    return frame


def draw_counting_line(frame, line_y, line_y_away=None):
    """Garis virtual di bidang jalan: kiri median pada line_y, kanan median pada line_y_away."""
    h = frame.shape[0]
    half = cfg.LANE_HALF_WIDTH_M
    if line_y_away is None or line_y_away == line_y:
        segs = [(-half, half, line_y, (0, 0, 255))]
    else:
        segs = [(-half, cfg.MEDIAN_X_M, line_y, (0, 0, 255)),
                (cfg.MEDIAN_X_M, half, line_y_away, (255, 0, 255))]
    for xa, xb, z, col in segs:
        p0 = world_to_image(np.array([xa, 0.0, z]), h)
        p1 = world_to_image(np.array([xb, 0.0, z]), h)
        cv2.line(frame, p0, p1, col, 3)
    return frame




#### fungsi `draw_counts` — Panel hitungan per kelas di pojok kiri bawah; kotak warna = warna kelas pada kotak 3D.

Bersama: fungsi `draw_debug` — Deteksi mentah: hijau = lolos, kuning = lolos skor rendah, merah = ditolak (+alasan).

In [ ]:
%%writefile -a kode_notebook/pipeline/visualize.py
def draw_counts(frame, counts):
    """Panel hitungan per kelas di pojok kiri bawah; kotak warna = warna kelas pada kotak 3D."""
    lines = [f"Total: {sum(counts.values())}"] + [f"{c}: {counts[c]}" for c in cfg.CLASSES]
    h = frame.shape[0]
    box_h = 26 * len(lines) + 10
    cv2.rectangle(frame, (10, h - box_h - 10), (210, h - 10), (0, 0, 0), -1)
    for i, text in enumerate(lines):
        y = h - box_h + 18 + 26 * i
        if i > 0:
            col = cfg.CLASS_COLORS.get(cfg.CLASSES[i - 1], (200, 200, 200))
            cv2.rectangle(frame, (18, y - 14), (34, y + 2), col, -1)
            cv2.putText(frame, text, (42, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
        else:
            cv2.putText(frame, text, (18, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
    return frame


def draw_debug(frame, marks):
    """Deteksi mentah: hijau = lolos, kuning = lolos skor rendah, merah = ditolak (+alasan)."""
    for x, y, reason, low in marks:
        pt = (int(x), int(y))
        if reason is None:
            cv2.circle(frame, pt, 6, (0, 255, 255) if low else (0, 255, 0), -1)
        else:
            cv2.circle(frame, pt, 6, (0, 0, 255), -1)
            cv2.putText(frame, reason, (pt[0] + 8, pt[1] - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    return frame




#### fungsi `draw_frame_id`

In [ ]:
%%writefile -a kode_notebook/pipeline/visualize.py
def draw_frame_id(frame, frame_id):
    cv2.putText(frame, f"Frame {frame_id}", (frame.shape[1] - 150, frame.shape[0] - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)


### `detector.py`

Pembungkus detektor CenterNet: memuat bobot, menjalankan jaringan pada satu frame, dan mendekode keluarannya menjadi daftar deteksi.

*Asli: `~/projects/uji_session0_pipeline/detector.py`, 128 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/detector.py
"""
Deteksi kendaraan dengan CenterNet versi sederhana (ResNet-18, stride 32).

Keluaran model per frame (peta ~23x40 untuk input 1280x720):
  heatmap [1,H,W], offset [2,H,W], ukuran w/h/l [3,H,W].
Puncak heatmap diambil dengan max-pooling 3x3 sebagai pengganti NMS.

Rujukan: X. Zhou, D. Wang, P. Krahenbuhl, "Objects as Points",
arXiv:1904.07850, 2019, bagian 4 (peak extraction). Arsitektur dan bobot
awal: proyek antoinekeller/3d_bbox_highway [kode asal].
Head heatmap hanya 1 kanal, jadi detektor TIDAK membedakan jenis kendaraan.
"""

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from torch import nn
from torchvision import transforms

import config as cfg




#### kelas `CenterNet` — ResNet-18 sampai layer4 (512 kanal, stride 32) + 3 head.

Bersama: fungsi `load_model`

In [ ]:
%%writefile -a kode_notebook/pipeline/detector.py
class CenterNet(nn.Module):
    """ResNet-18 sampai layer4 (512 kanal, stride 32) + 3 head."""

    def __init__(self):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.base_model = nn.Sequential(*list(base.children())[:-2])  # buang avgpool dan fc

        def head(out_ch, num_ch=512, head_conv=64):
            return nn.Sequential(
                nn.Conv2d(num_ch, head_conv, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(head_conv, out_ch, kernel_size=1, stride=1),
            )

        self.outc = head(1)  # heatmap pusat kendaraan
        self.outo = head(2)  # offset sub-sel
        self.outr = head(3)  # ukuran w, h, l (rasio terhadap jarak)

    def forward(self, x):
        x = self.base_model(x)
        return self.outc(x), self.outo(x), self.outr(x)


def load_model(path, device="cpu"):
    model = CenterNet()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.to(device)
    model.eval()
    return model




#### fungsi `build_preprocess`

Bersama: fungsi `select_pool` — Pertahankan hanya sel yang maksimum lokal dalam jendela 3x3 (stride 1).; fungsi `decode_points` — hm: [H,W] probabilitas (sudah sigmoid), offset: [2,H,W].

In [ ]:
%%writefile -a kode_notebook/pipeline/detector.py
def build_preprocess():
    return transforms.Compose(
        [
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def select_pool(hm):
    """
    Pertahankan hanya sel yang maksimum lokal dalam jendela 3x3 (stride 1).
    Tidak bergantung pada ambang; ambang diterapkan di extract_peaks.
    Dengan stride 32, radius 1 sel ~ 32 piksel, sehingga dua mobil berdekatan
    tidak saling menekan.
    """
    t = torch.from_numpy(np.ascontiguousarray(hm)).unsqueeze(0).unsqueeze(0)
    keep = F.max_pool2d(t, kernel_size=3, stride=1, padding=1) == t
    return hm * keep.squeeze(0).squeeze(0).numpy()


def decode_points(hm, offset, conf):
    """
    hm: [H,W] probabilitas (sudah sigmoid), offset: [2,H,W].
    Mengembalikan puncak > conf sebagai dict {x, y (piksel), score, cy, cx}.
    Urutan baris-per-baris (row-major).
    """
    peaks = select_pool(hm)
    out = []
    for cy, cx in np.argwhere(peaks > conf):
        out.append(
            {
                "x": float((cx + offset[0, cy, cx]) * cfg.MODEL_SCALE),
                "y": float((cy + offset[1, cy, cx]) * cfg.MODEL_SCALE),
                "score": float(peaks[cy, cx]),
                "cy": int(cy),
                "cx": int(cx),
            }
        )
    return out




#### fungsi `decode_boxes` — Seperti decode_points, ditambah ukuran w, h, l; ukuran negatif dibuang.

Bersama: kelas `Detector`

In [ ]:
%%writefile -a kode_notebook/pipeline/detector.py
def decode_boxes(hm, offset, whl, conf):
    """Seperti decode_points, ditambah ukuran w, h, l; ukuran negatif dibuang."""
    boxes = []
    for p in decode_points(hm, offset, conf):
        w, h, l = (float(v) for v in whl[:, p["cy"], p["cx"]])
        if w < 0 or h < 0 or l < 0:
            continue
        boxes.append({"x": p["x"], "y": p["y"], "w": w, "h": h, "l": l, "score": p["score"]})
    return boxes


class Detector:
    def __init__(self, model_path, conf):
        self.conf = conf
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Device: {self.device}")
        self.model = load_model(model_path, self.device)
        self.preprocess = build_preprocess()

    def infer(self, frame):
        """Frame BGR 1280x720 -> (heatmap [H,W] sigmoid, offset [2,H,W], whl [3,H,W])."""
        x = self.preprocess(frame).float().unsqueeze(0).to(self.device)
        with torch.no_grad():
            hm, offset, whl = self.model(x)
        hm = torch.sigmoid(hm).cpu().numpy()[0, 0]
        return hm, offset.cpu().numpy()[0], whl.cpu().numpy()[0]

    def detect(self, frame):
        hm, offset, whl = self.infer(frame)
        return decode_boxes(hm, offset, whl, self.conf)


### `run.py`

Program utama pipeline: membaca video, lalu tiap frame menjalankan deteksi → proyeksi dan filter → pelacakan → klasifikasi gambar → penghitungan, dan menyimpan video hasil serta `counting_log.csv`.

*Asli: `~/projects/uji_session0_pipeline/run.py`, 197 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/run.py
"""
Program utama: deteksi -> tracking -> klasifikasi -> counting pada video jalan.

Contoh:
  python run.py VIDEO.mp4 MODEL.pth
  python run.py VIDEO.mp4 MODEL.pth --no-show --no-save --line-y 15
  python run.py VIDEO.mp4 MODEL.pth --cache det_low.pkl --conf 0.5 --conf-low 0.3 --no-show --no-save
"""

import os
from argparse import ArgumentParser

import cv2
from tqdm import tqdm

import config as cfg
import detection_cache
from counter import LineCounter
from detection_filter import to_world_detections
from kalman_filter import TIME_INTERVAL_FALLBACK
from klasifikasi_gambar import buat_klasifier_gambar
from tracker import Tracker
from visualize import draw_counting_line, draw_counts, draw_debug, draw_frame_id, draw_tracks




#### fungsi `parse_args`

In [ ]:
%%writefile -a kode_notebook/pipeline/run.py
def parse_args():
    p = ArgumentParser(description="Deteksi, tracking, klasifikasi, dan counting kendaraan")
    p.add_argument("video")
    p.add_argument("model", help="Bobot CenterNet (.pth)")
    p.add_argument("--conf", type=float, default=cfg.DEFAULT_CONF, help="Ambang skor deteksi")
    p.add_argument(
        "--conf-low", type=float, default=None,
        help="Aktifkan asosiasi dua tahap (ByteTrack, Zhang dkk. 2022): deteksi skor antara "
        "ini dan --conf hanya menyambung track yang ada. Harus > 0 dan < --conf",
    )
    p.add_argument("--line-y", type=float, default=cfg.LINE_Y_DEFAULT_M,
                   help="Posisi garis counting (meter dari kamera)")
    p.add_argument("--line-y-away", type=float, default=cfg.LINE_Y_AWAY_M,
                   help="Garis counting jalur yang MENJAUH, x >= MEDIAN_X_M (meter dari kamera)")
    p.add_argument("--cache", default=None,
                   help="File cache deteksi mentah: ada = dimuat, belum ada = dibuat")
    p.add_argument("--cache-conf", type=float, default=cfg.CACHE_CONF_DEFAULT,
                   help="Ambang skor terendah saat membuat cache")
    p.add_argument("--no-show", action="store_true", help="Tanpa jendela (SSH / VS Code remote)")
    p.add_argument("--no-save", action="store_true", help="Jangan simpan hasil (video/PNG)")
    p.add_argument("--out", default=None,
                   help="Berkas video hasil (.mp4). Bawaan: ~/projects/hasil/video/<nama_video>_hasil.mp4")
    p.add_argument("--png", action="store_true", help="Simpan juga PNG tiap frame ke folder output di config")
    p.add_argument("--klas", default=cfg.KLAS_MODEL, help="Bobot klasifier gambar 4 kelas (.pth)")
    p.add_argument("--tanpa-gambar", action="store_true", help="Klasifikasi dari panjang saja (tanpa klasifier gambar)")
    p.add_argument("--simpan-crop", default=None, help="Folder: simpan crop yang dikirim ke klasifier (untuk dicek mata)")
    p.add_argument("--play", action="store_true", help="Langsung jalan tanpa menunggu tombol")
    p.add_argument("-f", type=int, default=float("inf"),
                   help="Putar otomatis sampai frame ke-N, setelah itu berhenti tiap frame")
    p.add_argument("--debug", action="store_true",
                   help="Tandai deteksi mentah: hijau = lolos, kuning = skor rendah, merah = ditolak")
    return p.parse_args()




#### fungsi `main`

In [ ]:
%%writefile -a kode_notebook/pipeline/run.py
def main():
    args = parse_args()
    if args.conf_low is not None and not (0 < args.conf_low < args.conf):
        raise SystemExit("--conf-low harus > 0 dan lebih kecil dari --conf")
    min_conf = args.conf_low if args.conf_low is not None else args.conf
    conf_split = args.conf if args.conf_low is not None else None

    # --- sumber deteksi: cache (jika ada) atau detektor ---
    cached_frames, detector, new_frames, detect_conf = None, None, None, min_conf
    if args.cache and os.path.exists(args.cache):
        cached_frames = detection_cache.load(args.cache, min_conf)
        print(f"Memakai cache deteksi: {args.cache} ({len(cached_frames)} frame)")
    else:
        from detector import Detector  # impor torch hanya bila detektor dipakai

        detect_conf = min(args.cache_conf, min_conf) if args.cache else min_conf
        detector = Detector(args.model, detect_conf)
        new_frames = [] if args.cache else None

    cap = cv2.VideoCapture(args.video)
    if not cap.isOpened():
        raise SystemExit("Gagal membuka video")
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    dt = 1.0 / fps if fps and fps > 1 else TIME_INTERVAL_FALLBACK
    print(f"Video FPS: {fps:.1f} -> dt = {dt:.4f} s")
    print(f"Opsi: conf={args.conf} conf_low={args.conf_low} line_y={args.line_y} line_y_away={args.line_y_away}")

    writer, out_path, tmp_path = None, None, None
    if not args.no_save:
        stem = os.path.splitext(os.path.basename(args.video))[0]
        out_path = os.path.abspath(os.path.expanduser(
            args.out or os.path.join("~/projects/hasil/video", stem + "_hasil.mp4")))
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        tmp_path = out_path + ".tmp.mp4"
        writer = cv2.VideoWriter(tmp_path, cv2.VideoWriter_fourcc(*"mp4v"),
                                 fps if fps and fps > 1 else 25.0, (cfg.IMG_W, cfg.IMG_H))
        if not writer.isOpened():
            raise SystemExit(f"Gagal membuat berkas video: {tmp_path}")
        if args.png:
            os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

    tracker = Tracker(dt)
    counter = LineCounter(args.line_y, args.line_y_away)
    klas = None if args.tanpa_gambar else buat_klasifier_gambar(args.klas, args.simpan_crop)
    print("Klasifikasi:", "panjang saja" if klas is None else "gambar + panjang (" + os.path.basename(args.klas) + ")")
    rejected_total = {}
    play = args.play
    need_draw = (not args.no_show) or (not args.no_save)
    finished = True

    for idx in tqdm(range(n_frames)):
        ok, frame = cap.read()
        if not ok:
            finished = False
            break
        if frame.shape[:2] != (cfg.IMG_H, cfg.IMG_W):
            raise SystemExit(
                f"Ukuran video {frame.shape[1]}x{frame.shape[0]}, harus {cfg.IMG_W}x{cfg.IMG_H} "
                "(kalibrasi dan detektor mengasumsikan ukuran ini)."
            )

        # 1. deteksi
        if cached_frames is not None:
            raw = cached_frames[idx] if idx < len(cached_frames) else []
        else:
            raw = detector.detect(frame)
            if new_frames is not None:
                new_frames.append(raw)
        boxes = [b for b in raw if b["score"] >= min_conf]

        # 2. proyeksi ke jalan dan filter
        dets, marks, rejected = to_world_detections(boxes, conf_split)
        for reason, n in rejected.items():
            rejected_total[reason] = rejected_total.get(reason, 0) + n

        # 3. tracking, lalu klasifikasi (di dalam Track) dan counting
        tracker.update(dets, conf_high=conf_split)
        tracker.assign_display_ids()
        if klas is not None:
            klas.perbarui(frame, tracker.tracks, idx)  # frame masih bersih (belum digambar)
        counter.update(tracker.tracks, idx)

        if need_draw:
            frame = draw_tracks(frame, tracker)
            frame = draw_counting_line(frame, args.line_y, args.line_y_away)
            if args.debug:
                frame = draw_debug(frame, marks)
            frame = draw_counts(frame, counter.counts)
            draw_frame_id(frame, idx)

        if not args.no_show:
            cv2.imshow("Detection", frame)
            k = cv2.waitKey(10 if (idx < args.f or play) else 0)
            if k == 27:  # ESC
                finished = False
                break
            if k == 32:  # SPASI
                play = not play

        if writer is not None:
            writer.write(frame)
            if args.png:
                cv2.imwrite(f"{cfg.OUTPUT_DIR}/image_{idx:04d}.png", frame)

    cap.release()
    if writer is not None:
        writer.release()
        # Ubah ke H.264 bila ffmpeg ada, supaya bisa diputar di VS Code / peramban / pemutar bawaan.
        import shutil
        import subprocess
        ok = False
        if shutil.which("ffmpeg"):
            ok = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp_path, "-c:v", "libx264",
                                 "-preset", "veryfast", "-crf", "23", "-pix_fmt", "yuv420p", out_path]).returncode == 0
        if ok:
            os.remove(tmp_path)
        else:
            os.replace(tmp_path, out_path)
            print("Catatan: ffmpeg tidak dipakai; video berformat mp4v (bila tidak bisa diputar: sudo apt install ffmpeg).")
        print(f"\nVideo hasil disimpan: {out_path}")
    if not args.no_show:
        cv2.destroyAllWindows()

    if new_frames is not None and finished and len(new_frames) == n_frames:
        detection_cache.save(args.cache, new_frames, detect_conf)
        print(f"\nCache deteksi disimpan: {args.cache}")

    counter.save_log(cfg.COUNT_LOG)
    counter.print_report(cfg.COUNT_LOG)
    if rejected_total:
        print("\nDeteksi mentah yang dibuang filter (per alasan):")
        for reason, n in sorted(rejected_total.items(), key=lambda kv: -kv[1]):
            print(f"  {reason:12}: {n}")




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/pipeline/run.py
if __name__ == "__main__":
    main()


### `hitung_gt.py`

Evaluasi hitungan: menghitung kendaraan yang sebenarnya melintasi garis dari anotasi XML, lalu mencocokkannya dengan `counting_log.csv` (TP, FP, FN).

*Asli: `~/projects/alat/hitung_gt.py`, 200 baris.*

#### Impor dan konstanta

In [ ]:
%%writefile kode_notebook/pipeline/hitung_gt.py
#!/usr/bin/env python3
"""hitung_gt.py - hitungan kebenaran dasar (ground truth) dari anotasi XML session0_right, lalu
bandingkan dengan counting_log.csv hasil pipeline.

Cara kerja:
  1. Untuk tiap frame --first..--last baca XML: tiap <object> punya type, vertex2d, veh_size (p x l x t, meter).
     Titik tanah kendaraan = rata-rata 4 titik alas (4 vertex2d pertama), diskalakan ke IMG_W x IMG_H lalu
     diproyeksikan ke jalan memakai geometry.py repo (kamera yang SAMA dengan pipeline).
  2. XML tidak punya ID antar-frame, jadi kendaraan dihubungkan sendiri antar-frame (Hungarian, jarak posisi).
  3. Kendaraan dihitung "melintas" bila posisi z-nya berpindah dari satu sisi garis --line ke sisi lain
     (aturan sama dengan counter.py). z mengecil = mendekat, z membesar = menjauh.
  4. Dengan --log: tiap baris counting_log.csv dicocokkan ke kendaraan GT (selisih frame <= --tol-frame,
     selisih x <= --tol-x). Dilaporkan TP, FP (terhitung pipeline tapi tidak ada di GT), FN (ada di GT tapi
     tidak terhitung), dan galat panjang terukur.
Jalankan DARI folder repo pipeline (agar config.py dan geometry.py yang sama terbaca):
  cd ~/projects/uji_session0_pipeline
  python3 ~/projects/alat/hitung_gt.py --first 2545 --last 3181 --line 50 --log counting_log.csv
Frame video = nomor gambar - --first (video uji dimulai dari --first).
"""
import argparse, csv, glob, os, re, sys
import xml.etree.ElementTree as ET

import numpy as np
from scipy.optimize import linear_sum_assignment

sys.path.insert(0, os.getcwd())
import config as cfg                      # noqa: E402
from geometry import get_position_ground  # noqa: E402

NUM = re.compile(r"\(\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)\s*\)")




#### fungsi `baca_frame`

Bersama: fungsi `hubungkan`

In [ ]:
%%writefile -a kode_notebook/pipeline/hitung_gt.py
def baca_frame(path):
    root = ET.parse(path).getroot()
    w = float(root.findtext("size/width", "1920"))
    s = cfg.IMG_W / w
    out = []
    for o in root.findall("object"):
        pts = [(float(a), float(b)) for a, b in NUM.findall(o.findtext("vertex2d", ""))]
        if len(pts) < 4:
            continue
        gx = float(np.mean([p[0] for p in pts[:4]])) * s
        gy = float(np.mean([p[1] for p in pts[:4]])) * s
        g = get_position_ground(gx, gy)
        size = [float(v) for v in o.findtext("veh_size", "0 0 0").split()]
        out.append({"type": o.findtext("type", "?"), "x": float(g[0]), "z": float(g[2]),
                    "l": size[0], "w": size[1], "h": size[2]})
    return out


def hubungkan(frames, first):
    tracks = []
    for fi, dets in enumerate(frames):
        f = first + fi
        aktif = [t for t in tracks if f - t["last_f"] <= 3]
        cost = np.full((len(aktif), len(dets)), 1e6)
        for a, t in enumerate(aktif):
            gap = f - t["last_f"]
            px = t["pts"][-1][1] + t["vx"] * gap
            pz = t["pts"][-1][2] + t["vz"] * gap
            for j, d in enumerate(dets):
                dist = float(np.hypot(d["x"] - px, d["z"] - pz))
                if dist <= 2.5 + 0.8 * gap:
                    cost[a, j] = dist
        used = set()
        if aktif and dets:
            for a, j in zip(*linear_sum_assignment(cost)):
                if cost[a, j] < 1e6:
                    t, d = aktif[a], dets[j]
                    gap = f - t["last_f"]
                    t["vx"] = (d["x"] - t["pts"][-1][1]) / gap
                    t["vz"] = (d["z"] - t["pts"][-1][2]) / gap
                    t["pts"].append((f, d["x"], d["z"]))
                    t["last_f"] = f
                    t["ls"].append(d["l"])
                    used.add(j)
        for j, d in enumerate(dets):
            if j not in used:
                tracks.append({"pts": [(f, d["x"], d["z"])], "last_f": f, "vx": 0.0, "vz": 0.0,
                               "type": d["type"], "ls": [d["l"]]})
    return tracks




#### fungsi `persilangan`

Bersama: fungsi `main`

In [ ]:
%%writefile -a kode_notebook/pipeline/hitung_gt.py
def persilangan(t, line):
    p = t["pts"]
    for (f0, _, z0), (f1, _, z1) in zip(p, p[1:]):
        if z0 != z1 and (z0 - line) * (z1 - line) <= 0:
            frac = (line - z0) / (z1 - z0)
            return f0 + frac * (f1 - f0), ("mendekat" if z1 < z0 else "menjauh")
    return None, None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ann", default=os.path.expanduser("~/projects/data_session0_right/annotations"))
    ap.add_argument("--first", type=int, default=2545)
    ap.add_argument("--last", type=int, default=3181)
    ap.add_argument("--line", type=float, default=cfg.LINE_Y_DEFAULT_M)
    ap.add_argument("--line-away", type=float, default=getattr(cfg, "LINE_Y_AWAY_M", None),
                    help="garis untuk x >= --median-x (jalur menjauh); bawaan dari config bila ada")
    ap.add_argument("--median-x", type=float, default=getattr(cfg, "MEDIAN_X_M", 7.0))
    ap.add_argument("--log", default=None)
    ap.add_argument("--tol-frame", type=int, default=25)
    ap.add_argument("--tol-x", type=float, default=4.0)
    ap.add_argument("--min-frames", type=int, default=3, help="abaikan lintasan GT yang lebih pendek dari ini")
    ap.add_argument("--out", default="gt_hitung.csv")
    a = ap.parse_args()

    frames = []
    for n in range(a.first, a.last + 1):
        p = os.path.join(a.ann, "session0_right_%06d.xml" % n)
        frames.append(baca_frame(p) if os.path.exists(p) else [])
    tracks = [t for t in hubungkan(frames, a.first) if len(t["pts"]) >= a.min_frames]
    for k, t in enumerate(tracks):
        t["id"] = k + 1
        t["x_med"] = float(np.median([p[1] for p in t["pts"]]))
        t["l_med"] = float(np.median(t["ls"]))
        t["line"] = a.line_away if (a.line_away is not None and t["x_med"] >= a.median_x) else a.line
        t["cross_f"], t["arah"] = persilangan(t, t["line"])
        t["cross_vid"] = None if t["cross_f"] is None else t["cross_f"] - a.first

    cross = [t for t in tracks if t["cross_f"] is not None]
    print("Frame %d-%d (%d frame), garis z = %.1f m%s" % (a.first, a.last, len(frames), a.line,
          "" if a.line_away is None else "; x >= %.1f memakai garis z = %.1f m" % (a.median_x, a.line_away)))
    print("Lintasan GT: %d (>= %d frame); melintasi garis: %d" % (len(tracks), a.min_frames, len(cross)))
    for arah in ("mendekat", "menjauh"):
        ts = [t for t in cross if t["arah"] == arah]
        tipe = {}
        for t in ts:
            tipe[t["type"]] = tipe.get(t["type"], 0) + 1
        print("  %-9s: %d  %s" % (arah, len(ts), tipe))
    print("\nLintasan yang MELINTASI garis (frame video = nomor gambar - %d):" % a.first)
    print("  id  tipe    p(m)  x_med  frame_gambar   z awal->akhir   frame_melintas  arah")
    for t in sorted(cross, key=lambda t: t["cross_vid"]):
        print("  %2d  %-6s %5.2f  %5.1f  %5d-%-5d  %6.1f->%-6.1f  %8.1f      %s" % (
            t["id"], t["type"], t["l_med"], t["x_med"], t["pts"][0][0], t["pts"][-1][0],
            t["pts"][0][2], t["pts"][-1][2], t["cross_vid"], t["arah"]))
    nc = [t for t in tracks if t["cross_f"] is None]
    print("\nLintasan yang TIDAK melintasi garis: %d" % len(nc))
    print("  id  tipe    x_med  frame_gambar   z awal->akhir   keterangan")
    for t in nc:
        z0, z1 = t["pts"][0][2], t["pts"][-1][2]
        if abs(z1 - z0) < 2.0:
            ket = "diam (kendaraan parkir)"
        elif min(z0, z1) > t["line"]:
            ket = "seluruhnya lebih jauh dari garis"
        else:
            ket = "seluruhnya lebih dekat dari garis"
        print("  %2d  %-6s %5.1f  %5d-%-5d  %6.1f->%-6.1f   %s" % (
            t["id"], t["type"], t["x_med"], t["pts"][0][0], t["pts"][-1][0], z0, z1, ket))

    with open(a.out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "tipe", "panjang_m", "x_med", "frame_awal", "frame_akhir", "z_awal", "z_akhir", "melintas_frame_video", "arah"])
        for t in tracks:
            w.writerow([t["id"], t["type"], round(t["l_med"], 2), round(t["x_med"], 2), t["pts"][0][0], t["pts"][-1][0],
                        round(t["pts"][0][2], 1), round(t["pts"][-1][2], 1),
                        "" if t["cross_vid"] is None else round(t["cross_vid"], 1), t["arah"] or ""])
    print("\nDaftar lengkap GT disimpan:", a.out)

    if a.log:
        ev = list(csv.DictReader(open(a.log)))
        pasang, dipakai = [], set()
        for e in sorted(ev, key=lambda e: int(e["frame"])):
            ef, ex = int(e["frame"]), float(e["x_m"])
            best = None
            for t in cross:
                if t["id"] in dipakai:
                    continue
                df, dx = abs(ef - t["cross_vid"]), abs(ex - t["x_med"])
                if df <= a.tol_frame and dx <= a.tol_x and (best is None or df < best[0]):
                    best = (df, t)
            if best:
                dipakai.add(best[1]["id"]); pasang.append((e, best[1]))
            else:
                pasang.append((e, None))
        tp = sum(1 for _, t in pasang if t); fp = len(pasang) - tp
        fn = [t for t in cross if t["id"] not in dipakai]
        print("\nPERBANDINGAN dengan %s (toleransi %d frame, %.1f m)" % (a.log, a.tol_frame, a.tol_x))
        print("  pipeline menghitung: %d | GT melintasi: %d" % (len(ev), len(cross)))
        print("  TP (cocok): %d | FP (terhitung, tidak ada di GT): %d | FN (ada di GT, tidak terhitung): %d" % (tp, fp, len(fn)))
        print("  presisi = %s | recall = %s" % (
            "%.2f" % (tp / len(ev)) if ev else "-", "%.2f" % (tp / len(cross)) if cross else "-"))
        print("\n  Terhitung pipeline        -> kendaraan GT                 panjang pipeline vs GT")
        for e, t in pasang:
            if t:
                print("   f%-4s id%-3s %-5s x=%5.1f -> GT#%-2d %-6s x=%5.1f  %s   %.2f vs %.2f m" % (
                    e["frame"], e["id"], e["kelas"], float(e["x_m"]), t["id"], t["type"], t["x_med"], t["arah"],
                    float(e["panjang_m"]), t["l_med"]))
            else:
                print("   f%-4s id%-3s %-5s x=%5.1f -> (tidak ada di GT)  <- FP" % (e["frame"], e["id"], e["kelas"], float(e["x_m"])))
        if fn:
            print("\n  Tidak terhitung pipeline (FN):")
            for t in fn:
                print("   GT#%-2d %-6s x=%5.1f frame_gambar %d-%d z %.1f->%.1f %s (melintas di frame video %.0f)" % (
                    t["id"], t["type"], t["x_med"], t["pts"][0][0], t["pts"][-1][0], t["pts"][0][2], t["pts"][-1][2], t["arah"], t["cross_vid"]))




#### Bagian utama (`if __name__ == "__main__"`)

In [ ]:
%%writefile -a kode_notebook/pipeline/hitung_gt.py
if __name__ == "__main__":
    main()


### Menjalankan pipeline pada klip uji

Deteksi diambil dari cache, jadi selesai dalam belasan detik.

In [ ]:
import pandas as pd, cv2
D = os.path.join(KODE, "pipeline")
VIDEO_HASIL = os.path.join(HASIL, "video", "notebook_lengkap_hasil.mp4")
cmd = ["python3", "run.py", VIDEO_UJI, MODEL_DET, "--cache", CACHE_DET, "--conf", "0.4", "--conf-low", "0.2",
       "--no-show", "--line-y", "50", "--out", VIDEO_HASIL]
if not PAKAI_KLASIFIER_GAMBAR:
    cmd.append("--tanpa-gambar")
print(" ".join(x.replace(RUMAH, "~") for x in cmd), "\n")
_ = jalankan(cmd, cwd=D, ekor=45)

In [ ]:
log_csv = os.path.join(D, "counting_log.csv")
hitung = pd.read_csv(log_csv) if os.path.exists(log_csv) else pd.DataFrame()
if len(hitung):
    _c = cv2.VideoCapture(VIDEO_UJI); fps = _c.get(cv2.CAP_PROP_FPS) or 25.0; _c.release()
    hitung["detik"] = (hitung["frame"] / fps).round(1)
    hitung["arah"] = hitung["x_m"].apply(lambda x: "menjauh" if x >= 7.0 else "mendekat")
    display(hitung)
    display(hitung["kelas"].value_counts().to_frame("jumlah"))
    display(hitung["arah"].value_counts().to_frame("jumlah"))
    cap = cv2.VideoCapture(VIDEO_HASIL)
    for f in list(hitung["frame"])[:3]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f) + 3)
        ok, fr = cap.read()
        if ok:
            print("frame", int(f) + 3); tampil_frame(fr, lebar=800)
    cap.release()
else:
    print("(tidak ada kendaraan yang dihitung)")

## 5. Evaluasi hitungan terhadap ground truth

`hitung_gt.py` (ditulis di bagian 4) menghitung kendaraan yang sebenarnya melintasi garis dari anotasi XML, lalu
mencocokkannya dengan `counting_log.csv`. TP = dihitung dan memang ada; FP = dihitung tetapi tidak ada; FN = ada tetapi
tidak dihitung.

In [ ]:
_, teks_gt = jalankan(["python3", "hitung_gt.py", "--first", str(FRAME_AWAL), "--last", str(FRAME_AKHIR),
                       "--line", "50", "--log", "counting_log.csv"], cwd=D, tampil=False)
awal = teks_gt.find("PERBANDINGAN")
print(teks_gt[awal:] if awal >= 0 else teks_gt[-1500:])
m = re.search(r"TP \(cocok\): (\d+) \| FP .*?: (\d+) \| FN .*?: (\d+)", teks_gt)
pasangan = re.findall(r"f(\d+)\s+id(\d+)\s+(\w+)\s+x=\s*([-\d.]+) -> GT#(\d+)\s+(\w+).*?(mendekat|menjauh)\s+([\d.]+) vs ([\d.]+) m", teks_gt)
if m:
    tp, fp, fn = map(int, m.groups())
    display(pd.DataFrame([{"TP": tp, "FP": fp, "FN": fn, "presisi": round(tp / (tp + fp), 2) if tp + fp else None,
                           "recall": round(tp / (tp + fn), 2) if tp + fn else None}]))
if pasangan:
    t = pd.DataFrame(pasangan, columns=["frame", "id", "kelas", "x (m)", "GT", "tipe anotasi", "arah", "panjang pipeline (m)", "panjang anotasi (m)"])
    t["selisih panjang (m)"] = (t["panjang pipeline (m)"].astype(float) - t["panjang anotasi (m)"].astype(float)).round(2)
    display(t)

## 6. Ringkasan dan keterbatasan

- **Sampel uji kecil.** Hasil di atas berasal dari satu klip sekitar 25 detik, jadi belum bisa disebut angka akurasi.
- **Kelas selain mobil belum teruji pada video.** Klip uji hanya berisi mobil pribadi.
- **Panjang kurang akurat di jarak jauh**, lebih besar selisihnya pada garis 85 m daripada garis 50 m.
- **Semua pengujian memakai data publik.** Kalibrasi, garis hitung, dan bobot perlu disesuaikan saat kamera sendiri terpasang.
- **Tanpa GPU**, deteksi berjalan sekitar 2–3 frame per detik.
- **Cara menggabungkan gambar dan panjang bersifat heuristik.**

Antarmuka web (`antarmuka/`) menjalankan `run.py` yang sama; kodenya tidak dimuat di notebook ini.

### Pemeriksaan: kode di notebook sama dengan kode asli

Sel ini membandingkan berkas yang ditulis notebook dengan berkas aslinya di `~/projects`. Jalankan setelah semua sel di
atas. Bila kode asli sudah diubah setelah notebook disusun, berkas yang berbeda akan disebut; susun ulang notebook dengan
`python3 ~/projects/alat/buat_notebook_lengkap.py`.

In [ ]:
ASAL = {
 "kode_notebook/latih_detektor/kamera.py": "~/projects/latih_detektor/kamera.py",
 "kode_notebook/latih_detektor/utils.py": "~/projects/acuan/3d_bbox_highway/utils.py",
 "kode_notebook/latih_detektor/target.py": "~/projects/latih_detektor/target.py",
 "kode_notebook/latih_detektor/train_session0_right.py": "~/projects/latih_detektor/train_session0_right.py",
 "kode_notebook/latih_detektor/dekode.py": "~/projects/latih_detektor/dekode.py",
 "kode_notebook/latih_detektor/prediksi.py": "~/projects/latih_detektor/prediksi.py",
 "kode_notebook/latih_detektor/evaluasi.py": "~/projects/latih_detektor/evaluasi.py",
 "kode_notebook/latih_detektor/analisis_galat.py": "~/projects/latih_detektor/analisis_galat.py",
 "kode_notebook/latih_klasifikasi/siapkan_mio_tcd.py": "~/projects/latih_klasifikasi/siapkan_mio_tcd.py",
 "kode_notebook/latih_klasifikasi/potong_session0.py": "~/projects/latih_klasifikasi/potong_session0.py",
 "kode_notebook/latih_klasifikasi/terapkan_label_truk.py": "~/projects/latih_klasifikasi/terapkan_label_truk.py",
 "kode_notebook/latih_klasifikasi/latih_klasifikasi.py": "~/projects/latih_klasifikasi/latih_klasifikasi.py",
 "kode_notebook/pipeline/config.py": "~/projects/uji_session0_pipeline/config.py",
 "kode_notebook/pipeline/detection_cache.py": "~/projects/uji_session0_pipeline/detection_cache.py",
 "kode_notebook/pipeline/counter.py": "~/projects/uji_session0_pipeline/counter.py",
 "kode_notebook/pipeline/geometry.py": "~/projects/uji_session0_pipeline/geometry.py",
 "kode_notebook/pipeline/detection_filter.py": "~/projects/uji_session0_pipeline/detection_filter.py",
 "kode_notebook/pipeline/kalman_filter.py": "~/projects/uji_session0_pipeline/kalman_filter.py",
 "kode_notebook/pipeline/box3d.py": "~/projects/uji_session0_pipeline/box3d.py",
 "kode_notebook/pipeline/klasifikasi_gambar.py": "~/projects/uji_session0_pipeline/klasifikasi_gambar.py",
 "kode_notebook/pipeline/classifier.py": "~/projects/uji_session0_pipeline/classifier.py",
 "kode_notebook/pipeline/tracker.py": "~/projects/uji_session0_pipeline/tracker.py",
 "kode_notebook/pipeline/visualize.py": "~/projects/uji_session0_pipeline/visualize.py",
 "kode_notebook/pipeline/detector.py": "~/projects/uji_session0_pipeline/detector.py",
 "kode_notebook/pipeline/run.py": "~/projects/uji_session0_pipeline/run.py",
 "kode_notebook/pipeline/hitung_gt.py": "~/projects/alat/hitung_gt.py"
}
def _norm(t):
    return [b.rstrip() for b in t.splitlines() if b.strip()]
beda, cocok, tak_ada = [], 0, []
for rel, asli in ASAL.items():
    asli = os.path.expanduser(asli)
    if not os.path.exists(asli):
        tak_ada.append(rel); continue
    tulis = os.path.join(os.path.dirname(KODE), rel)
    if os.path.exists(tulis) and _norm(open(tulis, encoding="utf-8").read()) == _norm(open(asli, encoding="utf-8").read()):
        cocok += 1
    else:
        beda.append(rel)
print("Sama dengan aslinya:", cocok, "dari", len(ASAL))
if beda:    print("BERBEDA:", beda)
if tak_ada: print("Asli tidak ada di komputer ini (wajar di Colab):", len(tak_ada))